<img src="https://raw.githubusercontent.com/senanurcetin/enveda-casmi26/main/site/assets/nexus-midnight.svg" width="360" alt="NEXUS">

# CASMI26: GNPS2 External Independence

Feasibility audit before model selection: pinned CC0 source records are compared with every CASMI training row for standard/metric identity and exact stored spectral content. The embedded protocol records the source, license basis and eligibility policy. This stage reports overlap and eligibility counts only. It trains no model, inspects no prediction scores and does not establish laboratory replication. Only aggregate counts and provenance hashes are exported.


In [ ]:
import hashlib,json,os,subprocess,sys,re
from pathlib import Path
sources = json.loads('{"src/casmi/__init__.py": "", "src/casmi/channels/__init__.py": "", "src/casmi/channels/analog.py": "\\"\\"\\"D2 \\u2014 mass-shifted analog propagation.\\n\\nA Class-2 molecule has no reference spectrum, but its structural relatives usually do. Relatives\\nthat differ by one substituent fragment into largely the same ions, either at the same m/z (the\\nsubstituent was lost) or shifted by the precursor difference (it was retained). We find such\\nanalog spectra in two stages and propagate their structures onto the candidates:\\n\\n  1. prefilter: every library spectrum is a sparse vector over fragment-m/z bins and\\n     neutral-loss bins; one sparse mat-vec scores a query spectrum against the whole library;\\n  2. refine: the top hits are rescored with mass-shifted entropy similarity;\\n  3. propagate: candidate score = max over analogs a of  sim(a)^p \\u00b7 Tanimoto(fp_c, fp_a).\\n\\"\\"\\"\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\nimport hashlib\\nimport json\\nimport zipfile\\n\\nimport numpy as np\\nimport pandas as pd\\nimport scipy.sparse as sp\\n\\nfrom casmi.spectra.clean import CleanConfig, clean_peaks_fast\\nfrom casmi.channels.prefilter import eligible_top_k\\nfrom casmi.channels.reference_selection import select_reference_rows, SELECTION_VERSION\\nfrom casmi.spectra.similarity import entropy_similarity\\n\\nBIN_DA = 0.05\\nMAX_MZ = 1300.0\\nN_BINS = int(MAX_MZ / BIN_DA) + 1\\n\\n\\n@dataclass(frozen=True)\\nclass AnalogConfig:\\n    max_shift_da: float = 200.0\\n    prefilter_top: int = 200\\n    keep_structures: int = 100\\n    per_structure_adduct: int = 3\\n    reference_policy: str = \\"legacy\\"  # opt-in: source_balanced (requires new index cache)\\n    tol_da: float = 0.02\\n    powers: tuple = (1.0, 2.0, 4.0)\\n    clean: CleanConfig = CleanConfig(min_rel_intensity=0.01, top_k_per_window=6, window_da=50, max_peaks=64)\\n\\n\\ndef _vector(mz, it, precursor_mz):\\n    \\"\\"\\"Sparse (indices, values) over [fragment bins | neutral-loss bins], L2-normalised.\\"\\"\\"\\n    w = np.sqrt(it.astype(np.float64))\\n    frag = np.minimum((mz / BIN_DA).astype(np.int64), N_BINS - 1)\\n    loss_mz = precursor_mz - mz\\n    ok = loss_mz > 0.5\\n    loss = N_BINS + np.minimum((loss_mz[ok] / BIN_DA).astype(np.int64), N_BINS - 1)\\n    idx = np.concatenate([frag, loss])\\n    val = np.concatenate([w, w[ok]])\\n    # duplicate bins (two peaks in one bin) are summed by the CSR constructor\\n    norm = np.sqrt((val ** 2).sum())\\n    return idx, val / norm if norm > 0 else val\\n\\n\\nclass AnalogIndex:\\n    def __init__(self, store, key_of_row, cfg: AnalogConfig = AnalogConfig(), cache: Path | None = None):\\n        self.store, self.cfg = store, cfg\\n        self._clean_cache = {}\\n        meta = store.meta\\n        expected = None\\n        stamp = None\\n        if cfg.reference_policy != \\"legacy\\":\\n            # A policy-specific cache never overwrites the frozen leaderboard\\n            # baseline\'s analog_index.npz. Nonlegacy loads are fail-closed.\\n            expected = self._select(meta, key_of_row)\\n            stamp = self._cache_stamp(store, key_of_row, expected)\\n            if cache is not None:\\n                path = Path(cache)\\n                cache = path.with_name(\\n                    f\\"{path.stem}_{cfg.reference_policy}_v1{path.suffix}\\"\\n                )\\n        loaded = False\\n        if cache is not None and Path(cache).exists():\\n            try:\\n                with np.load(cache, allow_pickle=False) as z:\\n                    if expected is not None and (\\n                        str(z[\\"selection_version\\"].item()) != SELECTION_VERSION\\n                        or str(z[\\"selection_digest\\"].item()) != stamp\\n                        or not np.array_equal(z[\\"rows\\"], expected)\\n                    ):\\n                        raise ValueError(\\"Cached D2 selection provenance mismatch\\")\\n                    cached_rows = z[\\"rows\\"]\\n                    cached_matrix = sp.csr_matrix(\\n                        (z[\\"data\\"], z[\\"indices\\"], z[\\"indptr\\"]),\\n                        shape=tuple(z[\\"shape\\"]),\\n                    )\\n                    if cached_matrix.shape != (len(cached_rows), 2 * N_BINS):\\n                        raise ValueError(\\"Cached D2 matrix shape is invalid\\")\\n                    if len(cached_rows) and (cached_rows.min() < 0 or cached_rows.max() >= len(meta)):\\n                        raise ValueError(\\"Cached D2 reference row index out of range\\")\\n                    self.rows, self.matrix = cached_rows, cached_matrix\\n                    loaded = True\\n            except (KeyError, OSError, ValueError, zipfile.BadZipFile):\\n                if expected is None:\\n                    raise  # fail closed on malformed frozen legacy caches\\n        if not loaded:\\n            self.rows = expected if expected is not None else self._select(meta, key_of_row)\\n            self.matrix = self._build(self.rows)\\n            if cache is not None:\\n                output = dict(rows=self.rows, data=self.matrix.data,\\n                              indices=self.matrix.indices, indptr=self.matrix.indptr,\\n                              shape=np.array(self.matrix.shape))\\n                if expected is not None:\\n                    output.update(selection_version=np.array(SELECTION_VERSION),\\n                                  selection_digest=np.array(stamp))\\n                np.savez(cache, **output)\\n        # Legacy and selection-validated caches are useful for exploratory\\n        # runs but cannot independently attest a freshly rebuilt D2 matrix.\\n        # Only cache=None sets the scientifically stronger fresh-origin bit.\\n        self.cache_origin = (\\n            \\"recomputed_without_cache\\" if cache is None\\n            else \\"loaded_cache\\" if loaded else \\"built_with_cache\\"\\n        )\\n        self.key = key_of_row[self.rows]\\n        self.prec = meta.precursor_mz.to_numpy()[self.rows]\\n        self.adduct = meta.adduct.astype(str).to_numpy()[self.rows]\\n        self.nmass = meta.neutral_mass.to_numpy()[self.rows]\\n\\n    def fresh_index_provenance(self):\\n        \\"\\"\\"Content digest for a D2 matrix rebuilt from the current store.\\n\\n        Only callable on a cache-free index. The digest binds selected rows,\\n        sparse matrix bytes and its configuration to the run record.\\n        It is a reproducibility checksum, NOT an external dataset signature.\\n        \\"\\"\\"\\n        if self.cache_origin != \\"recomputed_without_cache\\":\\n            raise ValueError(\\n                \\"D2 index was not freshly rebuilt; cached indices cannot \\"\\n                \\"support a decision-grade provenance record\\"\\n            )\\n        digest = hashlib.sha256(b\\"casmi-d2-fresh-index-v1\\\\\\\\n\\")\\n        cfg = json.dumps(asdict(self.cfg), sort_keys=True, separators=(\\",\\", \\":\\"))\\n        digest.update(cfg.encode(\\"utf-8\\"))\\n        for field in (\\n            self.rows, np.asarray(self.matrix.shape, dtype=np.int64),\\n            self.matrix.indptr, self.matrix.indices, self.matrix.data,\\n        ):\\n            array = np.ascontiguousarray(field)\\n            digest.update(str(array.dtype).encode(\\"ascii\\"))\\n            digest.update(np.asarray(array.shape, dtype=np.int64).tobytes())\\n            blob = memoryview(array).cast(\\"B\\")\\n            for start in range(0, len(blob), 1024 * 1024):\\n                digest.update(blob[start:start + 1024 * 1024])\\n        return {\\n            \\"schema\\": \\"casmi-d2-fresh-index-v1\\",\\n            \\"status\\": \\"REBUILT_WITHOUT_CACHE\\",\\n            \\"index_sha256\\": digest.hexdigest(),\\n            \\"n_references\\": int(self.matrix.shape[0]),\\n            \\"n_nonzero\\": int(self.matrix.nnz),\\n            \\"reference_policy\\": self.cfg.reference_policy,\\n            \\"cache_used\\": False,\\n        }\\n\\n    def _select(self, meta, key_of_row):\\n        return select_reference_rows(\\n            meta, key_of_row,\\n            per_structure_adduct=self.cfg.per_structure_adduct,\\n            policy=self.cfg.reference_policy,\\n        )\\n\\n    def _cache_stamp(self, store, keys, rows):\\n        \\"\\"\\"Hash selected metadata, source-array file stamps and index config.\\n\\n        The selected rows are recomputed on every nonlegacy cache load, so\\n        updated eligibility or changes in available libraries invalidate the\\n        cached index even if the number of spectra is unchanged.\\n        \\"\\"\\"\\n        digest = hashlib.sha256()\\n        digest.update(SELECTION_VERSION.encode())\\n        digest.update(json.dumps(asdict(self.cfg), sort_keys=True).encode())\\n        digest.update(np.asarray(rows, dtype=np.int64).tobytes())\\n        selected = store.meta.iloc[rows][[\\n            \\"ingest_lib\\", \\"adduct\\", \\"precursor_mz\\", \\"neutral_mass\\",\\n            \\"ppm\\", \\"n_peaks\\", \\"is_timstof\\", \\"test_adduct\\",\\n        ]]\\n        digest.update(pd.util.hash_pandas_object(selected, index=False).values.tobytes())\\n        digest.update(pd.util.hash_pandas_object(\\n            pd.Series(np.asarray(keys)[rows], dtype=\\"string\\"), index=False,\\n        ).values.tobytes())\\n        for field in (\\"mz\\", \\"it\\", \\"offsets\\"):\\n            ref = getattr(store, field, None)\\n            filename = getattr(ref, \\"filename\\", None)\\n            if filename is None:\\n                continue\\n            stat = Path(filename).stat()\\n            digest.update(\\n                f\\"{field}:{stat.st_size}:{stat.st_mtime_ns}\\".encode()\\n            )\\n        return digest.hexdigest()\\n\\n    def _clean(self, row):\\n        hit = self._clean_cache.get(row)\\n        if hit is None:\\n            mz, it = self.store.peaks(row)\\n            hit = clean_peaks_fast(mz, it, float(self.store.meta.precursor_mz.iat[row]), self.cfg.clean)\\n            if len(self._clean_cache) < 300_000:\\n                self._clean_cache[row] = hit\\n        return hit\\n\\n    def _build(self, rows):\\n        if len(rows) == 0:\\n            return sp.csr_matrix((0, 2 * N_BINS), dtype=np.float32)\\n        indptr, indices, data = [0], [], []\\n        prec = self.store.meta.precursor_mz.to_numpy()\\n        for r in rows:\\n            mz, it = self._clean(r)\\n            i, v = _vector(mz, it, prec[r]) if mz.size else (np.empty(0, np.int64), np.empty(0))\\n            indices.append(i)\\n            data.append(v.astype(np.float32))\\n            indptr.append(indptr[-1] + i.size)\\n        self._clean_cache.clear()\\n        m = sp.csr_matrix((np.concatenate(data), np.concatenate(indices), np.array(indptr)),\\n                          shape=(len(rows), 2 * N_BINS))\\n        m.sum_duplicates()\\n        return m\\n\\n    def search(self, query_spectra, query_mass, excluded_rows=None):\\n        \\"\\"\\"Best refined similarity per analog structure: dict key -> (sim, analog neutral mass).\\"\\"\\"\\n        best = {}\\n        # A structure-level knockout must be applied BEFORE selecting the top\\n        # analog hits. Previously self-references displaced allowed hits in the\\n        # shortlist and were discarded only afterwards.\\n        excluded = (np.asarray(excluded_rows, dtype=np.int64)\\n                    if excluded_rows is not None else np.empty(0, np.int64))\\n        not_excluded = (~np.isin(self.rows, excluded)\\n                        if excluded.size else np.ones(len(self.rows), dtype=bool))\\n        mass_ok = np.isfinite(self.nmass) & np.isfinite(query_mass) & (\\n            np.abs(self.nmass - query_mass) <= self.cfg.max_shift_da\\n        )\\n        for q in query_spectra:\\n            if q[\\"mz_c\\"].size == 0:\\n                continue\\n            i, v = _vector(q[\\"mz_c\\"], q[\\"it_c\\"], q[\\"precursor_mz\\"])\\n            # Dense CSR mat-vec avoids a large sparse x sparse temporary.\\n            dq = np.zeros(2 * N_BINS, np.float32)\\n            np.add.at(dq, i, v.astype(np.float32))\\n            scores = self.matrix @ dq\\n            allowed = not_excluded & mass_ok & (self.adduct == q[\\"adduct\\"])\\n            top = eligible_top_k(scores, allowed, self.cfg.prefilter_top)\\n            for t in top:\\n                rm, ri = self._clean(self.rows[t])\\n                sim = entropy_similarity(q[\\"mz_c\\"], q[\\"it_c\\"], rm, ri, self.cfg.tol_da,\\n                                         float(q[\\"precursor_mz\\"] - self.prec[t]))\\n                k = self.key[t]\\n                if sim > best.get(k, (0.0, 0.0))[0]:\\n                    best[k] = (sim, float(self.nmass[t]))\\n        items = sorted(best.items(), key=lambda kv: -kv[1][0])[: self.cfg.keep_structures]\\n        return dict(items)\\n\\n\\ndef tanimoto_matrix(fa, fb):\\n    \\"\\"\\"Tanimoto between packed-bit fingerprint rows: fa [n, 256] vs fb [m, 256] -> [n, m].\\"\\"\\"\\n    a = np.unpackbits(fa, axis=1).astype(np.float32)\\n    b = np.unpackbits(fb, axis=1).astype(np.float32)\\n    inter = a @ b.T\\n    return inter / (a.sum(1)[:, None] + b.sum(1)[None, :] - inter + 1e-9)\\n\\n\\ndef propagate(analogs: dict, cand_rows, cand_mass, fps, row_of, powers):\\n    \\"\\"\\"Per-candidate analog features. `analogs`: key -> (sim, analog neutral mass).\\"\\"\\"\\n    n = len(cand_rows)\\n    feats = {f\\"an_p{int(p)}\\": np.zeros(n) for p in powers}\\n    feats.update(an_sim=np.zeros(n), an_tani=np.zeros(n), an_dmass=np.full(n, 999.0),\\n                 an_n_close=np.zeros(n), an_self=np.zeros(n))\\n    keys = [k for k in analogs if k in row_of]\\n    if not keys or n == 0:\\n        return feats\\n    sims = np.array([analogs[k][0] for k in keys])\\n    amass = np.array([analogs[k][1] for k in keys])\\n    T = tanimoto_matrix(fps[cand_rows], fps[[row_of[k] for k in keys]])\\n    for p in powers:\\n        feats[f\\"an_p{int(p)}\\"] = (T * sims[None, :] ** p).max(axis=1)\\n    best = (T * sims[None, :] ** 4).argmax(axis=1)\\n    feats[\\"an_sim\\"] = sims[best]\\n    feats[\\"an_tani\\"] = T[np.arange(n), best]\\n    feats[\\"an_dmass\\"] = cand_mass - amass[best]\\n    feats[\\"an_n_close\\"] = ((T >= 0.7) & (sims[None, :] >= 0.5)).sum(axis=1)\\n    feats[\\"an_self\\"] = np.where(T >= 0.999, sims[None, :], 0).max(axis=1)\\n    return feats\\n", "src/casmi/channels/bondmodel.py": "\\"\\"\\"D5 \\u2014 learned bond-cleavage likelihood.\\n\\nD4 counts how much of the spectrum a candidate *can* explain, which favours candidates that\\nfragment into many pieces. D5 asks the reverse question as well: of the cleavages this candidate\\n*should* show, which do we see? Each cleavage event (one acyclic bond, or two bonds of the same\\nring) has a type; its probability of producing an observed fragment ion is learned from library\\nspectra of training structures. A candidate\'s score is the Bernoulli log-likelihood of the\\nobserved/unobserved pattern of its events, averaged over events and spectra.\\n\\"\\"\\"\\nimport numpy as np\\nfrom rdkit import Chem\\n\\nfrom casmi.chem.masses import ELECTRON, PROTON\\n\\nH = 1.00782503207\\nNA = 22.9897692820\\nTOL_DA, TOL_PPM = 0.01, 10.0\\nMIN_REL = 0.01\\n\\n\\ndef _atom_masses(mol):\\n    pt = Chem.GetPeriodicTable()\\n    return np.array([pt.GetMassForIsotope(a.GetSymbol(), pt.GetMostCommonIsotope(a.GetSymbol()))\\n                     + a.GetTotalNumHs() * H for a in mol.GetAtoms()])\\n\\n\\ndef _bond_key(b):\\n    a1, a2 = b.GetBeginAtom(), b.GetEndAtom()\\n    ends = sorted([(a1.GetSymbol(), a1.GetDegree(), a1.GetIsAromatic()), (a2.GetSymbol(), a2.GetDegree(), a2.GetIsAromatic())])\\n    return f\\"{ends[0][0]}{ends[0][1]}{\'a\' if ends[0][2] else \'\'}-{b.GetBondTypeAsDouble():g}-{ends[1][0]}{ends[1][1]}{\'a\' if ends[1][2] else \'\'}\\"\\n\\n\\ndef _split(mol, bonds, masses):\\n    \\"\\"\\"Masses of the two pieces after removing `bonds`, or None if the molecule stays connected.\\"\\"\\"\\n    em = Chem.RWMol(mol)\\n    for b in bonds:\\n        bd = mol.GetBondWithIdx(b)\\n        em.RemoveBond(bd.GetBeginAtomIdx(), bd.GetEndAtomIdx())\\n    frags = Chem.GetMolFrags(em, asMols=False, sanitizeFrags=False)\\n    if len(frags) != 2:\\n        return None\\n    return tuple(float(masses[list(f)].sum()) for f in frags)\\n\\n\\ndef cleavage_events(smiles):\\n    \\"\\"\\"[(event key, (piece mass 1, piece mass 2))]: acyclic single bonds + bond pairs within a ring.\\"\\"\\"\\n    mol = Chem.MolFromSmiles(smiles)\\n    if mol is None or mol.GetNumAtoms() > 120:\\n        return []\\n    masses = _atom_masses(mol)\\n    out = []\\n    ring_bonds = set()\\n    for ring in mol.GetRingInfo().BondRings():\\n        ring_bonds.update(ring)\\n    for b in mol.GetBonds():\\n        if b.GetIdx() in ring_bonds or b.GetBondTypeAsDouble() > 1.5:\\n            continue\\n        pieces = _split(mol, [b.GetIdx()], masses)\\n        if pieces:\\n            out.append((\\"A:\\" + _bond_key(b), pieces))\\n    for ring in mol.GetRingInfo().BondRings():\\n        if len(ring) > 7 or any(mol.GetBondWithIdx(i).GetIsAromatic() for i in ring):\\n            continue\\n        for i in range(len(ring)):\\n            for j in range(i + 2, len(ring)):\\n                if i == 0 and j == len(ring) - 1:\\n                    continue\\n                pieces = _split(mol, [ring[i], ring[j]], masses)\\n                if pieces:\\n                    k = \\"|\\".join(sorted([_bond_key(mol.GetBondWithIdx(ring[i])), _bond_key(mol.GetBondWithIdx(ring[j]))]))\\n                    out.append((\\"R:\\" + k, pieces))\\n    return out\\n\\n\\ndef _carriers(mode, adduct):\\n    if mode.lower().startswith(\\"n\\") or adduct.endswith(\\"-\\"):\\n        return [-PROTON]\\n    return [PROTON] + ([NA - ELECTRON] if \\"Na\\" in adduct else [])\\n\\n\\ndef observed(events, mz, rel, mode, adduct, precursor_mz):\\n    \\"\\"\\"Per event: 1 if one of its pieces (\\u00b12 H, precursor carrier) matches a peak >= MIN_REL.\\"\\"\\"\\n    keep = (rel >= MIN_REL) & (mz < precursor_mz - 1.5)\\n    mz = np.sort(mz[keep])\\n    obs = np.zeros(len(events), np.int8)\\n    if len(mz) == 0:\\n        return obs\\n    shifts = np.array([c + k * H for c in _carriers(mode, adduct) for k in (-2, -1, 0, 1, 2)])\\n    for e, (_, pieces) in enumerate(events):\\n        ions = (np.array(pieces)[:, None] + shifts[None, :]).ravel()\\n        pos = np.searchsorted(mz, ions)\\n        lo = mz[np.clip(pos - 1, 0, len(mz) - 1)]\\n        hi = mz[np.clip(pos, 0, len(mz) - 1)]\\n        d = np.minimum(np.abs(ions - lo), np.abs(ions - hi))\\n        obs[e] = int((d <= np.maximum(TOL_DA, ions * TOL_PPM * 1e-6)).any())\\n    return obs\\n\\n\\nclass BondModel:\\n    \\"\\"\\"Event-type propensities: P(observed | event key, ion mode), Beta-smoothed toward the global rate.\\"\\"\\"\\n\\n    def __init__(self, counts=None, prior=4.0):\\n        self.counts = counts or {}          # (key, mode) -> [n, observed]\\n        self.prior = prior\\n        self._p = None\\n\\n    def add(self, events, obs, mode):\\n        for (k, _), o in zip(events, obs):\\n            c = self.counts.setdefault((k, mode), [0, 0])\\n            c[0] += 1\\n            c[1] += int(o)\\n\\n    def finalize(self):\\n        n = sum(c[0] for c in self.counts.values())\\n        o = sum(c[1] for c in self.counts.values())\\n        base = o / max(n, 1)\\n        self.base = base\\n        self._p = {k: (c[1] + self.prior * base) / (c[0] + self.prior) for k, c in self.counts.items()}\\n        return self\\n\\n    def p(self, key, mode):\\n        return self._p.get((key, mode), self.base)\\n\\n    def score(self, events, spectra):\\n        \\"\\"\\"(mean log-likelihood per event, observed share weighted by propensity) averaged over spectra.\\"\\"\\"\\n        if not events or not spectra:\\n            return 0.0, 0.0\\n        lls, ws = [], []\\n        for s in spectra:\\n            mode = \\"neg\\" if (str(s[\\"mode\\"]).lower().startswith(\\"n\\") or str(s[\\"adduct\\"]).endswith(\\"-\\")) else \\"pos\\"\\n            it = np.asarray(s[\\"it\\"], float)\\n            rel = it / it.max() if len(it) and it.max() > 0 else it\\n            obs = observed(events, np.asarray(s[\\"mz\\"], float), rel, mode, str(s[\\"adduct\\"]), float(s[\\"precursor_mz\\"]))\\n            p = np.clip(np.array([self.p(k, mode) for k, _ in events]), 1e-3, 1 - 1e-3)\\n            lls.append(float(np.mean(np.where(obs == 1, np.log(p), np.log(1 - p)))))\\n            ws.append(float((p * obs).sum() / p.sum()))\\n        return float(np.mean(lls)), float(np.mean(ws))\\n", "src/casmi/channels/fpinfer.py": "\\"\\"\\"D3 inference shared by validation (scripts/add_d3_features.py) and the submission.\\"\\"\\"\\nimport numpy as np\\nimport torch\\n\\nfrom casmi.chem.masses import TEST_ADDUCTS\\nfrom casmi.spectra.clean import CleanConfig, clean_peaks_fast\\n\\nMAX_PEAKS = 64\\nFP_CLEAN = CleanConfig(min_rel_intensity=0.005, top_k_per_window=8, window_da=50, max_peaks=MAX_PEAKS)\\nADDUCT_ID = {a: i for i, a in enumerate(TEST_ADDUCTS)}\\n\\n\\n@torch.no_grad()\\ndef molecule_logp(models, spectra, device):\\n    \\"\\"\\"Mean predicted bit probability over a molecule\'s spectra and over models.\\n    Returns (log p(bit=1), log p(bit=0)) as NumPy arrays of length n_bits.\\"\\"\\"\\n    n = len(spectra)\\n    mz, it, npk = np.zeros((n, MAX_PEAKS), np.float32), np.zeros((n, MAX_PEAKS), np.float32), np.zeros(n, np.int64)\\n    for j, s in enumerate(spectra):\\n        cm, ci = clean_peaks_fast(s[\\"mz\\"], s[\\"it\\"], s[\\"precursor_mz\\"], FP_CLEAN)\\n        mz[j, :cm.size], it[j, :cm.size], npk[j] = cm, ci, cm.size\\n    t = lambda a, dt=torch.float32: torch.as_tensor(np.asarray(a), dtype=dt, device=device)\\n    x = dict(mz=t(mz), it=t(it), npk=t(npk, torch.long),\\n             prec=t([s[\\"precursor_mz\\"] for s in spectra]),\\n             adduct=t([ADDUCT_ID.get(s[\\"adduct\\"], 0) for s in spectra], torch.long),\\n             ce=t([s[\\"ce_ev\\"] for s in spectra]), positive=t([float(s[\\"mode\\"] == \\"positive\\") for s in spectra]))\\n    p = torch.stack([torch.sigmoid(m(**x)) for m in models]).mean(dim=(0, 1)).clamp(1e-4, 1 - 1e-4)\\n    return torch.log(p).cpu().numpy(), torch.log1p(-p).cpu().numpy()\\n", "src/casmi/channels/fpmodel.py": "\\"\\"\\"D3 \\u2014 spectrum \\u2192 fingerprint model and candidate scoring.\\n\\nA small set transformer reads \\u2264 64 peaks (no positional encoding: peaks are a set) plus one\\nglobal token (precursor mass, adduct, collision energy, polarity) and predicts the 2048-bit\\nMorgan fingerprint used by the pool, so a prediction can be scored directly against candidates.\\n\\nm/z values need ~0.01 Da resolution over a 1,000 Da range, which a single linear projection cannot\\nrepresent; they are encoded with sinusoids at log-spaced wavelengths (as in DreaMS / MIST).\\nEach peak is described both by its m/z and by its neutral loss from the precursor.\\n\\"\\"\\"\\nfrom dataclasses import asdict, dataclass\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn as nn\\n\\nN_ADDUCTS = 10\\n\\n\\n@dataclass\\nclass FPModelConfig:\\n    n_bits: int = 2048\\n    d_model: int = 256\\n    n_layers: int = 4\\n    n_heads: int = 8\\n    d_ff: int = 1024\\n    dropout: float = 0.1\\n    n_freq: int = 32\\n    min_wavelength: float = 0.01\\n    max_wavelength: float = 1000.0\\n\\n\\nclass FourierMass(nn.Module):\\n    def __init__(self, n_freq, lo, hi):\\n        super().__init__()\\n        wl = torch.logspace(np.log10(lo), np.log10(hi), n_freq)\\n        self.register_buffer(\\"w\\", 2 * np.pi / wl)\\n\\n    def forward(self, x):                      # x [..., ] -> [..., 2 * n_freq]\\n        a = x.unsqueeze(-1) * self.w\\n        return torch.cat([torch.sin(a), torch.cos(a)], dim=-1)\\n\\n\\nclass FPModel(nn.Module):\\n    def __init__(self, cfg: FPModelConfig = FPModelConfig()):\\n        super().__init__()\\n        self.cfg = cfg\\n        f = 2 * cfg.n_freq\\n        self.enc = FourierMass(cfg.n_freq, cfg.min_wavelength, cfg.max_wavelength)\\n        self.peak_proj = nn.Sequential(nn.Linear(2 * f + 2, cfg.d_model), nn.GELU(), nn.Linear(cfg.d_model, cfg.d_model))\\n        self.adduct_emb = nn.Embedding(N_ADDUCTS, 32)\\n        self.glob_proj = nn.Sequential(nn.Linear(f + 32 + 2, cfg.d_model), nn.GELU(), nn.Linear(cfg.d_model, cfg.d_model))\\n        layer = nn.TransformerEncoderLayer(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout,\\n                                           activation=\\"gelu\\", batch_first=True, norm_first=True)\\n        self.encoder = nn.TransformerEncoder(layer, cfg.n_layers, enable_nested_tensor=False)\\n        self.head = nn.Sequential(nn.LayerNorm(cfg.d_model), nn.Linear(cfg.d_model, cfg.d_model),\\n                                  nn.GELU(), nn.Linear(cfg.d_model, cfg.n_bits))\\n\\n    def forward(self, mz, it, npk, prec, adduct, ce, positive):\\n        \\"\\"\\"mz, it [B, P]; npk [B]; prec, ce, positive [B]; adduct [B] (long). Returns logits [B, n_bits].\\"\\"\\"\\n        B, P = mz.shape\\n        pad = torch.arange(P, device=mz.device)[None, :] >= npk[:, None]\\n        loss = (prec[:, None] - mz).clamp(min=0)\\n        peak = torch.cat([self.enc(mz), self.enc(loss), it.sqrt().unsqueeze(-1), torch.log1p(it * 100).unsqueeze(-1)], -1)\\n        tok = self.peak_proj(peak)\\n        ce_n = torch.nan_to_num(ce, nan=30.0) / 100.0\\n        glob = torch.cat([self.enc(prec), self.adduct_emb(adduct), ce_n[:, None], positive[:, None]], -1)\\n        x = torch.cat([self.glob_proj(glob)[:, None, :], tok], dim=1)\\n        mask = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=mz.device), pad], dim=1)\\n        h = self.encoder(x, src_key_padding_mask=mask)\\n        return self.head(h[:, 0])\\n\\n\\ndef save(model, path, extra=None):\\n    torch.save({\\"cfg\\": asdict(model.cfg), \\"state\\": model.state_dict(), **(extra or {})}, path)\\n\\n\\ndef load(path, device=\\"cpu\\"):\\n    ck = torch.load(path, map_location=device, weights_only=False)\\n    m = FPModel(FPModelConfig(**ck[\\"cfg\\"])).to(device)\\n    m.load_state_dict(ck[\\"state\\"])\\n    return m.eval()\\n\\n\\ndef candidate_scores(logp_on, logp_off, cand_bits):\\n    \\"\\"\\"Bernoulli log-likelihood of each candidate fingerprint under the predicted bit probabilities.\\n\\n    logp_on / logp_off: [n_bits] molecule-level log p(bit=1) / log p(bit=0)\\n    cand_bits: [n_cand, n_bits] {0,1}\\n    Returns (mean log-likelihood per bit, cosine between probabilities and candidate bits).\\n    \\"\\"\\"\\n    ll = (cand_bits * logp_on + (1 - cand_bits) * logp_off).mean(axis=1)\\n    p = np.exp(logp_on)\\n    cos = (cand_bits @ p) / (np.linalg.norm(p) * np.sqrt(cand_bits.sum(axis=1)) + 1e-9)\\n    return ll, cos\\n", "src/casmi/channels/fragments.py": "\\"\\"\\"D4 \\u2014 fragment explanation: how much of the query spectrum a candidate structure can explain.\\n\\nCandidates in one mass window are mostly isomers, so formula-level evidence cannot separate them;\\nbond topology can. Every candidate is broken at one or two bonds (combinatorial fragmentation in\\nthe spirit of MetFrag, depth 2), every resulting connected piece becomes a neutral fragment mass,\\nand each fragment is ionised with the precursor\'s charge carrier and \\u00b12 hydrogen rearrangements.\\nA query peak is explained when some fragment ion lies within tolerance.\\n\\nFeatures per candidate (averaged and maxed over the molecule\'s spectra):\\n  d4_int   share of fragment-peak intensity explained\\n  d4_pk    share of fragment peaks explained\\nPeaks within 1.5 Da of the precursor are ignored (every candidate explains the precursor).\\n\\"\\"\\"\\nimport numpy as np\\nfrom numba import njit\\n\\nfrom casmi.chem.masses import ELECTRON, PROTON\\nfrom casmi.spectra.clean import CleanConfig, clean_peaks_fast\\n\\nH = 1.00782503207\\nNA = 22.9897692820\\nFRAG_CLEAN = CleanConfig(max_above_precursor=-1.5, min_rel_intensity=0.01, max_peaks=60)\\nTOL_DA, TOL_PPM = 0.01, 10.0\\nMAX_ATOMS = 160\\n\\n\\n@njit(cache=True)\\ndef _find(parent, a):\\n    while parent[a] != a:\\n        parent[a] = parent[parent[a]]\\n        a = parent[a]\\n    return a\\n\\n\\n@njit(cache=True)\\ndef _pieces(n, atom_mass, bu, bv, skip1, skip2, out, n_out, total):\\n    parent = np.arange(n)\\n    for b in range(len(bu)):\\n        if b == skip1 or b == skip2:\\n            continue\\n        ra, rb = _find(parent, bu[b]), _find(parent, bv[b])\\n        if ra != rb:\\n            parent[ra] = rb\\n    acc = np.zeros(n)\\n    for a in range(n):\\n        acc[_find(parent, a)] += atom_mass[a]\\n    for a in range(n):\\n        m = acc[a]\\n        if m > 0.0 and m < total - 1e-6 and n_out < len(out):\\n            out[n_out] = m\\n            n_out += 1\\n    return n_out\\n\\n\\n@njit(cache=True)\\ndef _fragments(atom_mass, bu, bv):\\n    n, m = len(atom_mass), len(bu)\\n    total = atom_mass.sum()\\n    out = np.empty(3 * (m + m * (m - 1) // 2) + 1)\\n    k = 0\\n    for i in range(m):\\n        k = _pieces(n, atom_mass, bu, bv, i, -1, out, k, total)\\n        for j in range(i + 1, m):\\n            k = _pieces(n, atom_mass, bu, bv, i, j, out, k, total)\\n    return out[:k]\\n\\n\\n@njit(cache=True)\\ndef _fragments3(atom_mass, bu, bv):\\n    \\"\\"\\"Depth 3: every bond triple as well (breaks rings in two places plus one more cut).\\"\\"\\"\\n    n, m = len(atom_mass), len(bu)\\n    total = atom_mass.sum()\\n    base = _fragments(atom_mass, bu, bv)\\n    out = np.empty(len(base) + 4 * (m * (m - 1) * (m - 2) // 6) + 1)\\n    out[:len(base)] = base\\n    k = len(base)\\n    for i in range(m):\\n        for j in range(i + 1, m):\\n            for l in range(j + 1, m):\\n                parent = np.arange(n)\\n                for b in range(m):\\n                    if b == i or b == j or b == l:\\n                        continue\\n                    ra, rb = _find(parent, bu[b]), _find(parent, bv[b])\\n                    if ra != rb:\\n                        parent[ra] = rb\\n                acc = np.zeros(n)\\n                for a in range(n):\\n                    acc[_find(parent, a)] += atom_mass[a]\\n                for a in range(n):\\n                    if acc[a] > 0.0 and acc[a] < total - 1e-6 and k < len(out):\\n                        out[k] = acc[a]\\n                        k += 1\\n    return out[:k]\\n\\n\\nMAX_BONDS_DEPTH3 = 70\\n\\n\\ndef fragment_masses(mol, depth=2):\\n    \\"\\"\\"Sorted unique neutral fragment masses of an RDKit molecule (heavy atoms carry their Hs).\\"\\"\\"\\n    n = mol.GetNumAtoms()\\n    if n < 2 or n > MAX_ATOMS:\\n        return np.empty(0)\\n    atom_mass = np.array([_mono(a) for a in mol.GetAtoms()])\\n    bu = np.array([b.GetBeginAtomIdx() for b in mol.GetBonds()], np.int64)\\n    bv = np.array([b.GetEndAtomIdx() for b in mol.GetBonds()], np.int64)\\n    if len(bu) == 0:\\n        return np.empty(0)\\n    f = _fragments3(atom_mass, bu, bv) if depth >= 3 and len(bu) <= MAX_BONDS_DEPTH3 else _fragments(atom_mass, bu, bv)\\n    return np.unique(np.round(f, 4))\\n\\n\\n_MONO = {}\\n\\n\\ndef _mono(atom):\\n    from rdkit.Chem import GetPeriodicTable\\n    s = atom.GetSymbol()\\n    if s not in _MONO:\\n        pt = GetPeriodicTable()\\n        _MONO[s] = pt.GetMassForIsotope(s, pt.GetMostCommonIsotope(s))\\n    return _MONO[s] + atom.GetTotalNumHs() * H\\n\\n\\ndef ion_masses(frags, mode, adduct, h_max=2):\\n    \\"\\"\\"Fragment ions for the precursor\'s charge carrier, with -h_max..+h_max hydrogen rearrangements.\\"\\"\\"\\n    if mode.lower().startswith(\\"n\\") or adduct.endswith(\\"-\\"):\\n        carriers = [-PROTON]\\n    else:\\n        carriers = [PROTON] + ([NA - ELECTRON] if \\"Na\\" in adduct else [])\\n    shifts = np.array([c + k * H for c in carriers for k in range(-h_max, h_max + 1)])\\n    return np.sort((frags[:, None] + shifts[None, :]).ravel())\\n\\n\\ndef explained(peaks_mz, peaks_it, ions, tol_da=TOL_DA, tol_ppm=TOL_PPM):\\n    \\"\\"\\"(intensity share, peak share) of the peaks within tolerance of some ion.\\"\\"\\"\\n    if len(peaks_mz) == 0 or len(ions) == 0:\\n        return 0.0, 0.0\\n    pos = np.searchsorted(ions, peaks_mz)\\n    lo = ions[np.clip(pos - 1, 0, len(ions) - 1)]\\n    hi = ions[np.clip(pos, 0, len(ions) - 1)]\\n    d = np.minimum(np.abs(peaks_mz - lo), np.abs(peaks_mz - hi))\\n    hit = d <= np.maximum(tol_da, peaks_mz * tol_ppm * 1e-6)\\n    return float(peaks_it[hit].sum() / peaks_it.sum()), float(hit.mean())\\n\\n\\nclass FragmentScorer:\\n    \\"\\"\\"Caches fragment masses per structure; scores (query spectra \\u00d7 candidates).\\"\\"\\"\\n\\n    def __init__(self, smiles_of, tol_da=TOL_DA, tol_ppm=TOL_PPM, h_max=2, weight=\\"sqrt\\", depth=2):\\n        \\"\\"\\"Defaults reproduce the D4 the current rankers were trained with.\\"\\"\\"\\n        self.smiles_of = smiles_of\\n        self.tol_da, self.tol_ppm, self.h_max, self.weight, self.depth = tol_da, tol_ppm, h_max, weight, depth\\n        self.cache = {}\\n\\n    def frags(self, key):\\n        f = self.cache.get(key)\\n        if f is None:\\n            from rdkit import Chem\\n            mol = Chem.MolFromSmiles(self.smiles_of(key))\\n            f = fragment_masses(mol, self.depth) if mol is not None else np.empty(0)\\n            self.cache[key] = f\\n        return f\\n\\n    def prepare(self, spectra):\\n        \\"\\"\\"Cleaned fragment peaks per spectrum (sqrt intensity, precursor region removed).\\"\\"\\"\\n        out = []\\n        for s in spectra:\\n            mz, it = clean_peaks_fast(s[\\"mz\\"], s[\\"it\\"], s[\\"precursor_mz\\"], FRAG_CLEAN)\\n            mz, it = np.asarray(mz, np.float64), np.asarray(it, np.float64)\\n            it = {\\"sqrt\\": np.sqrt(it), \\"raw\\": it, \\"log\\": np.log1p(100 * it), \\"count\\": np.ones_like(it)}[self.weight]\\n            if len(mz):\\n                out.append((mz, it, str(s.get(\\"mode\\", \\"\\")), str(s.get(\\"adduct\\", \\"\\"))))\\n        return out\\n\\n    def score(self, spectra, keys):\\n        peaks = self.prepare(spectra)\\n        d4 = np.zeros((len(keys), 4))\\n        if not peaks:\\n            return d4\\n        for i, k in enumerate(keys):\\n            f = self.frags(k)\\n            v = np.array([explained(mz, it, ion_masses(f, mode, adduct, self.h_max), self.tol_da, self.tol_ppm)\\n                          for mz, it, mode, adduct in peaks])\\n            d4[i] = (v[:, 0].mean(), v[:, 0].max(), v[:, 1].mean(), v[:, 1].max())\\n        return d4\\n\\n\\nD4_COLUMNS = [\\"d4_int\\", \\"d4_int_max\\", \\"d4_pk\\", \\"d4_pk_max\\"]\\nD4X = dict(depth=3)              # second D4 block (\\"d4x_\\"): depth-3 cleavage (probe_d4.py: best alone)\\n\\n\\ndef add_d4(f, scorer, spectra, prefix=\\"d4\\"):\\n    \\"\\"\\"D4 columns plus within-list rank / gap / z for the molecule\'s candidate table `f`.\\"\\"\\"\\n    import pandas as pd\\n    d4 = scorer.score(spectra, f.pool_key.to_numpy())\\n    names = [c.replace(\\"d4\\", prefix, 1) for c in D4_COLUMNS]\\n    f = f.assign(**{c: d4[:, j] for j, c in enumerate(names)})\\n    for c in (f\\"{prefix}_int\\", f\\"{prefix}_pk\\"):\\n        v = f[c].to_numpy()\\n        sd = v.std()\\n        f[f\\"{c}_rank\\"] = pd.Series(v).rank(ascending=False, method=\\"min\\").to_numpy()\\n        f[f\\"{c}_gap\\"] = v.max() - v\\n        f[f\\"{c}_z\\"] = (v - v.mean()) / sd if sd > 0 else 0.0\\n    return f\\n\\n\\ndef add_d4_table(tbl, spectra_of_qid, smiles_of, cache_path=None, prefix=\\"d4\\", **scorer_kw):\\n    \\"\\"\\"D4 for every (qid, cls) group of a ranker table; fragment masses cached on disk.\\"\\"\\"\\n    import pickle\\n    import pandas as pd\\n    scorer = FragmentScorer(smiles_of, **scorer_kw)\\n    if cache_path is not None and cache_path.exists():\\n        scorer.cache.update(pickle.loads(cache_path.read_bytes()))\\n    parts = [add_d4(g.reset_index(drop=True), scorer, spectra_of_qid[qid], prefix)\\n             for (qid, _), g in tbl.groupby([\\"qid\\", \\"cls\\"], sort=False)]\\n    if cache_path is not None:\\n        cache_path.write_bytes(pickle.dumps(scorer.cache, protocol=pickle.HIGHEST_PROTOCOL))\\n    return pd.concat(parts, ignore_index=True)\\n", "src/casmi/channels/library.py": "\\"\\"\\"D1 \\u2014 direct library match: compare the query\'s spectra with each candidate\'s own reference spectra.\\n\\nFor every candidate structure we look up its library spectra (by metric key), keep the most\\ncomparable ones (same ionisation mode, test adduct, plausible mass), and score the best\\nquery \\u00d7 reference entropy similarity. Knocked-out spectra are never looked at.\\n\\"\\"\\"\\nfrom dataclasses import dataclass\\n\\nimport numpy as np\\n\\nfrom casmi.spectra.clean import CleanConfig, clean_peaks_fast\\nfrom casmi.spectra.similarity import entropy_similarity\\n\\n\\n@dataclass(frozen=True)\\nclass LibraryConfig:\\n    tol_da: float = 0.02\\n    max_ppm: float = 20.0              # reference spectra with worse precursor error are not trusted\\n    max_refs_per_candidate: int = 40   # cap work per candidate; timsTOF + same adduct first\\n    clean: CleanConfig = CleanConfig(min_rel_intensity=0.01, top_k_per_window=6, window_da=50, max_peaks=64)\\n\\n\\nclass LibraryIndex:\\n    \\"\\"\\"Store rows grouped by metric key, pre-filtered for trustworthiness.\\"\\"\\"\\n\\n    def __init__(self, store, key_of_row, cfg: LibraryConfig = LibraryConfig()):\\n        self.store, self.cfg = store, cfg\\n        meta = store.meta\\n        ok = (np.abs(meta.ppm.to_numpy()) <= cfg.max_ppm) & (meta.n_peaks.to_numpy() >= 3)\\n        rows = np.flatnonzero(ok)\\n        keys = key_of_row[rows]\\n        order = np.argsort(keys.astype(str), kind=\\"stable\\")\\n        rows, keys = rows[order], keys[order].astype(str)\\n        starts = np.r_[0, np.flatnonzero(keys[1:] != keys[:-1]) + 1]\\n        ends = np.r_[starts[1:], len(keys)]\\n        self.rows_of = {keys[s]: rows[s:e] for s, e in zip(starts, ends)}\\n        # priority for picking references: timsTOF first, then same-adduct (applied per query)\\n        self.is_timstof = meta.is_timstof.to_numpy()\\n        self.adduct = meta.adduct.astype(str).to_numpy()\\n        self.mode = meta.ionization_mode.astype(str).to_numpy()\\n        self._cache = {}\\n\\n    def _clean(self, row):\\n        hit = self._cache.get(row)\\n        if hit is None:\\n            mz, it = self.store.peaks(row)\\n            hit = clean_peaks_fast(mz, it, float(self.store.meta.precursor_mz.iat[row]), self.cfg.clean)\\n            if len(self._cache) < 400_000:\\n                self._cache[row] = hit\\n        return hit\\n\\n    def references(self, key, mode, adducts, excluded):\\n        rows = self.rows_of.get(key)\\n        if rows is None:\\n            return rows\\n        if excluded is not None and len(excluded):\\n            rows = rows[~np.isin(rows, excluded)]\\n        rows = rows[self.mode[rows] == mode]\\n        if len(rows) > self.cfg.max_refs_per_candidate:\\n            prio = self.is_timstof[rows].astype(int) * 2 + np.isin(self.adduct[rows], list(adducts)).astype(int)\\n            rows = rows[np.argsort(-prio, kind=\\"stable\\")[: self.cfg.max_refs_per_candidate]]\\n        return rows\\n\\n    def score(self, query_spectra, key, excluded):\\n        \\"\\"\\"Best and mean per-spectrum match using ONLY same-mode references.\\n\\n        A molecule can have spectra collected in different ionization modes.\\n        Each query spectrum must be compared exclusively with references from\\n        its own mode; using query_spectra[0][\\"mode\\"] for every spectrum\\n        incorrectly compares positive and negative ionizations. If a mode has\\n        no valid references, its query spectra contribute zero to the mean.\\n        \\"\\"\\"\\n        if not query_spectra:\\n            return 0.0, 0.0, 0\\n        by_mode = {}\\n        for index, spectrum in enumerate(query_spectra):\\n            by_mode.setdefault(spectrum[\\"mode\\"], []).append(index)\\n        per_spec = np.zeros(len(query_spectra), dtype=np.float64)\\n        used_rows = set()\\n        for mode, indices in by_mode.items():\\n            rows = self.references(\\n                key, mode,\\n                {query_spectra[i][\\"adduct\\"] for i in indices},\\n                excluded,\\n            )\\n            if rows is None or len(rows) == 0:\\n                continue\\n            references = [self._clean(int(row)) for row in rows]\\n            used_rows.update(int(row) for row in rows)\\n            for i in indices:\\n                query = query_spectra[i]\\n                per_spec[i] = max(\\n                    entropy_similarity(\\n                        query[\\"mz_c\\"], query[\\"it_c\\"], rm, ri,\\n                        self.cfg.tol_da, 0.0,\\n                    )\\n                    for rm, ri in references\\n                )\\n        if not used_rows:\\n            return 0.0, 0.0, 0\\n        return float(per_spec.max()), float(per_spec.mean()), len(used_rows)\\n", "src/casmi/channels/prefilter.py": "\\"\\"\\"Numerically safe, deterministic top-k prefilter for MS/MS analog retrieval.\\n\\nAll eligibility restrictions, especially structure-level knockout rows, must\\nbe enforced *before* top-k selection. Otherwise withheld/self-references can\\nconsume the entire shortlist and silently suppress genuine analogs.\\n\\nThis module requires only NumPy, so offline numerical regression tests need\\nneither the full spectrum store nor GPU/chemistry dependencies.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\n\\ndef eligible_top_k(scores, allowed, k: int) -> np.ndarray:\\n    \\"\\"\\"Indices of up to k highest-scoring eligible, positive, finite entries.\\n\\n    Ties are resolved by ascending original index (stable, deterministic);\\n    inputs are never mutated. The order is descending score, then index.\\n    \\"\\"\\"\\n    scores = np.asarray(scores)\\n    allowed = np.asarray(allowed)\\n    if scores.ndim != 1 or allowed.ndim != 1 or scores.shape != allowed.shape:\\n        raise ValueError(\\"scores and allowed must be aligned 1D vectors\\")\\n    if not (np.issubdtype(scores.dtype, np.floating)\\n            or np.issubdtype(scores.dtype, np.integer)):\\n        raise ValueError(\\"scores must be real numeric values\\")\\n    if allowed.dtype != np.dtype(bool):\\n        raise ValueError(\\"allowed must be a boolean vector\\")\\n    if type(k) is not int or k < 0:\\n        raise ValueError(\\"k must be a nonnegative integer\\")\\n    if k == 0 or scores.size == 0:\\n        return np.empty(0, dtype=np.int64)\\n    eligible = np.flatnonzero(allowed & np.isfinite(scores) & (scores > 0))\\n    if eligible.size <= k:\\n        return eligible[np.lexsort((eligible, -scores[eligible]))]\\n    # Use the score of the kth-best candidate as a cutoff, then break ties\\n    # deterministically by index without sorting the whole reference library.\\n    local_values = scores[eligible]\\n    cutoff = np.partition(local_values, local_values.size - k)[local_values.size - k]\\n    better = eligible[local_values > cutoff]\\n    tied = eligible[local_values == cutoff]\\n    chosen = np.concatenate((better, tied[: k - better.size]))\\n    return chosen[np.lexsort((chosen, -scores[chosen]))]\\n", "src/casmi/channels/reference_selection.py": "\\"\\"\\"Reference sampling for the CASMI D2 analog index.\\n\\nThe historical policy selects up to N spectra per (standard key, adduct),\\npreferring timsTOF rows. This can eliminate every independent reference\\nacquired in another library before a cross-library C1 knockout is applied.\\n\\nThe opt-in source_balanced policy retains the best row, reserves one\\nnon-timsTOF source (if the best is timsTOF and N >= 2), then chooses distinct\\nlibraries before filling unused slots by quality. It controls reference\\ncoverage; it does NOT guarantee improved MRR or metric-key equivalence.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nPOLICIES = (\\"legacy\\", \\"source_balanced\\")\\nSELECTION_VERSION = \\"casmi-d2-reference-selection-v1\\"\\n\\n\\ndef select_reference_rows(meta: pd.DataFrame, key_of_row, *,\\n                          per_structure_adduct: int = 3,\\n                          policy: str = \\"legacy\\") -> np.ndarray:\\n    \\"\\"\\"Return sorted, unique spectrum row IDs for building the D2 index.\\n\\n    The legacy branch intentionally reproduces AnalogIndex._select,\\n    including its global timsTOF and peak-count priority and stable ties.\\n    Source balancing does not raise the existing per-(key, adduct) cap.\\n    \\"\\"\\"\\n    if policy not in POLICIES:\\n        raise ValueError(f\\"Unknown selection policy {policy!r}\\")\\n    if type(per_structure_adduct) is not int or per_structure_adduct < 1:\\n        raise ValueError(\\"per_structure_adduct must be a positive integer\\")\\n    keys = np.asarray(key_of_row)\\n    if len(meta) != len(keys):\\n        raise ValueError(\\"Reference key and metadata lengths differ\\")\\n    needed = {\\"test_adduct\\", \\"ppm\\", \\"n_peaks\\", \\"is_timstof\\", \\"ingest_lib\\", \\"adduct\\"}\\n    if not needed.issubset(meta.columns):\\n        raise ValueError(f\\"Missing index metadata: {sorted(needed - set(meta.columns))}\\")\\n    ok = (meta.test_adduct.to_numpy(dtype=bool) &\\n          (np.abs(meta.ppm.to_numpy(dtype=float)) <= 20) &\\n          (meta.n_peaks.to_numpy(dtype=float) >= 5))\\n    rows = np.flatnonzero(ok)\\n    if len(rows) == 0:\\n        return np.empty(0, dtype=np.int64)\\n    t = pd.DataFrame({\\"row\\": rows})\\n    t[\\"key\\"] = keys[rows]\\n    t[\\"adduct\\"] = meta.adduct.to_numpy()[rows]\\n    t[\\"prio\\"] = (\\n        meta.is_timstof.to_numpy(dtype=bool)[rows].astype(np.int64) * 1000\\n        + np.clip(meta.n_peaks.to_numpy(dtype=float)[rows], 0, 999)\\n    )\\n    t = t.sort_values(\\"prio\\", ascending=False, kind=\\"stable\\")\\n    if policy == \\"legacy\\":\\n        # Exact historical selection path.\\n        t = t.groupby([\\"key\\", \\"adduct\\"], observed=True, sort=False).head(per_structure_adduct)\\n        return np.sort(t.row.to_numpy(dtype=np.int64))\\n\\n    t[\\"lib\\"] = meta.ingest_lib.astype(str).to_numpy()[t.row.to_numpy()]\\n    t[\\"timstof\\"] = meta.is_timstof.to_numpy(dtype=bool)[t.row.to_numpy()]\\n    selected = []\\n    for _, group in t.groupby([\\"key\\", \\"adduct\\"], observed=True, sort=False):\\n        group_rows = group.row.to_numpy(dtype=np.int64)\\n        if len(group_rows) <= per_structure_adduct:\\n            selected.extend(group_rows)\\n            continue\\n        libs = group.lib.to_numpy()\\n        timstof = group.timstof.to_numpy(dtype=bool)\\n        chosen = [0]  # Never displace the highest-quality reference.\\n        used_libs = {libs[0]}\\n        if per_structure_adduct > 1 and timstof[0]:\\n            for i in range(1, len(group_rows)):\\n                if not timstof[i] and libs[i] not in used_libs:\\n                    chosen.append(i)\\n                    used_libs.add(libs[i])\\n                    break\\n        for i in range(1, len(group_rows)):\\n            if len(chosen) == per_structure_adduct:\\n                break\\n            if libs[i] not in used_libs:\\n                chosen.append(i)\\n                used_libs.add(libs[i])\\n        if len(chosen) < per_structure_adduct:\\n            known = set(chosen)\\n            for i in range(1, len(group_rows)):\\n                if len(chosen) == per_structure_adduct:\\n                    break\\n                if i not in known:\\n                    chosen.append(i)\\n        selected.extend(group_rows[chosen])\\n    return np.sort(np.asarray(selected, dtype=np.int64))\\n", "src/casmi/chem/__init__.py": "", "src/casmi/chem/canon.py": "\\"\\"\\"Structure identity exactly as the official metric computes it.\\n\\nThe metric canonicalises every SMILES with RDKit 2026.03.3\'s TautomerEnumerator and compares the\\nfirst InChIKey block. Our pool keys, deduplication and validation labels all go through the same\\nfunction, so \\"same key here\\" means \\"same key on the leaderboard\\".\\n\\"\\"\\"\\nimport warnings\\nfrom functools import lru_cache\\n\\nimport rdkit\\nfrom rdkit import Chem, RDLogger\\nfrom rdkit.Chem.MolStandardize import rdMolStandardize\\n\\nEXPECTED_RDKIT = \\"2026.03.3\\"\\nif rdkit.__version__ != EXPECTED_RDKIT:\\n    warnings.warn(f\\"RDKit {rdkit.__version__} != metric\'s {EXPECTED_RDKIT}; InChIKey14 keys may not match the leaderboard.\\")\\n\\nRDLogger.DisableLog(\\"rdApp.*\\")\\n_ENUMERATOR = rdMolStandardize.TautomerEnumerator()\\n\\n\\ndef ik14_from_mol(mol):\\n    try:\\n        return Chem.MolToInchiKey(_ENUMERATOR.Canonicalize(mol))[:14]\\n    except (ValueError, RuntimeError, TypeError):\\n        return None\\n\\n\\n@lru_cache(maxsize=500_000)\\ndef canonical_ik14(smiles):\\n    \\"\\"\\"Tautomer-canonical InChIKey14 (metric-equivalent), or None if RDKit cannot parse it.\\"\\"\\"\\n    if not isinstance(smiles, str):\\n        return None\\n    try:\\n        mol = Chem.MolFromSmiles(smiles)\\n    except (ValueError, RuntimeError, TypeError):\\n        return None\\n    return None if mol is None else ik14_from_mol(mol)\\n\\n\\ndef heavy_atom_key(smiles):\\n    \\"\\"\\"Heavy-atom composition, the metric\'s cheap pre-filter before tautomer canonicalisation.\\"\\"\\"\\n    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None\\n    if mol is None:\\n        return None\\n    counts = {}\\n    for atom in mol.GetAtoms():\\n        if atom.GetSymbol() != \\"H\\":\\n            counts[atom.GetSymbol()] = counts.get(atom.GetSymbol(), 0) + 1\\n    return tuple(sorted(counts.items()))\\n\\n\\n@lru_cache(maxsize=1_000_000)\\ndef skeleton_key(smiles):\\n    \\"\\"\\"Heavy-atom connectivity only: bonds single, no aromaticity, charges, H or stereo.\\n    Tautomers (and protomers) share it, so two structures can only have the same metric key if\\n    they share this key. It costs ~0.2 ms against 10\\u2013300 ms for tautomer canonicalisation, so it\\n    serves as an exact pre-filter before the expensive comparison.\\"\\"\\"\\n    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None\\n    if mol is None:\\n        return None\\n    rw = Chem.RWMol(Chem.RemoveHs(mol, sanitize=False))\\n    for atom in rw.GetAtoms():\\n        atom.SetFormalCharge(0)\\n        atom.SetIsAromatic(False)\\n        atom.SetNoImplicit(True)\\n        atom.SetNumExplicitHs(0)\\n        atom.SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)\\n    for bond in rw.GetBonds():\\n        bond.SetBondType(Chem.BondType.SINGLE)\\n        bond.SetIsAromatic(False)\\n        bond.SetStereo(Chem.BondStereo.STEREONONE)\\n    return Chem.MolToSmiles(rw, canonical=True)\\n", "src/casmi/chem/keys.py": "\\"\\"\\"Lazy, persistent map from pool key (standard InChIKey14) to metric key (tautomer-canonical).\\n\\nMeasured: the two keys agree for only 88.5% of COCONUT structures (natural products carry many\\ntautomerisable groups), so scoring on the standard key would be wrong ~1 time in 9. Computing the\\nmetric key costs 10\\u2013300 ms per structure, so we compute it only where it matters \\u2014 query truths,\\nthe head of each ranked list, and output deduplication \\u2014 and cache it on disk across runs.\\n\\"\\"\\"\\nfrom pathlib import Path\\n\\nimport pandas as pd\\n\\nfrom casmi.chem.canon import canonical_ik14\\n\\n\\nclass MetricKeys:\\n    def __init__(self, cache_path: Path):\\n        self.path = Path(cache_path)\\n        self.map = {}\\n        if self.path.exists():\\n            t = pd.read_parquet(self.path)\\n            self.map = dict(zip(t.pool_key, t.metric_key))\\n        self._dirty = 0\\n\\n    def get(self, pool_key: str, smiles: str):\\n        key = self.map.get(pool_key)\\n        if key is None and pool_key not in self.map:\\n            key = canonical_ik14(smiles)\\n            self.map[pool_key] = key\\n            self._dirty += 1\\n        return key\\n\\n    def save(self):\\n        if self._dirty:\\n            pd.DataFrame({\\"pool_key\\": list(self.map), \\"metric_key\\": list(self.map.values())}).to_parquet(self.path, index=False)\\n            self._dirty = 0\\n\\n    def dedupe_ranked(self, pool_keys, smiles_of, top_n=25):\\n        \\"\\"\\"Map a ranked list of pool keys to metric keys, drop unparseable and duplicate structures,\\n        keep the first `top_n`. Returns [(pool_key, metric_key)] in rank order.\\"\\"\\"\\n        out, seen = [], set()\\n        for pk in pool_keys:\\n            mk = self.get(pk, smiles_of(pk))\\n            if mk is None or mk in seen:\\n                continue\\n            seen.add(mk)\\n            out.append((pk, mk))\\n            if len(out) == top_n:\\n                break\\n        return out\\n", "src/casmi/chem/masses.py": "\\"\\"\\"Monoisotopic masses, formula parsing and adduct arithmetic.\\n\\nEverything here is plain Python/NumPy (no RDKit) so it can run over millions of rows.\\nThe adduct parser handles every string in train (`[2M+Na]+`, `[M-2H2O+H]+`, `[M+CH2O2-H]-`, ...),\\nnot only the ten test adducts, because library spectra with other adducts are still evidence.\\n\\"\\"\\"\\nimport re\\nfrom functools import lru_cache\\n\\nimport numpy as np\\n\\nELECTRON = 0.000548579909\\nPROTON = 1.007276466621\\n\\nMONOISOTOPIC = {\\n    \\"H\\": 1.00782503207, \\"D\\": 2.0141017778, \\"C\\": 12.0, \\"N\\": 14.0030740048, \\"O\\": 15.99491461956,\\n    \\"P\\": 30.97376163, \\"S\\": 31.97207100, \\"F\\": 18.99840322, \\"Cl\\": 34.96885268, \\"Br\\": 78.9183371,\\n    \\"I\\": 126.904473, \\"Si\\": 27.9769265325, \\"B\\": 11.0093054, \\"Se\\": 79.9165213, \\"Na\\": 22.9897692809,\\n    \\"K\\": 38.96370668, \\"Li\\": 7.01600455, \\"Mg\\": 23.9850417, \\"Ca\\": 39.96259098, \\"Fe\\": 55.9349375,\\n    \\"Cu\\": 62.9295975, \\"Zn\\": 63.9291422, \\"As\\": 74.9215965, \\"Al\\": 26.98153863, \\"Co\\": 58.933195,\\n    \\"Ni\\": 57.9353429, \\"Mn\\": 54.9380451, \\"Cr\\": 51.9405075, \\"Sn\\": 119.9021947, \\"Hg\\": 201.970643,\\n    \\"Pt\\": 194.9647911, \\"Ag\\": 106.905097, \\"Au\\": 196.9665687, \\"Gd\\": 157.9241039, \\"Ge\\": 73.9211778,\\n    \\"Ti\\": 47.9479463, \\"V\\": 50.9439595, \\"Mo\\": 97.9054082, \\"W\\": 183.9509312, \\"Te\\": 129.9062244,\\n    \\"Ru\\": 101.9043493, \\"Rh\\": 102.905504, \\"Pd\\": 105.903486, \\"Sb\\": 120.9038157, \\"Bi\\": 208.9803987,\\n    \\"Ba\\": 137.9052472, \\"Sr\\": 87.9056121, \\"Tl\\": 204.9744275, \\"Pb\\": 207.9766521, \\"Cs\\": 132.905451933,\\n    \\"Rb\\": 84.911789738,\\n}\\n\\n_ELEMENT_RE = re.compile(r\\"([A-Z][a-z]?)(\\\\d*)\\")\\n\\n\\n@lru_cache(maxsize=200_000)\\ndef formula_counts(formula: str) -> dict:\\n    \\"\\"\\"\'C15H10O5\' -> {\'C\': 15, \'H\': 10, \'O\': 5}. Ignores charge suffixes such as \'+\' or \'-\'.\\"\\"\\"\\n    counts = {}\\n    for element, n in _ELEMENT_RE.findall(formula.rstrip(\\"+-\\")):\\n        counts[element] = counts.get(element, 0) + (int(n) if n else 1)\\n    return counts\\n\\n\\n@lru_cache(maxsize=200_000)\\ndef formula_mass(formula: str) -> float:\\n    \\"\\"\\"Monoisotopic neutral mass of a molecular formula; NaN if it contains an element we do not\\n    tabulate (organometallics such as Ir or Os, far outside this competition\'s chemistry).\\"\\"\\"\\n    counts = formula_counts(formula)\\n    if any(e not in MONOISOTOPIC for e in counts):\\n        return float(\\"nan\\")\\n    return sum(MONOISOTOPIC[e] * n for e, n in counts.items())\\n\\n\\n# ----------------------------------------------------------------------------- adducts\\n_ADDUCT_RE = re.compile(r\\"^\\\\[(\\\\d*)M((?:[+-]\\\\d*[A-Za-z0-9]+)*)\\\\](\\\\d*)([+-])$\\")\\n_PART_RE = re.compile(r\\"([+-])(\\\\d*)([A-Za-z][A-Za-z0-9]*)\\")\\n# Common adduct shorthands that are not literal formulas\\n_ALIASES = {\\"ACN\\": \\"C2H3N\\", \\"FA\\": \\"CH2O2\\", \\"Hac\\": \\"C2H4O2\\", \\"HAc\\": \\"C2H4O2\\", \\"TFA\\": \\"C2HF3O2\\", \\"IsoProp\\": \\"C3H8O\\"}\\n\\nTEST_ADDUCTS = (\\"[M+H]+\\", \\"[M+NH4]+\\", \\"[M-H2O+H]+\\", \\"[M-2H2O+H]+\\", \\"[M+Na]+\\", \\"[M+K]+\\",\\n                \\"[M-H]-\\", \\"[M-H2O-H]-\\", \\"[M+CH2O2-H]-\\", \\"[M+Cl]-\\")\\n\\n\\n@lru_cache(maxsize=4096)\\ndef parse_adduct(adduct: str):\\n    \\"\\"\\"Return (n_molecules, mass_shift, charge) with ion_mz = (n*M + mass_shift) / |charge|.\\n\\n    `mass_shift` already includes the electron mass for the charge state. Returns None when the\\n    string is not a recognisable adduct (the caller decides whether to drop the spectrum).\\n    \\"\\"\\"\\n    if not isinstance(adduct, str):\\n        return None\\n    m = _ADDUCT_RE.match(adduct.strip())\\n    if not m:\\n        return None\\n    n_mol = int(m.group(1)) if m.group(1) else 1\\n    z = int(m.group(3)) if m.group(3) else 1\\n    charge = z if m.group(4) == \\"+\\" else -z\\n    shift = 0.0\\n    for sign, mult, part in _PART_RE.findall(m.group(2)):\\n        mass = formula_mass(_ALIASES.get(part, part))\\n        if np.isnan(mass):\\n            return None\\n        shift += (1 if sign == \\"+\\" else -1) * (int(mult) if mult else 1) * mass\\n    shift -= charge * ELECTRON          # a cation has lost electrons, an anion gained them\\n    return n_mol, shift, charge\\n\\n\\ndef ion_mz(neutral_mass: float, adduct: str) -> float:\\n    \\"\\"\\"Ion m/z for a neutral mass and adduct; NaN when the adduct string is not recognised.\\"\\"\\"\\n    parsed = parse_adduct(adduct)\\n    if parsed is None:\\n        return np.nan\\n    n, shift, charge = parsed\\n    return (n * neutral_mass + shift) / abs(charge)\\n\\n\\ndef neutral_mass(precursor_mz, adduct: str):\\n    \\"\\"\\"Invert `ion_mz`. Works on scalars or NumPy arrays of precursor m/z for one adduct.\\"\\"\\"\\n    parsed = parse_adduct(adduct)\\n    if parsed is None:\\n        return np.nan if np.ndim(precursor_mz) == 0 else np.full(np.shape(precursor_mz), np.nan)\\n    n, shift, charge = parsed\\n    return (np.asarray(precursor_mz, dtype=float) * abs(charge) - shift) / n\\n\\n\\ndef ppm(measured, theoretical):\\n    return (np.asarray(measured, dtype=float) - theoretical) / theoretical * 1e6\\n", "src/casmi/chem/npscore.py": "\\"\\"\\"RDKit natural-product likeness (Ertl et al. 2008, rdkit.Contrib.NP_Score), cached per key.\\n\\nComputed from the structure for every candidate alike: unlike COCONUT\'s own np_likeness column it\\ncarries no information about which database a candidate came from.\\n\\"\\"\\"\\n_MODEL = None\\n_CACHE = {}\\n\\n\\ndef np_likeness(key, smiles):\\n    global _MODEL\\n    v = _CACHE.get(key)\\n    if v is None:\\n        from rdkit import Chem\\n        from rdkit.Contrib.NP_Score import npscorer\\n        if _MODEL is None:\\n            _MODEL = npscorer.readNPModel()\\n        mol = Chem.MolFromSmiles(smiles)\\n        v = float(npscorer.scoreMol(mol, _MODEL)) if mol is not None else 0.0\\n        _CACHE[key] = v\\n    return v\\n", "src/casmi/config.py": "\\"\\"\\"Paths and runtime environment. Resolves the same code for local runs and Kaggle notebooks.\\"\\"\\"\\nimport os\\nfrom pathlib import Path\\n\\nON_KAGGLE = Path(\\"/kaggle/input\\").exists()\\nPROJECT_ROOT = Path(__file__).resolve().parents[2]\\n\\n\\ndef _find_competition_data() -> Path:\\n    if ON_KAGGLE:\\n        hits = sorted(Path(\\"/kaggle/input\\").rglob(\\"test.parquet\\"))\\n        if hits:\\n            return hits[0].parent\\n    return Path(os.environ.get(\\"CASMI_DATA\\", \\"C:/Users/MONSTER/kaggle-data/casmi26\\"))\\n\\n\\nDATA_DIR = _find_competition_data()\\n# Built artifacts (spectrum store, pool, models). Large, never committed.\\n# /tmp on Kaggle: scratch disk, not saved as notebook output (which would slow every commit)\\nARTIFACT_DIR = Path(os.environ.get(\\"CASMI_ARTIFACTS\\",\\n                                   \\"/tmp/casmi_artifacts\\" if ON_KAGGLE else str(DATA_DIR / \\"artifacts\\")))\\nOUTPUT_DIR = Path(\\"/kaggle/working\\") if ON_KAGGLE else PROJECT_ROOT / \\"runs\\"\\nSEED = 42\\n\\n# Candidate-pool variant: \\"\\" = train \\u222a COCONUT (default), \\"_pcl\\" = + PubChemLite. All pool-derived\\n# artifacts carry the tag, so variants can live side by side and be compared on the same queries.\\nPOOL_TAG = os.environ.get(\\"CASMI_POOL_TAG\\", \\"\\")\\nPOOL_FILE = f\\"pool{POOL_TAG}.parquet\\"\\nPOOL_FP_FILE = f\\"pool_fp2048{POOL_TAG}.npy\\"\\nPOOL_FP_KEYS_FILE = f\\"pool_fp_keys{POOL_TAG}.parquet\\"\\n", "src/casmi/engine.py": "\\"\\"\\"Per-molecule engine: neutral mass -> candidates -> channel features -> ranking.\\n\\nThe baseline ranks by D1 (direct library similarity), breaking ties by mass error. Later stages\\nreplace `rank` with the learned ranker; the candidate/feature interface stays the same, so every\\nchannel is validated through exactly the code path the submission uses.\\n\\"\\"\\"\\nfrom dataclasses import dataclass, field\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom casmi.channels.analog import AnalogConfig, propagate\\nfrom casmi.channels.library import LibraryConfig, LibraryIndex\\nfrom casmi.chem.masses import neutral_mass\\nfrom casmi.spectra.clean import clean_peaks_fast\\n\\n\\n@dataclass(frozen=True)\\nclass EngineConfig:\\n    tol_ppm: float = 10.0\\n    mass_offset_ppm: float = 0.0      # calibrated instrument offset (measured +1.5 ppm on np-examples)\\n    top_n: int = 25\\n    library: LibraryConfig = field(default_factory=LibraryConfig)\\n    analog: AnalogConfig = field(default_factory=AnalogConfig)\\n    generate: bool = False            # Class-3 hedge: add biotransformed analogs as candidates\\n    gen_max_analogs: int = 15\\n    gen_cap: int = 300\\n    gen_consensus: bool = False       # v17 features: seed / route consensus and transform type\\n    gen_expand: bool = False          # research opt-in, legacy-preserving ordered-pair expansion\\n    np_score: bool = False            # RDKit NP-likeness for every candidate (chemistry, not membership)\\n\\n\\ndef molecule_mass(spectra, offset_ppm=0.0):\\n    masses = [float(neutral_mass(s[\\"precursor_mz\\"], s[\\"adduct\\"])) for s in spectra]\\n    masses = [m for m in masses if np.isfinite(m)]\\n    m = float(np.median(masses)) if masses else np.nan\\n    return m / (1 + offset_ppm * 1e-6)\\n\\n\\nclass BaselineEngine:\\n    def __init__(self, pool, library: LibraryIndex, cfg: EngineConfig = EngineConfig(), analog=None, fps=None):\\n        \\"\\"\\"`analog` (AnalogIndex) and `fps` (packed pool fingerprints) enable the D2 channel.\\"\\"\\"\\n        self.pool, self.library, self.cfg = pool, library, cfg\\n        self.analog, self.fps = analog, fps\\n\\n    def features(self, spectra, excluded_rows=None, excluded_keys=None) -> pd.DataFrame:\\n        for s in spectra:   # clean once per query, with the library\'s cleaning settings\\n            s[\\"mz_c\\"], s[\\"it_c\\"] = clean_peaks_fast(s[\\"mz\\"], s[\\"it\\"], s[\\"precursor_mz\\"], self.library.cfg.clean)\\n        mass = molecule_mass(spectra, self.cfg.mass_offset_ppm)\\n        win = self.pool.window(mass, self.cfg.tol_ppm, excluded_keys)\\n        keys = self.pool.key[win.idx]\\n        d1 = np.array([self.library.score(spectra, k, excluded_rows) for k in keys]).reshape(-1, 3)\\n        f = pd.DataFrame({\\"pool_key\\": keys, \\"ppm\\": win.ppm, \\"d1_max\\": d1[:, 0], \\"d1_mean\\": d1[:, 1],\\n                          \\"d1_nref\\": d1[:, 2].astype(int), \\"n_cand\\": len(keys)})\\n        analogs = None\\n        if self.analog is not None and len(keys):\\n            analogs = self.analog.search(spectra, mass, excluded_rows)\\n            for name, v in propagate(analogs, win.idx, self.pool.mass[win.idx], self.fps,\\n                                     self.pool.row_of, self.cfg.analog.powers).items():\\n                f[name] = v\\n        if self.cfg.generate:\\n            if analogs is None and self.analog is not None:\\n                analogs = self.analog.search(spectra, mass, excluded_rows)\\n            f = f.assign(gen=0, gen_sim=0.0, gen_depth=0)\\n            if self.cfg.gen_expand:\\n                f = f.assign(audit_legacy=1)\\n            if self.cfg.gen_consensus:\\n                f = f.assign(gen_sim_sum=0.0, gen_n_seeds=0, gen_n_paths=0, gen_t1=-1, gen_t2=-1)\\n            g = self._generated(analogs or {}, mass, excluded_keys or set())\\n            if len(g):\\n                f = pd.concat([f, g], ignore_index=True)\\n                f[\\"n_cand\\"] = len(f)\\n        if self.cfg.np_score and len(f):\\n            from casmi.chem.npscore import np_likeness\\n            f[\\"np_score\\"] = [np_likeness(k, self.pool.smiles_of(k)) for k in f.pool_key]\\n        return f\\n\\n    def _generated(self, analogs, mass, excluded_keys):\\n        \\"\\"\\"Biotransformed analogs at the query mass that are not already pool candidates.\\"\\"\\"\\n        from rdkit import Chem\\n        from rdkit.Chem.Descriptors import ExactMolWt\\n        from casmi.generate.transforms import propose, propose_agg\\n        from casmi.pool.pool import generated_candidate_identity\\n        order = sorted(analogs.items(), key=lambda kv: -kv[1][0])[: self.cfg.gen_max_analogs]\\n        seeds = [(self.pool.smiles_of(k), sim, am) for k, (sim, am) in order if k in self.pool.row_of]\\n        rows, seen = [], {}\\n        if self.cfg.gen_consensus:\\n            if self.cfg.gen_expand:\\n                from casmi.generate.expanded import expanded_proposals\\n                legacy_seeds = [(self.pool.smiles_of(k), sim, am) for k, (sim, am) in order[:15] if k in self.pool.row_of]\\n                props = expanded_proposals(seeds, mass, self.cfg.tol_ppm, self.cfg.gen_max_analogs, self.cfg.gen_cap,\\n                                           legacy_analogs=legacy_seeds)\\n                items = sorted(props.items(), key=lambda kv: (not kv[1][\\"audit_legacy\\"], -kv[1][\\"sim_max\\"]))\\n            else:\\n                props = propose_agg(seeds, mass, self.cfg.tol_ppm, self.cfg.gen_max_analogs, self.cfg.gen_cap)\\n                items = sorted(props.items(), key=lambda kv: -kv[1][\\"sim_max\\"])\\n        else:\\n            if self.cfg.gen_expand:\\n                raise ValueError(\\"Expanded generation requires consensus features\\")\\n            items = [(smi, dict(sim_max=sim, depth=path.count(\\"+\\") + 1))\\n                     for smi, sim, path in propose(seeds, mass, self.cfg.tol_ppm, self.cfg.gen_max_analogs, self.cfg.gen_cap)]\\n        for smi, r in items:\\n            mol = Chem.MolFromSmiles(smi)\\n            ik = Chem.MolToInchiKey(mol)[:14] if mol is not None else \\"\\"\\n            if not ik or (ik in self.pool.row_of and ik not in excluded_keys):\\n                continue\\n            if ik in seen:                                     # same structure reached as another SMILES\\n                if self.cfg.gen_consensus:\\n                    row = rows[seen[ik]]\\n                    if self.cfg.gen_expand and row[-1] != r[\\"audit_legacy\\"]:\\n                        continue  # Do not alter legacy candidate features with new routes.\\n                    row[5] += r[\\"sim_sum\\"]\\n                    row[6] += r[\\"n_seeds\\"]\\n                    row[7] += r[\\"n_paths\\"]\\n                continue\\n            seen[ik] = len(rows)\\n            generated_key, smi = generated_candidate_identity(mol)\\n            row = [generated_key, smi, (mass - ExactMolWt(mol)) / ExactMolWt(mol) * 1e6, r[\\"sim_max\\"], r[\\"depth\\"]]\\n            if self.cfg.gen_consensus:\\n                row += [r[\\"sim_sum\\"], r[\\"n_seeds\\"], r[\\"n_paths\\"], r[\\"t1\\"], r[\\"t2\\"]]\\n            if self.cfg.gen_expand:\\n                row += [r[\\"audit_legacy\\"]]\\n            rows.append(row)\\n        if not rows:\\n            return pd.DataFrame()\\n        cols = [\\"pool_key\\", \\"gen_smiles\\", \\"ppm\\", \\"gen_sim\\", \\"gen_depth\\"]\\n        if self.cfg.gen_consensus:\\n            cols += [\\"gen_sim_sum\\", \\"gen_n_seeds\\", \\"gen_n_paths\\", \\"gen_t1\\", \\"gen_t2\\"]\\n        if self.cfg.gen_expand:\\n            cols += [\\"audit_legacy\\"]\\n        g = pd.DataFrame(rows, columns=cols)\\n        self.pool.register_generated(g.pool_key, g.gen_smiles)\\n        g = g.drop(columns=\\"gen_smiles\\").assign(d1_max=0.0, d1_mean=0.0, d1_nref=0, gen=1)\\n        if analogs:\\n            keys = [k for k in analogs if k in self.pool.row_of]\\n            local = np.vstack([self.pool.packed_fps(g.pool_key.to_numpy(), self.fps),\\n                               self.fps[[self.pool.row_of[k] for k in keys]]])\\n            row_of = {k: len(g) + i for i, k in enumerate(keys)}\\n            gm = mass / (1 + g.ppm.to_numpy() * 1e-6)\\n            for name, v in propagate({k: analogs[k] for k in keys}, np.arange(len(g)), gm, local, row_of,\\n                                     self.cfg.analog.powers).items():\\n                g[name] = v\\n        return g\\n\\n    def rank(self, feats: pd.DataFrame, depth=60):\\n        \\"\\"\\"Ranked pool keys (deeper than 25, so output deduplication can refill the list).\\"\\"\\"\\n        if feats.empty:\\n            return []\\n        primary = feats.d1_max.to_numpy()\\n        secondary = feats.d1_mean.to_numpy()\\n        if \\"an_p4\\" in feats:   # interim rule until the learned ranker: best of direct and propagated evidence\\n            primary = np.maximum(primary, feats.an_p4.to_numpy())\\n            secondary = feats.an_p2.to_numpy()\\n        order = np.lexsort((np.abs(feats.ppm.to_numpy()), -secondary, -primary))\\n        return feats.pool_key.to_numpy()[order][:depth].tolist()\\n", "src/casmi/generate/__init__.py": "", "src/casmi/generate/expanded.py": "\\"\\"\\"Explicitly budgeted proposal expansion preserving historical candidates/features.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom .transforms import propose_agg\\n\\n\\ndef expanded_proposals(analogs, query_mass, tol_ppm=10.0, max_analogs=30, cap=900, *, legacy_analogs=None):\\n    if type(max_analogs) is not int or max_analogs < 15 or type(cap) is not int or cap < 300:\\n        raise ValueError(\\"Expanded policy must preserve the 15-seed / 300-candidate legacy budget\\")\\n    legacy = propose_agg(analogs[:15] if legacy_analogs is None else legacy_analogs, query_mass, tol_ppm, 15, 300)\\n    expanded = propose_agg(analogs, query_mass, tol_ppm, max_analogs, cap, ordered_pairs=True)\\n    out = {smiles: {**record, \\"audit_legacy\\": 1} for smiles, record in legacy.items()}\\n    for smiles, record in expanded.items():\\n        if smiles not in out and len(out) < cap:\\n            out[smiles] = {**record, \\"audit_legacy\\": 0}\\n    return out\\n", "src/casmi/generate/transforms.py": "\\"\\"\\"Class-3 hedge: propose unseen structures by applying one or two common natural-product\\nbiotransformations to spectral analogs of the query.\\n\\nA Class-3 molecule is in no database, so retrieval scores 0 on it by construction. Natural products\\ncome in families, though: the analog search (D2) often finds a library structure whose spectrum\\nmatches the query up to a precursor shift \\u0394. When \\u0394 equals a common biotransformation (methylation,\\nhydroxylation, glycosylation, acylation, ...) the query is plausibly that analog with the\\nmodification at one of its sites. Every site gives one candidate; the metric ignores stereo\\n(InChIKey block 1), so connectivity is all that has to be right.\\n\\n`propose(analogs, query_mass)` returns generated SMILES with the analog they came from; scoring\\nand placement in the ranked list happen elsewhere (they need D3/D4 and a confidence gate).\\n\\"\\"\\"\\nfrom dataclasses import dataclass\\n\\nimport numpy as np\\nfrom rdkit import Chem, RDLogger\\nfrom rdkit.Chem import AllChem\\nfrom rdkit.Chem.Descriptors import ExactMolWt\\n\\nRDLogger.DisableLog(\\"rdApp.*\\")\\n\\nHEX = \\"C1OC(CO)C(O)C(O)C1O\\"          # hexose (glucose/galactose connectivity)\\nDHEX = \\"C1OC(C)C(O)C(O)C1O\\"          # 6-deoxyhexose (rhamnose/fucose)\\nPENT = \\"C1OCC(O)C(O)C1O\\"             # pentose (pyranose)\\nGLCA = \\"C1OC(C(=O)O)C(O)C(O)C1O\\"     # glucuronic acid\\n\\n\\n@dataclass(frozen=True)\\nclass Transform:\\n    name: str\\n    delta: float\\n    smarts: tuple\\n\\n\\ndef _t(name, formula_delta, *smarts):\\n    return Transform(name, formula_delta, tuple(smarts))\\n\\n\\n# forward transforms; each is also applied in reverse where a removal SMARTS is given\\nTRANSFORMS = [\\n    _t(\\"methyl\\", 14.01565, \\"[OX2H1:1]>>[O:1]C\\", \\"[NX3;H1,H2;!$(N-C=O):1]>>[N:1]C\\", \\"[cH1:1]>>[c:1]C\\"),\\n    _t(\\"demethyl\\", -14.01565, \\"[O:1]-[CH3]>>[O:1]\\", \\"[N:1]-[CH3]>>[N:1]\\"),\\n    _t(\\"hydroxyl\\", 15.99491, \\"[cH1:1]>>[c:1]O\\", \\"[CX4;H1,H2:1]>>[C:1]O\\"),\\n    _t(\\"dehydroxyl\\", -15.99491, \\"[c:1]-[OH]>>[c:1]\\", \\"[CX4:1]-[OH]>>[C:1]\\"),\\n    _t(\\"methoxyl\\", 30.01056, \\"[cH1:1]>>[c:1]OC\\"),\\n    _t(\\"demethoxyl\\", -30.01056, \\"[c:1]-[OX2]-[CH3]>>[c:1]\\"),\\n    _t(\\"hydrogenation\\", 2.01565, \\"[C:1]=[C:2]>>[C:1]-[C:2]\\"),\\n    _t(\\"dehydrogenation\\", -2.01565, \\"[CX4;!H0:1]-[CX4;!H0:2]>>[C:1]=[C:2]\\", \\"[CX4;H1,H2:1]-[OH:2]>>[C:1]=[O:2]\\"),\\n    _t(\\"hydration\\", 18.01056, \\"[C:1]=[C:2]>>[C:1](O)-[C:2]\\"),\\n    _t(\\"dehydration\\", -18.01056, \\"[CX4;!H0:1]-[CX4:2]-[OH]>>[C:1]=[C:2]\\"),\\n    _t(\\"hexose\\", 162.05282, f\\"[OX2H1:1]>>[O:1]{HEX}\\"),\\n    _t(\\"dehexose\\", -162.05282, \\"[O:1]-[CH1;R1]1-[OX2;R1]-[CH1;R1](-[CH2]-[OH])-[CH1;R1](-[OH])-[CH1;R1](-[OH])-[CH1;R1]1-[OH]>>[O:1]\\"),\\n    _t(\\"deoxyhexose\\", 146.05791, f\\"[OX2H1:1]>>[O:1]{DHEX}\\"),\\n    _t(\\"dedeoxyhexose\\", -146.05791, \\"[O:1]-[CH1;R1]1-[OX2;R1]-[CH1;R1](-[CH3])-[CH1;R1](-[OH])-[CH1;R1](-[OH])-[CH1;R1]1-[OH]>>[O:1]\\"),\\n    _t(\\"pentose\\", 132.04226, f\\"[OX2H1:1]>>[O:1]{PENT}\\"),\\n    _t(\\"glucuronide\\", 176.03209, f\\"[OX2H1:1]>>[O:1]{GLCA}\\"),\\n    _t(\\"acetyl\\", 42.01057, \\"[OX2H1:1]>>[O:1]C(C)=O\\"),\\n    _t(\\"deacetyl\\", -42.01057, \\"[O:1]-C(=O)-[CH3]>>[O:1]\\"),\\n    _t(\\"malonyl\\", 86.00039, \\"[OX2H1:1]>>[O:1]C(=O)CC(=O)O\\"),\\n    _t(\\"sulfate\\", 79.95682, \\"[OX2H1:1]>>[O:1]S(=O)(=O)O\\"),\\n    _t(\\"prenyl\\", 68.06260, \\"[cH1:1]>>[c:1]CC=C(C)C\\", \\"[OX2H1:1]>>[O:1]CC=C(C)C\\"),\\n    _t(\\"coumaroyl\\", 146.03678, \\"[OX2H1:1]>>[O:1]C(=O)C=Cc1ccc(O)cc1\\"),\\n    _t(\\"caffeoyl\\", 162.03169, \\"[OX2H1:1]>>[O:1]C(=O)C=Cc1ccc(O)c(O)c1\\"),\\n    _t(\\"feruloyl\\", 176.04734, \\"[OX2H1:1]>>[O:1]C(=O)C=Cc1ccc(O)c(OC)c1\\"),\\n    _t(\\"galloyl\\", 152.01096, \\"[OX2H1:1]>>[O:1]C(=O)c1cc(O)c(O)c(O)c1\\"),\\n    # v2 (diagnose_c3.py: the formula changes the misses needed most often)\\n    _t(\\"homolog\\", 14.01565, \\"[CH2;!R:1]-[CH2;!R:2]>>[C:1]-C-[C:2]\\", \\"[CH3;!R:1]>>[C:1]C\\"),\\n    _t(\\"dehomolog\\", -14.01565, \\"[CH2;!R:1]-[CH2;!R]-[CH2;!R:2]>>[C:1]-[C:2]\\", \\"[CH2;!R:1]-[CH3]>>[C:1]\\"),\\n    _t(\\"carbonyl_reduction\\", 2.01565, \\"[#6X3;$([#6](-[#6])-[#6]):1]=[O:2]>>[C:1]-[O:2]\\"),\\n    _t(\\"carbonyl_to_ch2\\", -13.97926, \\"[#6X3;$([#6](-[#6])-[#6]):1]=[O]>>[C:1]\\"),\\n    _t(\\"decarboxyl\\", -43.98983, \\"[#6:1]-C(=O)-[OH]>>[#6:1]\\"),\\n    _t(\\"chloro\\", 33.96103, \\"[cH1:1]>>[c:1]Cl\\"),\\n    _t(\\"dechloro\\", -33.96103, \\"[#6:1]-Cl>>[#6:1]\\"),\\n    _t(\\"n_acetyl\\", 42.01057, \\"[NX3;H1,H2;!$(N-C=O):1]>>[N:1]C(C)=O\\"),\\n    _t(\\"c_hexose\\", 162.05282, f\\"[cH1:1]>>[c:1]{HEX}\\"),\\n    _t(\\"desulfate\\", -79.95682, \\"[O:1]-S(=O)(=O)-[OH]>>[O:1]\\"),\\n    _t(\\"demalonyl\\", -86.00039, \\"[O:1]-C(=O)-[CH2]-C(=O)-[OH]>>[O:1]\\"),\\n    _t(\\"depentose\\", -132.04226, \\"[O:1]-[CH1;R1]1-[OX2;R1]-[CH2;R1]-[CH1;R1](-[OH])-[CH1;R1](-[OH])-[CH1;R1]1-[OH]>>[O:1]\\"),\\n    _t(\\"deglucuronide\\", -176.03209, \\"[O:1]-[CH1;R1]1-[OX2;R1]-[CH1;R1](-C(=O)-[OH])-[CH1;R1](-[OH])-[CH1;R1](-[OH])-[CH1;R1]1-[OH]>>[O:1]\\"),\\n    _t(\\"degalloyl\\", -152.01096, \\"[O:1]-C(=O)-c1cc(O)c(O)c(O)c1>>[O:1]\\"),\\n    _t(\\"decoumaroyl\\", -146.03678, \\"[O:1]-C(=O)-C=C-c1ccc(O)cc1>>[O:1]\\"),\\n    _t(\\"decaffeoyl\\", -162.03169, \\"[O:1]-C(=O)-C=C-c1ccc(O)c(O)c1>>[O:1]\\"),\\n    _t(\\"deferuloyl\\", -176.04734, \\"[O:1]-C(=O)-C=C-c1ccc(O)c(OC)c1>>[O:1]\\"),\\n]\\n_RXN = {t.name: [AllChem.ReactionFromSmarts(s) for s in t.smarts] for t in TRANSFORMS}\\nDELTAS = np.array([t.delta for t in TRANSFORMS])\\n\\n\\ndef matching(delta, tol_da, *, ordered_pairs=False):\\n    \\"\\"\\"Single transforms and ordered pairs whose total mass change equals `delta`.\\"\\"\\"\\n    single = [(t,) for t in TRANSFORMS if abs(t.delta - delta) <= tol_da]\\n    pairs = [(a, b) for i, a in enumerate(TRANSFORMS) for b in TRANSFORMS[i:]\\n             if abs(a.delta + b.delta - delta) <= tol_da]\\n    if ordered_pairs:\\n        # Reaction order can change which sites exist in the intermediate.\\n        # Append reverse routes after historical routes to preserve their priority.\\n        return single + pairs + [(b, a) for a, b in pairs if a.name != b.name]\\n    return single + pairs\\n\\n\\ndef apply(mol, transform, cap=60):\\n    \\"\\"\\"Every single-site product of `transform` on `mol` (sanitised, unique by SMILES).\\"\\"\\"\\n    out = {}\\n    for rxn in _RXN[transform.name]:\\n        try:\\n            products = rxn.RunReactants((mol,), maxProducts=cap)\\n        except Exception:\\n            continue\\n        for (p,) in products:\\n            try:\\n                Chem.SanitizeMol(p)\\n                smi = Chem.MolToSmiles(p)\\n            except Exception:\\n                continue\\n            if smi and \\".\\" not in smi:\\n                out.setdefault(smi, p)\\n        if len(out) >= cap:\\n            break\\n    return out\\n\\n\\ndef propose(analogs, query_mass, tol_ppm=10.0, max_analogs=15, cap=300):\\n    \\"\\"\\"analogs: [(smiles, similarity, analog_mass)], best first.\\n    Returns [(smiles, similarity, transform names)] of structures at the query mass.\\"\\"\\"\\n    tol_da = query_mass * tol_ppm * 1e-6\\n    seen, out = set(), []\\n    for smi, sim, amass in analogs[:max_analogs]:\\n        paths = matching(query_mass - amass, tol_da * 2)\\n        if not paths:\\n            continue\\n        mol = Chem.MolFromSmiles(smi)\\n        if mol is None:\\n            continue\\n        for path in paths:\\n            level = {smi: mol}\\n            for t in path:\\n                nxt = {}\\n                for m in level.values():\\n                    nxt.update(apply(m, t))\\n                level = nxt\\n                if not level:\\n                    break\\n            for psmi, p in level.items():\\n                if psmi in seen:\\n                    continue\\n                seen.add(psmi)\\n                if abs(ExactMolWt(p) - query_mass) <= tol_da:\\n                    out.append((psmi, sim, \\"+\\".join(t.name for t in path)))\\n            if len(out) >= cap:\\n                return out\\n    return out\\n\\n\\nT_INDEX = {t.name: i for i, t in enumerate(TRANSFORMS)}\\n\\n\\ndef propose_agg(analogs, query_mass, tol_ppm=10.0, max_analogs=15, cap=300, *, ordered_pairs=False):\\n    \\"\\"\\"Like `propose`, but every product carries its consensus: how many distinct analog seeds and\\n    (seed, path) routes produce it. analogs: [(smiles, similarity, analog_mass)], best first.\\n    Returns {smiles: dict(sim_max, sim_sum, n_seeds, n_paths, depth, t1, t2)}.\\"\\"\\"\\n    tol_da = query_mass * tol_ppm * 1e-6\\n    out = {}\\n    for si, (smi, sim, amass) in enumerate(analogs[:max_analogs]):\\n        paths = matching(query_mass - amass, tol_da * 2, ordered_pairs=ordered_pairs)\\n        if not paths:\\n            continue\\n        mol = Chem.MolFromSmiles(smi)\\n        if mol is None:\\n            continue\\n        for path in paths:\\n            level = {smi: mol}\\n            for t in path:\\n                nxt = {}\\n                for m in level.values():\\n                    nxt.update(apply(m, t))\\n                level = nxt\\n                if not level:\\n                    break\\n            for psmi, prod in level.items():\\n                rec = out.get(psmi)\\n                if rec is None:\\n                    if len(out) >= cap or abs(ExactMolWt(prod) - query_mass) > tol_da:\\n                        continue\\n                    rec = out[psmi] = dict(sim_max=sim, seeds=set(), sims={}, n_paths=0, depth=len(path),\\n                                           t1=T_INDEX[path[0].name], t2=T_INDEX[path[1].name] if len(path) > 1 else -1)\\n                rec[\\"seeds\\"].add(si)\\n                rec[\\"sims\\"][si] = sim\\n                rec[\\"n_paths\\"] += 1\\n                rec[\\"sim_max\\"] = max(rec[\\"sim_max\\"], sim)\\n                if len(path) < rec[\\"depth\\"]:\\n                    rec[\\"depth\\"], rec[\\"t1\\"] = len(path), T_INDEX[path[0].name]\\n                    rec[\\"t2\\"] = T_INDEX[path[1].name] if len(path) > 1 else -1\\n    for rec in out.values():\\n        rec[\\"n_seeds\\"] = len(rec.pop(\\"seeds\\"))\\n        rec[\\"sim_sum\\"] = float(sum(rec.pop(\\"sims\\").values()))\\n    return out\\n", "src/casmi/pool/__init__.py": "", "src/casmi/pool/build_safety.py": "\\"\\"\\"Exclusive publication for a newly built immutable CASMI candidate pool.\\n\\nA preflight `exists()` check does not guard against a second writer racing\\nto publish its file. Stage the Parquet on the same filesystem and atomically\\ncreate a hard link at the destination; unlike os.replace, os.link refuses\\nan existing destination. Clean the staging file on all outcomes.\\n\\nFilesystems without hard-link support fail closed. Readers must use immutable\\nversioned pool tags rather than expecting an existing file to be replaced.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom contextlib import contextmanager\\nimport os\\nfrom pathlib import Path\\nimport tempfile\\n\\n\\n@contextmanager\\ndef fresh_pool_file(destination):\\n    \\"\\"\\"Yield a temporary Parquet path; publish it only on successful exit.\\"\\"\\"\\n    destination = Path(destination)\\n    if not destination.name or destination.name in (\\".\\", \\"..\\"):\\n        raise ValueError(\\"An explicit non-root pool output path is required\\")\\n    if os.path.lexists(destination):\\n        raise FileExistsError(\\"Refusing to overwrite frozen candidate pool\\")\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    fd, name = tempfile.mkstemp(\\n        prefix=f\\".{destination.name}-building-\\", suffix=\\".parquet\\",\\n        dir=destination.parent,\\n    )\\n    os.close(fd)\\n    stage = Path(name)\\n    try:\\n        yield stage\\n        if not stage.is_file() or stage.stat().st_size == 0:\\n            raise ValueError(\\"Candidate pool build produced an empty output file\\")\\n        # Hard-link creation, unlike rename or os.replace, fails if a second\\n        # process has created the final path during the build.\\n        os.link(stage, destination, follow_symlinks=False)\\n    finally:\\n        stage.unlink(missing_ok=True)\\n", "src/casmi/pool/fingerprint_audit.py": "\\"\\"\\"Read-only quality summary for packed CASMI Morgan fingerprints.\\n\\nA SHA-256 inventory can consistently sign invalid zero-bit vectors. This\\naudit adds a numeric sanity check without changing or disclosing source rows.\\nIt does not recompute RDKit chemistry from SMILES and cannot certify model\\naccuracy or upstream dataset provenance.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\nBITS_BYTES = 256\\n\\n\\ndef fingerprint_quality_summary(pool_keys, fp_keys, fps, *, batch_rows: int = 65536) -> dict:\\n    \\"\\"\\"Verify row identity, vector format, and count zero vectors in chunks.\\"\\"\\"\\n    if type(batch_rows) is not int or not 1 <= batch_rows <= 1000000:\\n        raise ValueError(\\"batch_rows must be an integer from 1 to 1000000\\")\\n    if (not isinstance(fps, np.ndarray) or fps.dtype != np.uint8\\n            or fps.ndim != 2 or fps.shape[1] != BITS_BYTES):\\n        raise ValueError(\\"Fingerprint vectors must be packed uint8 Morgan 2048 bits\\")\\n    source = np.asarray(pool_keys)\\n    recorded = np.asarray(fp_keys)\\n    if source.ndim != 1 or recorded.ndim != 1:\\n        raise ValueError(\\"Fingerprint keys must be one-dimensional\\")\\n    if len(source) != len(recorded) or len(source) != len(fps):\\n        raise ValueError(\\"Pool keys and fingerprint vectors have different row counts\\")\\n    if not np.array_equal(source, recorded):\\n        raise ValueError(\\"Fingerprint key order differs from candidate pool\\")\\n    if len(fps) == 0:\\n        raise ValueError(\\"Empty candidate pools cannot pass fingerprint quality audit\\")\\n\\n    zero_rows = 0\\n    for start in range(0, len(fps), batch_rows):\\n        fragment = fps[start:start + batch_rows]\\n        zero_rows += int(np.count_nonzero(~fragment.any(axis=1)))\\n    return {\\n        \\"status\\": \\"PASS_NONZERO_FINGERPRINTS\\" if zero_rows == 0\\n                  else \\"INVALID_ZERO_FINGERPRINTS\\",\\n        \\"candidate_count\\": len(fps),\\n        \\"zero_fingerprint_rows\\": zero_rows,\\n        \\"zero_fraction\\": float(zero_rows / len(fps)) if len(fps) else 0.0,\\n        \\"method\\": \\"Read-only, row-aligned uint8 bits scanned in bounded batches\\",\\n        \\"warning\\": (\\n            \\"Does not authenticate Kaggle source, recalculate chemistry or \\"\\n            \\"prove candidate SMILES are structurally correct\\"\\n        ),\\n    }\\n", "src/casmi/pool/fingerprint_provenance.py": "\\"\\"\\"Content-bound provenance for CASMI candidate-pool Morgan fingerprints.\\n\\nFingerprint row order alone does not prove that the bit vectors were computed\\nfrom the CURRENT candidate SMILES: a corrected de-salted COCONUT parent may\\nshare a key with an old molecule but have different input structure bytes.\\nAn immutable research snapshot binds pool_key + smiles + mass AND fingerprint\\nbytes to the same recorded generation, without disclosing source rows.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport math\\n\\nimport numpy as np\\n\\nSCHEMA = \\"casmi-morgan-fingerprint-provenance-v1\\"\\nFP_SIZE_BYTES = 256  # radius 2, Morgan 2048 bits\\n\\n\\ndef pool_row_digest(pool) -> str:\\n    \\"\\"\\"Hash ordered (key, SMILES, mass) without materializing a JSON dataset.\\"\\"\\"\\n    required = (\\"pool_key\\", \\"smiles\\", \\"mass\\")\\n    if any(col not in pool for col in required):\\n        raise ValueError(\\"Candidate pool lacks key, SMILES or mass\\")\\n    sha = hashlib.sha256()\\n    sha.update(b\\"casmi-pool-rows-v1\\\\n\\")\\n    for key, smiles, mass in pool.loc[:, list(required)].itertuples(index=False, name=None):\\n        if not isinstance(key, str) or not key:\\n            raise ValueError(\\"Invalid candidate pool key\\")\\n        if not isinstance(smiles, str) or not smiles:\\n            raise ValueError(\\"Missing candidate pool SMILES\\")\\n        try:\\n            mass_value = float(mass)\\n        except (ValueError, TypeError) as exc:\\n            raise ValueError(\\"Invalid candidate mass\\") from exc\\n        if not math.isfinite(mass_value) or mass_value <= 0:\\n            raise ValueError(\\"Invalid candidate mass\\")\\n        line = json.dumps([key, smiles, mass_value.hex()],\\n                          separators=(\\",\\", \\":\\"), ensure_ascii=True)\\n        sha.update(line.encode(\\"utf-8\\"))\\n        sha.update(b\\"\\\\n\\")\\n    return sha.hexdigest()\\n\\n\\ndef _array_digest(fps) -> str:\\n    if not isinstance(fps, np.ndarray) or fps.dtype != np.uint8:\\n        raise ValueError(\\"Packed fingerprints must be a NumPy uint8 array\\")\\n    if fps.ndim != 2 or fps.shape[1] != FP_SIZE_BYTES:\\n        raise ValueError(\\"Expected 2048-bit packed fingerprints with 256 columns\\")\\n    if not fps.flags.c_contiguous:\\n        raise ValueError(\\"Packed fingerprint array must be C-contiguous\\")\\n    payload = memoryview(fps).cast(\\"B\\")\\n    digest = hashlib.sha256()\\n    for start in range(0, len(payload), 1024 * 1024):\\n        digest.update(payload[start:start + 1024 * 1024])\\n    return digest.hexdigest()\\n\\n\\ndef make_fingerprint_provenance(pool, keys, fps) -> dict:\\n    \\"\\"\\"Reject mismatched row ordering and produce a content-bound descriptor.\\"\\"\\"\\n    names = np.asarray(keys)\\n    expected = pool[\\"pool_key\\"].to_numpy()\\n    if names.ndim != 1 or len(names) != len(expected):\\n        raise ValueError(\\"Fingerprint-key row count differs from candidate pool\\")\\n    if not np.array_equal(names, expected):\\n        raise ValueError(\\"Fingerprint keys do not match ordered candidate pool keys\\")\\n    if len(fps) != len(expected):\\n        raise ValueError(\\"Fingerprint-vector row count differs from candidate pool\\")\\n    return {\\n        \\"schema\\": SCHEMA,\\n        \\"n_structures\\": len(expected),\\n        \\"fingerprint\\": \\"Morgan radius=2, 2048 bits, packed uint8\\",\\n        \\"pool_rows_sha256\\": pool_row_digest(pool),\\n        \\"fingerprint_bytes_sha256\\": _array_digest(fps),\\n        \\"limitations\\": (\\n            \\"Local byte-alignment attestation only. Does not authenticate \\"\\n            \\"Kaggle provenance, reconstruct chemistry or prove model accuracy.\\"\\n        ),\\n    }\\n\\n\\ndef verify_fingerprint_provenance(pool, keys, fps, manifest) -> dict:\\n    \\"\\"\\"Fail closed for stale SMILES, reordered keys, mutated bits or old schema.\\"\\"\\"\\n    if not isinstance(manifest, dict) or manifest.get(\\"schema\\") != SCHEMA:\\n        raise ValueError(\\"Missing or unsupported fingerprint provenance schema\\")\\n    actual = make_fingerprint_provenance(pool, keys, fps)\\n    for field in (\\"n_structures\\", \\"fingerprint\\", \\"pool_rows_sha256\\",\\n                  \\"fingerprint_bytes_sha256\\"):\\n        if type(manifest.get(field)) is not type(actual[field]) or manifest[field] != actual[field]:\\n            raise ValueError(f\\"Fingerprint provenance mismatch: {field}\\")\\n    return {\\n        \\"status\\": \\"POOL_FINGERPRINTS_VERIFIED\\",\\n        \\"schema\\": SCHEMA,\\n        \\"n_structures\\": actual[\\"n_structures\\"],\\n        \\"pool_rows_sha256\\": actual[\\"pool_rows_sha256\\"],\\n        \\"fingerprint_bytes_sha256\\": actual[\\"fingerprint_bytes_sha256\\"],\\n    }\\n", "src/casmi/pool/pool.py": "\\"\\"\\"A3 \\u2014 candidate pool: unique structures (metric key) sorted by neutral monoisotopic mass.\\n\\nSources are tracked as a bitmask so the ranker can learn source priors (e.g. that structures known\\nonly from the drug-like enveda-180 library are unlikely answers for a natural-product test set)\\nand so validation can knock out a structure from the pool (Class-3 simulation).\\n\\"\\"\\"\\nfrom dataclasses import dataclass\\nimport hashlib\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nSOURCE_BITS = {\\"train\\": 1, \\"enveda180_only\\": 2, \\"coconut\\": 4, \\"chebi\\": 8, \\"lipidmaps\\": 16, \\"pubchem\\": 32, \\"pubchem_lit\\": 64}\\n\\n\\ndef generated_candidate_identity(mol):\\n    \\"\\"\\"Bind generated cache identity to full canonical representation, not InChI14 alone.\\n\\n    Standard InChI can merge tautomers whose representation-sensitive Morgan\\n    fingerprints and fragments differ. Content IDs keep those caches independent;\\n    official metric identity is still handled separately by MetricKeys.\\n    \\"\\"\\"\\n    from rdkit import Chem\\n    smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)\\n    standard = Chem.MolToInchiKey(mol)[:14]\\n    return \\"G\\" + standard + \\":\\" + hashlib.sha256(smiles.encode()).hexdigest(), smiles\\n\\n\\n@dataclass\\nclass Window:\\n    idx: np.ndarray          # pool row indices\\n    ppm: np.ndarray          # signed ppm of each candidate relative to the query mass\\n\\n\\nclass CandidatePool:\\n    def __init__(self, table: pd.DataFrame):\\n        \\"\\"\\"table: pool_key, smiles, mass, sources (bitmask) [+ any extra per-structure columns].\\"\\"\\"\\n        needed = {\\"pool_key\\", \\"smiles\\", \\"mass\\"}\\n        if not needed.issubset(table.columns):\\n            raise ValueError(f\\"Candidate pool missing columns: {sorted(needed - set(table.columns))}\\")\\n        # Missing candidates can be dropped as before, but present impossible\\n        # masses must not silently produce NaN/inf ppm values or divide by 0.\\n        t = table.dropna(subset=[\\"pool_key\\", \\"mass\\"]).copy()\\n        try:\\n            mass = pd.to_numeric(t[\\"mass\\"], errors=\\"raise\\").to_numpy(dtype=np.float64)\\n        except (TypeError, ValueError) as exc:\\n            raise ValueError(\\"Candidate pool contains non-numeric masses\\") from exc\\n        if np.any(~np.isfinite(mass)) or np.any(mass <= 0.0):\\n            raise ValueError(\\"Candidate pool masses must be finite and strictly positive\\")\\n        t[\\"mass\\"] = mass\\n        t = t.sort_values(\\"mass\\", kind=\\"stable\\")\\n        t = t.drop_duplicates(\\"pool_key\\", keep=\\"first\\").reset_index(drop=True)\\n        self.table = t\\n        self.mass = t[\\"mass\\"].to_numpy(np.float64)\\n        self.key = t[\\"pool_key\\"].to_numpy()\\n        self.smiles = t[\\"smiles\\"].to_numpy()\\n        self.row_of = dict(zip(self.key, range(len(t))))\\n        # generated (Class-3 hedge) structures live outside the table: key -> SMILES / packed fingerprint\\n        self.extra_smiles, self.extra_fp = {}, {}\\n\\n    def smiles_of(self, key):\\n        r = self.row_of.get(key)\\n        return self.smiles[r] if r is not None else self.extra_smiles[key]\\n\\n    def register_generated(self, keys, smiles):\\n        \\"\\"\\"Atomically register chemically valid generated structures.\\n\\n        Active IDs bind standard InChIKey14 and canonical-SMILES SHA-256.\\n        Legacy \'G\'+InChIKey14 remains readable but ambiguous reuse is rejected.\\n        Invalid records cannot contribute\\n        all-zero fingerprints or leave a half-written candidate overlay.\\n        \\"\\"\\"\\n        from rdkit import Chem\\n        from rdkit.Chem import rdFingerprintGenerator\\n\\n        keys = list(keys)\\n        smiles = list(smiles)\\n        if len(keys) != len(smiles):\\n            raise ValueError(\\"Generated key and SMILES counts differ\\")\\n        gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)\\n        staged = {}\\n        for i, (key, smi) in enumerate(zip(keys, smiles)):\\n            if not isinstance(key, str) or not isinstance(smi, str):\\n                raise ValueError(f\\"Malformed generated candidate at row {i}\\")\\n            try:\\n                mol = Chem.MolFromSmiles(smi)\\n                if mol is None or mol.GetNumHeavyAtoms() == 0:\\n                    raise ValueError\\n                expected = \\"G\\" + Chem.MolToInchiKey(mol)[:14]\\n                content_key, canonical_smiles = generated_candidate_identity(mol)\\n                if key not in (expected, content_key):\\n                    raise ValueError\\n                previous = self.extra_smiles.get(key, staged.get(key, (None, None))[0])\\n                if previous is not None:\\n                    prior = Chem.MolToSmiles(Chem.MolFromSmiles(previous), canonical=True, isomericSmiles=True)\\n                    if prior != canonical_smiles:\\n                        raise ValueError\\n                    continue\\n                bits = gen.GetFingerprintAsNumPy(mol).astype(np.uint8)\\n                if bits.size != 2048 or not np.any(bits):\\n                    raise ValueError\\n            except (ValueError, TypeError, RuntimeError):\\n                # Never log the source SMILES or derived molecular identity.\\n                raise ValueError(f\\"Invalid generated candidate identity at row {i}\\") from None\\n            staged[key] = (canonical_smiles, np.packbits(bits))\\n        for key, (smi, fp) in staged.items():\\n            self.extra_smiles[key] = smi\\n            self.extra_fp[key] = fp\\n\\n    def packed_fps(self, keys, fps):\\n        \\"\\"\\"Aligned packed fingerprints for frozen pool and generated candidates.\\"\\"\\"\\n        if (not isinstance(fps, np.ndarray) or fps.dtype != np.uint8\\n                or fps.ndim != 2 or fps.shape != (len(self), 256)):\\n            raise ValueError(\\"Pool fingerprints must be row-aligned packed uint8 Morgan 2048 bits\\")\\n        names = list(keys)\\n        out = np.empty((len(names), 256), np.uint8)\\n        for i, key in enumerate(names):\\n            row = self.row_of.get(key)\\n            if row is not None:\\n                out[i] = fps[row]\\n            elif key in self.extra_fp:\\n                out[i] = self.extra_fp[key]\\n            else:\\n                raise KeyError(\\"Requested fingerprint key is absent from pool and generated overlay\\")\\n        return out\\n\\n    def __len__(self):\\n        return len(self.mass)\\n\\n    def window(self, query_mass, tol_ppm=10.0, exclude=None) -> Window:\\n        \\"\\"\\"All candidates with |ppm(candidate vs query)| <= tol_ppm, minus excluded keys.\\"\\"\\"\\n        try:\\n            if isinstance(query_mass, (bool, np.bool_)) or not np.isscalar(query_mass):\\n                raise ValueError\\n            mass_value = float(query_mass)\\n            if not np.isfinite(mass_value) or mass_value <= 0:\\n                raise ValueError\\n        except (TypeError, ValueError, OverflowError) as exc:\\n            raise ValueError(\\"Query mass must be a finite positive scalar\\") from exc\\n        try:\\n            if isinstance(tol_ppm, (bool, np.bool_)) or not np.isscalar(tol_ppm):\\n                raise ValueError\\n            tolerance = float(tol_ppm)\\n            if not np.isfinite(tolerance) or tolerance < 0:\\n                raise ValueError\\n        except (TypeError, ValueError, OverflowError) as exc:\\n            raise ValueError(\\"Mass tolerance ppm must be a finite nonnegative scalar\\") from exc\\n        d = mass_value * tolerance * 1e-6\\n        lo = np.searchsorted(self.mass, mass_value - d, side=\\"left\\")\\n        hi = np.searchsorted(self.mass, mass_value + d, side=\\"right\\")\\n        idx = np.arange(lo, hi)\\n        if exclude is not None:\\n            excluded = list(exclude)\\n            if excluded:\\n                idx = idx[~np.isin(self.key[idx], excluded)]\\n        return Window(idx, (mass_value - self.mass[idx]) / self.mass[idx] * 1e6)\\n", "src/casmi/pool/source_reconciliation.py": "\\"\\"\\"Fail-closed reconciliation before merging CASMI candidate source records.\\n\\nSame first-block InChIKey indicates matched connectivity but does not justify\\njoining arbitrarily different neutral masses. Some source rows lack complete\\nidentity fields; those may be excluded, but an impossible *present* mass and\\na material same-key mass contradiction must never be silently accepted.\\n\\nThis is a source-identity quality gate, not a proof that source providers\\nhave correct chemical annotations.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nREQUIRED = (\\"pool_key\\", \\"smiles\\", \\"mass\\")\\nDEFAULT_MAX_MASS_SPREAD_DA = 0.01\\nDEFAULT_MAX_MASS_SPREAD_PPM = 20.0\\n\\n\\ndef reconcile_source_rows(\\n    records: pd.DataFrame, *,\\n    max_spread_da: float = DEFAULT_MAX_MASS_SPREAD_DA,\\n    max_spread_ppm: float = DEFAULT_MAX_MASS_SPREAD_PPM,\\n    quarantine_conflicts: bool = False,\\n) -> tuple[pd.DataFrame, dict]:\\n    \\"\\"\\"Return whole, complete source rows after auditing duplicate key masses.\\n\\n    The intended source order (training library before external providers)\\n    is preserved. Crucially, the first source\'s SMILES and mass must both\\n    exist on the SAME row before groupby.first() is used by the caller.\\n    \\"\\"\\"\\n    if not isinstance(records, pd.DataFrame):\\n        raise TypeError(\\"Candidate source records must be a DataFrame\\")\\n    absent = sorted(set(REQUIRED) - set(records.columns))\\n    if absent:\\n        raise ValueError(\\"Candidate sources missing required fields: \\" + \\", \\".join(absent))\\n    if (not np.isfinite(max_spread_da) or max_spread_da < 0\\n            or not np.isfinite(max_spread_ppm) or max_spread_ppm < 0):\\n        raise ValueError(\\"Mass disagreement tolerances must be finite nonnegative values\\")\\n    complete = records.dropna(subset=list(REQUIRED)).copy()\\n    omitted = len(records) - len(complete)\\n    if complete.empty:\\n        raise ValueError(\\"No complete candidate source identity rows available\\")\\n    # A source key is the standard InChIKey connectivity block. Reject\\n    # malformed *present* keys instead of merging them as new structures.\\n    if (not complete[\\"pool_key\\"].map(lambda value:\\n                                    isinstance(value, str) and len(value) == 14\\n                                    and value.isascii() and value.isalpha()\\n                                    and value.isupper()).all()):\\n        raise ValueError(\\"Malformed candidate source InChIKey14\\")\\n    if not complete[\\"smiles\\"].map(lambda value:\\n                                   isinstance(value, str) and bool(value.strip())).all():\\n        raise ValueError(\\"Empty or malformed candidate source SMILES\\")\\n    try:\\n        mass = pd.to_numeric(complete[\\"mass\\"], errors=\\"raise\\").to_numpy(dtype=np.float64)\\n    except (TypeError, ValueError, OverflowError) as exc:\\n        raise ValueError(\\"Non-numeric candidate source mass\\") from exc\\n    if np.any(~np.isfinite(mass)) or np.any(mass <= 0):\\n        raise ValueError(\\"Candidate source mass must be finite and positive\\")\\n    complete[\\"mass\\"] = mass\\n\\n    ranges = complete.groupby(\\"pool_key\\", sort=False)[\\"mass\\"].agg([\\"min\\", \\"max\\"])\\n    discrepancies = (ranges[\\"max\\"] - ranges[\\"min\\"]).to_numpy(dtype=float)\\n    thresholds = np.maximum(\\n        float(max_spread_da),\\n        ranges[\\"min\\"].to_numpy(dtype=float) * float(max_spread_ppm) * 1e-6,\\n    )\\n    conflict_keys = ranges.index[discrepancies > thresholds]\\n    conflicts = len(conflict_keys)\\n    if conflicts and not quarantine_conflicts:\\n        # Aggregate count only: no licensed source identifiers, chemical\\n        # structures, or per-molecule annotations escape into log messages.\\n        raise ValueError(\\n            f\\"Conflicting neutral mass for {conflicts} shared InChIKey14 groups; \\"\\n            \\"quarantine or correct providers before building a new candidate pool\\"\\n        )\\n    quarantined = complete.pool_key.isin(conflict_keys)\\n    quarantined_rows = int(quarantined.sum())\\n    if conflicts:\\n        complete = complete.loc[~quarantined].copy()\\n        if complete.empty:\\n            raise ValueError(\\"No complete candidate rows remain after conflict quarantine\\")\\n    summary = {\\n        \\"status\\": \\"SOURCE_ROWS_RECONCILED\\",\\n        \\"input_rows\\": int(len(records)),\\n        \\"complete_rows\\": int(len(complete)),\\n        \\"incomplete_rows_excluded\\": int(omitted),\\n        \\"distinct_pool_keys\\": int(complete.pool_key.nunique()),\\n        \\"shared_pool_keys\\": int((complete.groupby(\\"pool_key\\").size() > 1).sum()),\\n        \\"mass_conflict_groups\\": 0,\\n        \\"mass_conflict_policy\\": \\"quarantine_entire_identity\\" if quarantine_conflicts else \\"reject\\",\\n        \\"quarantined_mass_conflict_groups\\": int(conflicts),\\n        \\"quarantined_source_rows\\": quarantined_rows,\\n        \\"max_source_mass_spread_da\\": float(discrepancies.max(initial=0.0)),\\n        \\"mass_threshold_da\\": float(max_spread_da),\\n        \\"mass_threshold_ppm\\": float(max_spread_ppm),\\n        \\"interpretation\\": \\"Pre-aggregation identity consistency only; not RDKit verification\\",\\n    }\\n    return complete, summary\\n", "src/casmi/pool/structure_identity.py": "\\"\\"\\"Chemically consistent neutral parent identities for multi-fragment inputs.\\n\\nCOCONUT supplies some multi-component SMILES strings. Taking the largest\\nfragment without recomputing its standard InChIKey and monoisotopic mass\\ncreates an inconsistent candidate identity. Candidate pool keys are standard\\n(non-tautomer-canonical) InChIKey first blocks, as in the train.parquet pool.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\n\\ndef neutral_parent_identity(smiles: str) -> tuple[str, str, float, str] | None:\\n    \\"\\"\\"Return (parent canonical SMILES, standard InChIKey14, exact mass, formula).\\n\\n    None signifies malformed SMILES, missing InChI support, or a charged\\n    parent. No placeholder molecule is ever substituted for invalid input.\\n    Largest *heavy-atom* fragment is selected, preserving the original\\n    source\'s parent-fragment rule. Caller must decide which rows to discard.\\n    \\"\\"\\"\\n    from rdkit import Chem\\n    from rdkit.Chem.Descriptors import ExactMolWt\\n    from rdkit.Chem import rdMolDescriptors\\n    import math\\n\\n    if not isinstance(smiles, str) or not smiles.strip():\\n        return None\\n    try:\\n        molecule = Chem.MolFromSmiles(smiles)\\n        if molecule is None:\\n            return None\\n        parts = Chem.GetMolFrags(molecule, asMols=True)\\n        if not parts:\\n            return None\\n        parent = max(parts, key=lambda mol: mol.GetNumHeavyAtoms())\\n        if parent.GetNumHeavyAtoms() == 0 or Chem.GetFormalCharge(parent) != 0:\\n            return None\\n        canonical = Chem.MolToSmiles(parent, canonical=True)\\n        standard_key = Chem.MolToInchiKey(parent)\\n        mass = float(ExactMolWt(parent))\\n        formula = rdMolDescriptors.CalcMolFormula(parent)\\n    except (ValueError, RuntimeError, TypeError):\\n        return None\\n    if (not isinstance(standard_key, str) or len(standard_key) < 14\\n            or not standard_key[:14].isalpha()\\n            or not math.isfinite(mass) or mass <= 0):\\n        return None\\n    return canonical, standard_key[:14], mass, formula\\n", "src/casmi/pool/structure_quality.py": "\\"\\"\\"Explicit whole-candidate quarantine before immutable pool publication.\\n\\nThis verifies parseability and nonzero Morgan features, not chemical validity,\\nprovider identity accuracy or biological plausibility. No row details are logged.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom concurrent.futures import ProcessPoolExecutor\\n\\nimport numpy as np\\n\\n\\ndef _invalid_reasons(smiles):\\n    from rdkit import Chem, RDLogger\\n    from rdkit.Chem import rdFingerprintGenerator\\n\\n    RDLogger.DisableLog(\\"rdApp.*\\")\\n    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)\\n    reasons = []\\n    for value in smiles:\\n        try:\\n            mol = Chem.MolFromSmiles(value) if isinstance(value, str) else None\\n            if mol is None:\\n                reasons.append(\\"unparseable\\")\\n            elif mol.GetNumHeavyAtoms() == 0:\\n                reasons.append(\\"no_heavy_atoms\\")\\n            elif generator.GetFingerprint(mol).GetNumOnBits() == 0:\\n                reasons.append(\\"zero_morgan\\")\\n            else:\\n                reasons.append(\\"\\")\\n        except (ValueError, TypeError, RuntimeError):\\n            reasons.append(\\"fingerprint_error\\")\\n    return reasons\\n\\n\\ndef quarantine_invalid_candidates(pool, *, workers=1):\\n    \\"\\"\\"Return retained rows and aggregate accounting; never silently zero-fill.\\"\\"\\"\\n    if not pool.pool_key.is_unique:\\n        raise ValueError(\\"Structure quarantine requires one candidate per identity\\")\\n    values = pool.smiles.tolist()\\n    chunks = [values[start:start + 5000] for start in range(0, len(values), 5000)]\\n    if workers == 1:\\n        nested = [_invalid_reasons(chunk) for chunk in chunks]\\n    else:\\n        with ProcessPoolExecutor(max_workers=workers) as executor:\\n            nested = list(executor.map(_invalid_reasons, chunks))\\n    reasons = np.asarray([reason for chunk in nested for reason in chunk], dtype=object)\\n    kept = pool.loc[reasons == \\"\\"].copy().reset_index(drop=True)\\n    if kept.empty:\\n        raise ValueError(\\"No candidates remain after structure quarantine\\")\\n    return kept, {\\n        \\"policy\\": \\"quarantine_invalid_selected_candidate_identity\\",\\n        \\"input_candidates\\": len(pool), \\"retained_candidates\\": len(kept),\\n        \\"quarantined_candidates\\": int(np.count_nonzero(reasons != \\"\\")),\\n        \\"reasons\\": {name: int(np.count_nonzero(reasons == name))\\n                    for name in (\\"unparseable\\", \\"no_heavy_atoms\\", \\"zero_morgan\\", \\"fingerprint_error\\")},\\n        \\"scope\\": \\"Parseable, heavy-atom-containing, nonzero Morgan structures; not provider identity authentication\\",\\n    }\\n", "src/casmi/predict.py": "\\"\\"\\"End-to-end prediction for one molecule or a whole test file \\u2014 the code path the submission runs.\\n\\n    engine features (D1, D2, D3) -> priors + list context -> ranker score -> metric-key dedupe\\n    -> 25 SMILES, best first\\n\\nEvery molecule is wrapped in a fallback: if anything fails, it is answered with its mass-window\\ncandidates ordered by mass error, so a single bad molecule can never cost a row (a missing row\\nscores 0 and an exception would lose the whole submission).\\n\\"\\"\\"\\nimport time\\nimport traceback\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom casmi.rank.features import add_context, add_priors\\n\\n\\nclass Predictor:\\n    def __init__(self, engine, pool, keys, ranker=None, fp_models=None, device=\\"cpu\\"):\\n        self.engine, self.pool, self.keys = engine, pool, keys\\n        self.ranker, self.fp_models, self.device = ranker, fp_models or [], device\\n        self.timings = {\\"features\\": 0.0, \\"d3\\": 0.0, \\"d4\\": 0.0, \\"rank\\": 0.0, \\"dedupe\\": 0.0}\\n        self.frag = None\\n        self.frag_x = None\\n        self.failures = []\\n\\n    # ------------------------------------------------------------------ one molecule\\n    def _d3(self, f, spectra):\\n        from casmi.channels.fpmodel import candidate_scores\\n        from casmi.channels.fpinfer import molecule_logp\\n        t = time.perf_counter()\\n        lp_on, lp_off = molecule_logp(self.fp_models, spectra, self.device)\\n        bits = np.unpackbits(self.pool.packed_fps(f.pool_key.to_numpy(), self.engine.fps), axis=1).astype(np.float32)\\n        f[\\"d3_ll\\"], f[\\"d3_cos\\"] = candidate_scores(lp_on, lp_off, bits)\\n        for c in (\\"d3_ll\\", \\"d3_cos\\"):\\n            v = f[c].to_numpy()\\n            f[f\\"{c}_rank\\"] = pd.Series(v).rank(ascending=False, method=\\"min\\").to_numpy()\\n            f[f\\"{c}_gap\\"] = v.max() - v\\n            sd = v.std()\\n            f[f\\"{c}_z\\"] = (v - v.mean()) / sd if sd > 0 else 0.0\\n        self.timings[\\"d3\\"] += time.perf_counter() - t\\n        return f\\n\\n    def features(self, spectra, *, excluded_rows=None, excluded_keys=None,\\n                 knocked_keys=frozenset()):\\n        \\"\\"\\"Use the inference path with explicit research-only reference/pool knockouts.\\n\\n        Defaults preserve ordinary inference. Validation also clears reference-count\\n        priors for withheld structures rather than leaking their original membership.\\n        \\"\\"\\"\\n        started = time.perf_counter()\\n        f = self.engine.features([dict(s) for s in spectra], excluded_rows, excluded_keys)\\n        self.timings[\\"features\\"] += time.perf_counter() - started\\n        if f.empty:\\n            return f\\n        f = add_context(add_priors(f, self.pool, knocked_keys))\\n        if self.fp_models:\\n            f = self._d3(f, spectra)\\n        if self.ranker is not None and any(c.startswith(\\"d4_\\") for c in self.ranker.columns):\\n            from casmi.channels.fragments import FragmentScorer, add_d4\\n            t = time.perf_counter()\\n            if self.frag is None:\\n                self.frag = FragmentScorer(self.pool.smiles_of)\\n            f = add_d4(f, self.frag, spectra)\\n            if any(c.startswith(\\"d4x_\\") for c in self.ranker.columns):\\n                from casmi.channels.fragments import D4X\\n                if self.frag_x is None:\\n                    self.frag_x = FragmentScorer(self.pool.smiles_of, **D4X)\\n                f = add_d4(f, self.frag_x, spectra, \\"d4x\\")\\n            self.timings[\\"d4\\"] += time.perf_counter() - t\\n        return f\\n\\n    def rank(self, spectra, top_n=25, *, excluded_rows=None, excluded_keys=None,\\n             knocked_keys=frozenset()):\\n        f = self.features(spectra, excluded_rows=excluded_rows, excluded_keys=excluded_keys,\\n                          knocked_keys=knocked_keys)\\n        if f.empty:\\n            return []\\n        t = time.perf_counter()\\n        if self.ranker is not None:\\n            order = np.argsort(-self.ranker.predict(f), kind=\\"stable\\")\\n            ranked = f.pool_key.to_numpy()[order][:60].tolist()\\n        else:\\n            ranked = self.engine.rank(f)\\n        self.timings[\\"rank\\"] += time.perf_counter() - t\\n        t = time.perf_counter()\\n        out = self.keys.dedupe_ranked(ranked, self.pool.smiles_of, top_n)\\n        self.timings[\\"dedupe\\"] += time.perf_counter() - t\\n        return out                                   # [(pool_key, metric_key)] best first\\n\\n    def fallback_keys(self, spectra, top_n=25):\\n        \\"\\"\\"Mass-window candidates ordered by mass error: the answer when the model path fails.\\"\\"\\"\\n        from casmi.engine import molecule_mass\\n        win = self.pool.window(molecule_mass(spectra), 20.0)\\n        keys = self.pool.key[win.idx][np.argsort(np.abs(win.ppm))].tolist()\\n        return self.keys.dedupe_ranked(keys, self.pool.smiles_of, top_n)\\n\\n    # ------------------------------------------------------------------ whole test file\\n    def predict_file(self, test: pd.DataFrame, log_every=50) -> pd.DataFrame:\\n        rows, t0 = [], time.time()\\n        groups = test.groupby(\\"molecule_id\\", sort=False)\\n        for i, (mid, g) in enumerate(groups):\\n            spectra = [dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\\n                            precursor_mz=float(r.precursor_mz), adduct=str(r.adduct),\\n                            ce_ev=float(np.mean(r.collision_energy_ev)) if r.collision_energy_ev is not None and len(r.collision_energy_ev) else np.nan,\\n                            mode=str(r.ionization_mode)) for r in g.itertuples()]\\n            try:\\n                picked = self.rank(spectra)\\n            except Exception:\\n                self.failures.append((mid, traceback.format_exc(limit=2)))\\n                picked = []\\n            if len(picked) < 25:                     # refill by metric key, so no structure appears twice\\n                seen = {mk for _, mk in picked}\\n                picked += [(pk, mk) for pk, mk in self.fallback_keys(spectra) if mk not in seen][: 25 - len(picked)]\\n            smiles = [self.pool.smiles_of(pk) for pk, _ in picked]\\n            rows.append((mid, \\";\\".join(smiles) if smiles else \\"C\\"))   # never an empty field\\n            if log_every and (i + 1) % log_every == 0:\\n                print(f\\"  {i + 1}/{groups.ngroups} molecules, {time.time() - t0:.0f}s\\", flush=True)\\n        self.keys.save()\\n        return pd.DataFrame(rows, columns=[\\"molecule_id\\", \\"smiles\\"])\\n\\n\\ndef validate_submission(sub: pd.DataFrame, test: pd.DataFrame, max_rank=25):\\n    \\"\\"\\"The format rules the scorer enforces, checked before writing.\\"\\"\\"\\n    assert list(sub.columns) == [\\"molecule_id\\", \\"smiles\\"], sub.columns\\n    assert not sub.isna().any().any(), \\"nulls\\"\\n    assert sub.molecule_id.is_unique, \\"duplicate molecule_id\\"\\n    missing = set(test.molecule_id) - set(sub.molecule_id)\\n    assert not missing, f\\"{len(missing)} molecules missing\\"\\n    n = sub.smiles.str.split(\\";\\").map(len)\\n    assert (n >= 1).all() and (n <= max_rank).all(), \\"guess count out of range\\"\\n", "src/casmi/rank/__init__.py": "", "src/casmi/rank/features.py": "\\"\\"\\"Ranker features: channel outputs + knockout-consistent priors + within-molecule list context.\\n\\nTwo leakage rules are enforced here, because the ranker would otherwise learn them:\\n  * priors derived from training-set membership (has spectra, spectrum count, which libraries)\\n    are recomputed after the knockout \\u2014 a simulated Class-2 answer must look exactly like a real\\n    one, i.e. like a structure with no reference spectra;\\n  * list-context features are computed per molecule over the candidates the engine actually saw.\\n\\"\\"\\"\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom casmi.pool.pool import SOURCE_BITS\\n\\nCHANNEL_COLS = [\\"d1_max\\", \\"d1_mean\\", \\"an_p1\\", \\"an_p2\\", \\"an_p4\\", \\"an_sim\\", \\"an_tani\\", \\"an_self\\"]\\nCONTEXT_OF = [\\"d1_max\\", \\"an_p4\\", \\"an_p2\\", \\"an_tani\\"]\\n\\n\\ndef add_priors(feats: pd.DataFrame, pool, knocked_keys=frozenset()) -> pd.DataFrame:\\n    t = pool.table\\n    rows = np.fromiter((pool.row_of.get(k, -1) for k in feats.pool_key), np.int64, len(feats))\\n    gen = rows < 0                                     # generated structures: in no source\\n    knocked = feats.pool_key.isin(knocked_keys).to_numpy() | gen\\n    rows = np.where(gen, 0, rows)\\n    src = np.where(gen, 0, t.sources.to_numpy()[rows])\\n    in_train = ((src & SOURCE_BITS[\\"train\\"]) > 0) & ~knocked\\n    feats = feats.assign(\\n        prior_in_train=in_train.astype(np.int8),\\n        prior_in_coconut=((src & SOURCE_BITS[\\"coconut\\"]) > 0).astype(np.int8),\\n        prior_enveda180_only=(((src & SOURCE_BITS[\\"enveda180_only\\"]) > 0) & ~knocked).astype(np.int8),\\n        prior_log_spectra=np.where(knocked, 0.0, np.log1p(t.n_spectra.to_numpy()[rows])),\\n        prior_np_likeness=np.where(gen, np.nan, t.np_likeness.to_numpy(dtype=float)[rows]),\\n        prior_has_sugar=np.where(gen, np.nan, t.has_sugar.map({True: 1.0, False: 0.0}).to_numpy(dtype=float)[rows]),\\n        abs_ppm=np.abs(feats.ppm.to_numpy()),\\n    )\\n    return feats\\n\\n\\ndef add_context(feats: pd.DataFrame) -> pd.DataFrame:\\n    \\"\\"\\"Rank, gap-to-best and z-score of key channels within the molecule\'s candidate list.\\"\\"\\"\\n    out = {}\\n    for c in CONTEXT_OF:\\n        if c not in feats:\\n            continue\\n        v = feats[c].to_numpy(dtype=float)\\n        best = v.max() if len(v) else 0.0\\n        sd = v.std() if len(v) > 1 else 0.0\\n        out[f\\"{c}_rank\\"] = pd.Series(v).rank(ascending=False, method=\\"min\\").to_numpy()\\n        out[f\\"{c}_gap\\"] = best - v\\n        out[f\\"{c}_z\\"] = (v - v.mean()) / sd if sd > 0 else np.zeros_like(v)\\n    out[\\"log_n_cand\\"] = np.full(len(feats), np.log1p(len(feats)))\\n    return feats.assign(**out)\\n\\n\\ndef feature_columns(feats: pd.DataFrame):\\n    drop = {\\"pool_key\\", \\"ppm\\", \\"n_cand\\", \\"label\\", \\"qid\\", \\"cls\\", \\"gen_smiles\\", \\"stratum\\"}\\n    return [c for c in feats.columns if c not in drop and not c.startswith(\\"audit_\\")]\\n", "src/casmi/rank/ranker.py": "\\"\\"\\"E \\u2014 LightGBM LambdaRank over per-molecule candidate lists, seed-bagged.\\n\\nGrouping is by (query, simulated class): each group is one molecule\'s candidate list with exactly\\none relevant item (or none, for recall misses \\u2014 those groups carry no gradient and are dropped).\\n\\"\\"\\"\\nfrom dataclasses import dataclass, field\\n\\nimport lightgbm as lgb\\nimport numpy as np\\nimport pandas as pd\\n\\nMONOTONE_UP = {\\"d1_max\\", \\"d1_mean\\", \\"an_p1\\", \\"an_p2\\", \\"an_p4\\", \\"an_self\\"}\\n\\n\\n@dataclass\\nclass RankerConfig:\\n    seeds: tuple = (0, 1, 2, 3, 4)\\n    params: dict = field(default_factory=lambda: dict(\\n        objective=\\"lambdarank\\", metric=\\"map\\", eval_at=[25], lambdarank_truncation_level=25,\\n        learning_rate=0.05, num_leaves=31, min_data_in_leaf=50, feature_fraction=0.8,\\n        bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1))\\n    num_boost_round: int = 2000\\n    early_stopping: int = 100\\n\\n\\nclass Ranker:\\n    def __init__(self, cfg: RankerConfig = RankerConfig()):\\n        self.cfg, self.models, self.columns = cfg, [], None\\n\\n    @staticmethod\\n    def _groups(df):\\n        return df.groupby([\\"qid\\", \\"cls\\"], sort=False).size().to_numpy()\\n\\n    def fit(self, train: pd.DataFrame, valid: pd.DataFrame, columns, class_weight=None):\\n        self.columns = list(columns)\\n        keep = lambda d: d[d.groupby([\\"qid\\", \\"cls\\"]).label.transform(\\"max\\") > 0]\\n        train, valid = keep(train), keep(valid)\\n        mono = [1 if c in MONOTONE_UP else 0 for c in self.columns]\\n        w = train.cls.map(class_weight).fillna(1.0).to_numpy() if class_weight else None\\n        dtr = lgb.Dataset(train[self.columns], train.label, group=self._groups(train), weight=w, free_raw_data=False)\\n        dva = lgb.Dataset(valid[self.columns], valid.label, group=self._groups(valid), reference=dtr)\\n        self.models, self.best_iters = [], []\\n        for seed in self.cfg.seeds:\\n            params = {**self.cfg.params, \\"seed\\": seed, \\"monotone_constraints\\": mono}\\n            m = lgb.train(params, dtr, self.cfg.num_boost_round, valid_sets=[dva],\\n                          callbacks=[lgb.early_stopping(self.cfg.early_stopping, verbose=False)])\\n            self.models.append(m)\\n            self.best_iters.append(m.best_iteration)\\n        return self\\n\\n    def predict(self, feats: pd.DataFrame) -> np.ndarray:\\n        X = feats[self.columns]\\n        return np.mean([m.predict(X, num_iteration=m.best_iteration) for m in self.models], axis=0)\\n\\n    def importance(self) -> pd.Series:\\n        imp = np.mean([m.feature_importance(\\"gain\\") for m in self.models], axis=0)\\n        return pd.Series(imp, index=self.columns).sort_values(ascending=False)\\n", "src/casmi/spectra/__init__.py": "", "src/casmi/spectra/build_safety.py": "\\"\\"\\"Exclusive, staged publication of immutable CASMI spectrum-store artifacts.\\n\\nThe A1 builder must never open an existing spectrum store for writing.\\nWriting four related output files directly into the public experiment\'s\\nhistorical store can silently truncate valid metadata/peak snapshots if\\na build fails midway. This context manager stages a complete new directory\\non the same filesystem and publishes it only after the caller\'s verification.\\n\\nNo external dependencies or CI environment required.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom contextlib import contextmanager\\nimport os\\nfrom pathlib import Path\\nimport shutil\\nimport tempfile\\n\\n\\n@contextmanager\\ndef fresh_store_target(destination):\\n    \\"\\"\\"Yield a new staging directory; publish once only on successful exit.\\n\\n    Existing destinations, including empty directories and dangling symlinks,\\n    are refused. The staging directory is removed on errors. The caller must\\n    complete and verify every artifact before leaving the context.\\n    The same-file-system rename makes publication atomic on supported OSes;\\n    parallel writers should be managed externally with a project lock.\\n    \\"\\"\\"\\n    destination = Path(destination)\\n    if not destination.name or destination.name in (\\".\\", \\"..\\"):\\n        raise ValueError(\\"An explicit, non-root output directory is required\\")\\n    if os.path.lexists(destination):\\n        raise FileExistsError(\\n            f\\"Refusing to overwrite an existing spectrum store: {destination}\\"\\n        )\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    stage = Path(tempfile.mkdtemp(\\n        prefix=f\\".{destination.name}-building-\\", dir=destination.parent,\\n    ))\\n    try:\\n        yield stage\\n        if os.path.lexists(destination):\\n            raise FileExistsError(\\n                f\\"Output appeared during build; refusing to overwrite: {destination}\\"\\n            )\\n        stage.rename(destination)\\n    except BaseException:\\n        shutil.rmtree(stage, ignore_errors=True)\\n        raise\\n", "src/casmi/spectra/clean.py": "\\"\\"\\"Peak cleaning. One function is used for library spectra (offline) and query spectra (online),\\nso the two can never drift apart.\\n\\n`clean_peaks` is the readable NumPy reference; `clean_peaks_fast` is the numba version used on\\nmillions of spectra. tests/test_spectra.py asserts they agree on random spectra.\\n\\"\\"\\"\\nfrom dataclasses import dataclass\\n\\nimport numba as nb\\nimport numpy as np\\n\\n\\n@dataclass(frozen=True)\\nclass CleanConfig:\\n    max_above_precursor: float = 2.0   # Da; a fragment cannot outweigh its singly charged precursor\\n    min_rel_intensity: float = 0.001   # relative to base peak\\n    window_da: float = 50.0            # windowed top-k keeps low-mass fragments a global top-N drops\\n    top_k_per_window: int = 0          # 0 disables windowed filtering\\n    max_peaks: int = 150               # global cap after the filters above\\n    merge_tol_da: float = 0.005        # peaks closer than this are merged (centroiding artefacts)\\n\\n\\ndef clean_peaks(mz, intensity, precursor_mz, cfg: CleanConfig = CleanConfig()):\\n    \\"\\"\\"Return (mz, intensity) sorted by m/z, float32, intensity normalised to base peak = 1.\\"\\"\\"\\n    mz = np.asarray(mz, dtype=np.float64)\\n    it = np.asarray(intensity, dtype=np.float64)\\n    # Fragment m/z must be strictly positive; negative/zero bins can corrupt\\n    # downstream sparse-vector indexing and spectral similarity assumptions.\\n    keep = np.isfinite(mz) & (mz > 0) & np.isfinite(it) & (it > 0)\\n    if precursor_mz is not None and np.isfinite(precursor_mz):\\n        keep &= mz <= precursor_mz + cfg.max_above_precursor\\n    mz, it = mz[keep], it[keep]\\n    if mz.size == 0:\\n        return np.empty(0, np.float32), np.empty(0, np.float32)\\n\\n    order = np.argsort(mz, kind=\\"stable\\")\\n    mz, it = mz[order], it[order]\\n    if cfg.merge_tol_da > 0 and mz.size > 1:\\n        # merge runs of peaks closer than merge_tol_da: intensity-weighted m/z, summed intensity\\n        group = np.concatenate([[0], np.cumsum(np.diff(mz) > cfg.merge_tol_da)])\\n        if group[-1] + 1 < mz.size:\\n            w = np.bincount(group, weights=it)\\n            mz = np.bincount(group, weights=mz * it) / w\\n            it = w\\n\\n    it = it / it.max()\\n    keep = it >= cfg.min_rel_intensity\\n    mz, it = mz[keep], it[keep]\\n\\n    if cfg.top_k_per_window and mz.size > cfg.top_k_per_window:\\n        win = np.floor(mz / cfg.window_da).astype(np.int64)\\n        # rank peaks within each window by intensity (descending)\\n        order = np.lexsort((-it, win))\\n        rank = np.empty(mz.size, np.int64)\\n        starts = np.r_[0, np.flatnonzero(np.diff(win[order])) + 1]\\n        run = np.arange(mz.size) - np.repeat(starts, np.diff(np.r_[starts, mz.size]))\\n        rank[order] = run\\n        keep = rank < cfg.top_k_per_window\\n        mz, it = mz[keep], it[keep]\\n\\n    if mz.size > cfg.max_peaks:\\n        top = np.argpartition(-it, cfg.max_peaks - 1)[:cfg.max_peaks]\\n        top.sort()\\n        mz, it = mz[top], it[top]\\n    return mz.astype(np.float32), it.astype(np.float32)\\n\\n\\n@nb.njit(cache=True)\\ndef _clean_nb(mz, it, precursor_mz, max_above, min_rel, window_da, top_k, max_peaks, merge_tol):\\n    n = mz.size\\n    keep = np.empty(n, np.bool_)\\n    lim = precursor_mz + max_above if np.isfinite(precursor_mz) else np.inf\\n    m = 0\\n    for i in range(n):\\n        ok = np.isfinite(mz[i]) and mz[i] > 0 and np.isfinite(it[i]) and it[i] > 0 and mz[i] <= lim\\n        keep[i] = ok\\n        m += ok\\n    mz, it = mz[keep], it[keep]\\n    if m == 0:\\n        return np.empty(0, np.float32), np.empty(0, np.float32)\\n    order = np.argsort(mz, kind=\\"mergesort\\")\\n    mz, it = mz[order], it[order]\\n\\n    # chain-merge consecutive peaks closer than merge_tol\\n    out_mz = np.empty(m, np.float64)\\n    out_it = np.empty(m, np.float64)\\n    g = 0\\n    wsum, isum = mz[0] * it[0], it[0]\\n    for i in range(1, m):\\n        if merge_tol > 0 and mz[i] - mz[i - 1] <= merge_tol:\\n            wsum += mz[i] * it[i]\\n            isum += it[i]\\n        else:\\n            out_mz[g], out_it[g] = wsum / isum, isum\\n            g += 1\\n            wsum, isum = mz[i] * it[i], it[i]\\n    out_mz[g], out_it[g] = wsum / isum, isum\\n    g += 1\\n    mz, it = out_mz[:g], out_it[:g]\\n\\n    it = it / it.max()\\n    sel = it >= min_rel\\n    mz, it = mz[sel], it[sel]\\n\\n    if top_k > 0 and mz.size > top_k:\\n        win = np.floor(mz / window_da)\\n        keepw = np.zeros(mz.size, np.bool_)\\n        s = 0\\n        while s < mz.size:\\n            e = s\\n            while e < mz.size and win[e] == win[s]:\\n                e += 1\\n            seg = np.argsort(-it[s:e], kind=\\"mergesort\\")\\n            for j in range(min(top_k, e - s)):\\n                keepw[s + seg[j]] = True\\n            s = e\\n        mz, it = mz[keepw], it[keepw]\\n\\n    if mz.size > max_peaks:\\n        top = np.sort(np.argsort(-it, kind=\\"mergesort\\")[:max_peaks])\\n        mz, it = mz[top], it[top]\\n    return mz.astype(np.float32), it.astype(np.float32)\\n\\n\\ndef clean_peaks_fast(mz, intensity, precursor_mz, cfg: CleanConfig = CleanConfig()):\\n    \\"\\"\\"numba equivalent of `clean_peaks` (~50x faster).\\"\\"\\"\\n    p = np.nan if precursor_mz is None else float(precursor_mz)\\n    return _clean_nb(np.asarray(mz, np.float64), np.asarray(intensity, np.float64), p,\\n                     cfg.max_above_precursor, cfg.min_rel_intensity, cfg.window_da,\\n                     cfg.top_k_per_window, cfg.max_peaks, cfg.merge_tol_da)\\n", "src/casmi/spectra/similarity.py": "\\"\\"\\"Spectral similarities, compiled with numba.\\n\\nEntropy similarity (Li et al., Nature Methods 2021) with intensity weighting:\\n    sim = sum over matched peak pairs [ (a+b)ln(a+b) - a ln a - b ln b ] / ln 4\\nwith each spectrum\'s intensities normalised to sum 1. This closed form follows from\\n1 - (2 S_AB - S_A - S_B) / ln 4 and needs only the matched pairs.\\n\\n`shift` enables the mass-shifted (\\"modified\\") variant used for analog search: a peak may match\\neither at the same m/z or at m/z + shift, where shift = precursor difference between the spectra.\\n\\"\\"\\"\\nimport numba as nb\\nimport numpy as np\\n\\nLN4 = np.log(4.0)\\n\\n\\n@nb.njit(cache=True)\\ndef _entropy(p):\\n    s = 0.0\\n    for x in p:\\n        if x > 0:\\n            s -= x * np.log(x)\\n    return s\\n\\n\\n@nb.njit(cache=True)\\ndef _valid_spectrum(mz, it):\\n    \\"\\"\\"Reject invalid numerical peak arrays before any logarithm or square root.\\"\\"\\"\\n    if mz.ndim != 1 or it.ndim != 1 or mz.size != it.size:\\n        raise ValueError(\\"Spectrum m/z and intensity must be aligned 1D arrays\\")\\n    for i in range(mz.size):\\n        if (not np.isfinite(mz[i]) or mz[i] < 0.0\\n                or not np.isfinite(it[i]) or it[i] < 0.0):\\n            return False\\n    return True\\n\\n\\n@nb.njit(cache=True)\\ndef weight_intensity(it):\\n    \\"\\"\\"Normalise positive intensity; return zeros for zero/invalid total.\\"\\"\\"\\n    total = it.sum()\\n    if not np.isfinite(total) or total <= 0.0:\\n        return np.zeros(it.size, dtype=np.float64)\\n    p = it / total\\n    s = _entropy(p)\\n    if s < 3.0:\\n        w = 0.25 + 0.25 * s\\n        p = p ** w\\n        p = p / p.sum()\\n    return p\\n\\n\\n@nb.njit(cache=True)\\ndef _match(mz_a, mz_b, tol_da, shift):\\n    \\"\\"\\"Greedy one-to-one matching of peaks within tol_da, allowing an optional mass shift.\\n    Returns index pairs sorted by m/z proximity (closest pairs claimed first).\\"\\"\\"\\n    na, nb_ = mz_a.size, mz_b.size\\n    cand_i, cand_j, cand_d = [], [], []\\n    for s_idx in range(2 if shift != 0.0 else 1):\\n        off = shift if s_idx == 1 else 0.0\\n        j0 = 0\\n        for i in range(na):\\n            lo = mz_a[i] + off - tol_da\\n            while j0 < nb_ and mz_b[j0] < lo:\\n                j0 += 1\\n            j = j0\\n            while j < nb_ and mz_b[j] <= mz_a[i] + off + tol_da:\\n                cand_i.append(i)\\n                cand_j.append(j)\\n                cand_d.append(abs(mz_b[j] - mz_a[i] - off))\\n                j += 1\\n    n = len(cand_i)\\n    order = np.argsort(np.array(cand_d)) if n else np.empty(0, np.int64)\\n    used_a = np.zeros(na, np.bool_)\\n    used_b = np.zeros(nb_, np.bool_)\\n    pi, pj = [], []\\n    for k in order:\\n        i, j = cand_i[k], cand_j[k]\\n        if not used_a[i] and not used_b[j]:\\n            used_a[i] = True\\n            used_b[j] = True\\n            pi.append(i)\\n            pj.append(j)\\n    return pi, pj\\n\\n\\n@nb.njit(cache=True)\\ndef entropy_similarity(mz_a, it_a, mz_b, it_b, tol_da=0.02, shift=0.0):\\n    if not _valid_spectrum(mz_a, it_a) or not _valid_spectrum(mz_b, it_b):\\n        return 0.0\\n    if mz_a.size == 0 or mz_b.size == 0:\\n        return 0.0\\n    pa = weight_intensity(it_a.astype(np.float64))\\n    pb = weight_intensity(it_b.astype(np.float64))\\n    if pa.sum() <= 0.0 or pb.sum() <= 0.0:\\n        return 0.0\\n    pi, pj = _match(mz_a.astype(np.float64), mz_b.astype(np.float64), tol_da, shift)\\n    s = 0.0\\n    for k in range(len(pi)):\\n        a, b = pa[pi[k]], pb[pj[k]]\\n        if a <= 0.0 or b <= 0.0:\\n            continue  # 0*log(0) is mathematically zero, not NaN.\\n        s += (a + b) * np.log(a + b) - a * np.log(a) - b * np.log(b)\\n    return s / LN4\\n\\n\\n@nb.njit(cache=True)\\ndef cosine_similarity(mz_a, it_a, mz_b, it_b, tol_da=0.02, shift=0.0):\\n    \\"\\"\\"Greedy (modified, if shift != 0) cosine on sqrt intensities.\\"\\"\\"\\n    if not _valid_spectrum(mz_a, it_a) or not _valid_spectrum(mz_b, it_b):\\n        return 0.0\\n    if mz_a.size == 0 or mz_b.size == 0:\\n        return 0.0\\n    a = np.sqrt(it_a.astype(np.float64))\\n    b = np.sqrt(it_b.astype(np.float64))\\n    pi, pj = _match(mz_a.astype(np.float64), mz_b.astype(np.float64), tol_da, shift)\\n    s = 0.0\\n    for k in range(len(pi)):\\n        s += a[pi[k]] * b[pj[k]]\\n    norm = np.sqrt((a * a).sum()) * np.sqrt((b * b).sum())\\n    return s / norm if norm > 0.0 and np.isfinite(norm) else 0.0\\n", "src/casmi/spectra/store.py": "\\"\\"\\"A1 \\u2014 on-disk spectrum store: cleaned peaks as flat float32 memmaps + a metadata table.\\n\\nLayout (artifacts/store/):\\n  mz.f32, it.f32   all peaks back to back\\n  offsets.i64      spectrum i owns peaks[offsets[i]:offsets[i+1]]\\n  meta.parquet     one row per spectrum (library, structure, adduct, masses, ppm, CE, ...)\\n\\"\\"\\"\\nfrom pathlib import Path\\n\\nimport operator\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom casmi.spectra.store_integrity import inspect_store_layout\\n\\n\\nclass SpectrumStore:\\n    def __init__(self, root: Path):\\n        root = Path(root)\\n        self.meta = pd.read_parquet(root / \\"meta.parquet\\")\\n        if \\"n_peaks\\" not in self.meta:\\n            raise ValueError(\\"Spectrum metadata lacks per-row n_peaks counts\\")\\n        layout = inspect_store_layout(\\n            root, expected_spectra=len(self.meta),\\n            metadata_n_peaks=self.meta[\\"n_peaks\\"].to_numpy(),\\n        )\\n        self.offsets = layout.offsets\\n        self.layout = layout\\n        if layout.peak_count:\\n            self.mz = np.memmap(root / \\"mz.f32\\", dtype=np.float32, mode=\\"r\\")\\n            self.it = np.memmap(root / \\"it.f32\\", dtype=np.float32, mode=\\"r\\")\\n        else:\\n            # A valid empty peak store cannot be memmapped by NumPy.\\n            self.mz = np.empty(0, dtype=np.float32)\\n            self.it = np.empty(0, dtype=np.float32)\\n\\n    def close(self):\\n        \\"\\"\\"Release memory-mapped peak files (important for Windows file locks).\\n\\n        Any arrays returned by peaks() must not be accessed after close().\\n        Calling close() repeatedly is safe. The source files are not changed.\\n        \\"\\"\\"\\n        for name in (\\"mz\\", \\"it\\"):\\n            array = getattr(self, name, None)\\n            mapped = getattr(array, \\"_mmap\\", None)\\n            if mapped is not None and not mapped.closed:\\n                mapped.close()\\n        self.mz = None\\n        self.it = None\\n\\n    def __enter__(self):\\n        return self\\n\\n    def __exit__(self, exc_type, exc_value, traceback):\\n        self.close()\\n        return False\\n\\n    def __len__(self):\\n        return len(self.meta)\\n\\n    def peaks(self, i):\\n        if self.mz is None or self.it is None:\\n            raise ValueError(\\"SpectrumStore is closed\\")\\n        if isinstance(i, (bool, np.bool_)):\\n            raise TypeError(\\"Spectrum index must be an integer row ID\\")\\n        row = operator.index(i)\\n        if row < 0 or row >= len(self.meta):\\n            raise IndexError(f\\"Spectrum row {row} is outside the store\\")\\n        a, b = int(self.offsets[row]), int(self.offsets[row + 1])\\n        return np.asarray(self.mz[a:b]), np.asarray(self.it[a:b])\\n", "src/casmi/spectra/store_integrity.py": "\\"\\"\\"Fast fail-closed structural checks for an existing CASMI MS/MS peak store.\\n\\nThe store format consists of contiguous float32 m/z/intensity arrays, int64\\noffsets, and a separate metadata Parquet file. This module reads the entire\\noffset vector (small relative to spectra), but NEVER scans peak values.\\n\\nLayout verification confirms file lengths, offsets, row counts and per-row\\npeak counts. It does not establish chemical correctness, non-finite peak\\ncontent, upstream Kaggle provenance, or source-library independence.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\n\\n@dataclass(frozen=True)\\nclass StoreLayout:\\n    \\"\\"\\"Checked store offsets with metadata-only count diagnostics.\\"\\"\\"\\n    offsets: np.ndarray\\n    spectrum_count: int\\n    peak_count: int\\n    empty_spectra: int\\n    max_peaks_per_spectrum: int\\n\\n    def summary(self) -> dict:\\n        return {\\n            \\"status\\": \\"STRUCTURE_VERIFIED\\",\\n            \\"spectrum_count\\": self.spectrum_count,\\n            \\"peak_count\\": self.peak_count,\\n            \\"empty_spectra\\": self.empty_spectra,\\n            \\"max_peaks_per_spectrum\\": self.max_peaks_per_spectrum,\\n            \\"method\\": \\"offsets and file byte lengths; not a full peak-content scan\\",\\n        }\\n\\n\\ndef inspect_store_layout(root, *, expected_spectra=None,\\n                         metadata_n_peaks=None) -> StoreLayout:\\n    \\"\\"\\"Validate raw files before mmap; raise ValueError on structural drift.\\n\\n    Integers in offsets.i64 are native-endian signed int64 by store design,\\n    and mz.f32 / it.f32 are native-endian float32. Check byte alignment\\n    before trusting them; avoid Python assertions (which can be disabled).\\n    \\"\\"\\"\\n    root = Path(root)\\n    offsets_file = root / \\"offsets.i64\\"\\n    mz_file = root / \\"mz.f32\\"\\n    it_file = root / \\"it.f32\\"\\n    offset_bytes = offsets_file.stat().st_size\\n    mz_bytes = mz_file.stat().st_size\\n    it_bytes = it_file.stat().st_size\\n    if offset_bytes < 16 or offset_bytes % 8:\\n        raise ValueError(\\"Invalid offsets.i64 byte length: at least two aligned offsets required\\")\\n    if mz_bytes % 4 or it_bytes % 4:\\n        raise ValueError(\\"Spectrum peak files must have float32-aligned byte lengths\\")\\n    if mz_bytes != it_bytes:\\n        raise ValueError(\\"Spectrum m/z and intensity files have different peak counts\\")\\n    offsets = np.fromfile(offsets_file, dtype=np.int64)\\n    n_spectra = len(offsets) - 1\\n    n_peaks = mz_bytes // 4\\n    if expected_spectra is not None:\\n        if type(expected_spectra) is not int or expected_spectra < 0:\\n            raise ValueError(\\"Expected spectrum row count must be a nonnegative integer\\")\\n        if n_spectra != expected_spectra:\\n            raise ValueError(\\"Spectrum metadata row count and offset count differ\\")\\n    if offsets[0] != 0:\\n        raise ValueError(\\"First spectrum offset must equal zero\\")\\n    if np.any(offsets < 0) or np.any(offsets[1:] < offsets[:-1]):\\n        raise ValueError(\\"Spectrum offsets must be nonnegative and nondecreasing\\")\\n    if int(offsets[-1]) != n_peaks:\\n        raise ValueError(\\"Final spectrum offset does not equal flat peak count\\")\\n    counts = offsets[1:] - offsets[:-1]\\n\\n    if metadata_n_peaks is not None:\\n        try:\\n            declared = np.asarray(metadata_n_peaks, dtype=np.float64)\\n        except (TypeError, ValueError) as exc:\\n            raise ValueError(\\"Invalid metadata n_peaks values\\") from exc\\n        if declared.ndim != 1 or len(declared) != n_spectra:\\n            raise ValueError(\\"Metadata n_peaks must align with spectrum rows\\")\\n        if (not np.all(np.isfinite(declared)) or np.any(declared < 0)\\n                or np.any(declared != np.floor(declared))):\\n            raise ValueError(\\"Metadata n_peaks contains missing or invalid counts\\")\\n        if not np.array_equal(declared, counts):\\n            raise ValueError(\\"Metadata n_peaks disagrees with spectrum offsets\\")\\n\\n    return StoreLayout(\\n        offsets=offsets, spectrum_count=n_spectra, peak_count=n_peaks,\\n        empty_spectra=int(np.count_nonzero(counts == 0)),\\n        max_peaks_per_spectrum=int(np.max(counts, initial=0)),\\n    )\\n\\ndef scan_store_peak_content(root, *, chunk_peaks: int = 262144) -> dict:\\n    \\"\\"\\"Optionally scan every stored m/z and intensity using bounded memory.\\n\\n    The standard store reader and inspect_store_layout deliberately avoid the\\n    multi-GB content scan. This separate diagnostic does not load Parquet,\\n    mutate peak files, or reveal individual spectra. Only aggregate counts\\n    leave the analysis. A \\"clean\\" result is about numeric peak admissibility,\\n    not chemical correctness or the origin of the source files.\\n    \\"\\"\\"\\n    if type(chunk_peaks) is not int or not 1 <= chunk_peaks <= 1048576:\\n        raise ValueError(\\"chunk_peaks must be an integer from 1 to 1048576\\")\\n    root = Path(root)\\n    layout = inspect_store_layout(root)\\n    nonfinite_mz = nonpositive_mz = 0\\n    nonfinite_intensity = nonpositive_intensity = 0\\n    invalid_peak_pairs = 0\\n    read = 0\\n\\n    with (root / \\"mz.f32\\").open(\\"rb\\") as mz_file, (root / \\"it.f32\\").open(\\"rb\\") as it_file:\\n        remaining = layout.peak_count\\n        while remaining:\\n            count = min(chunk_peaks, remaining)\\n            mz = np.fromfile(mz_file, dtype=np.float32, count=count)\\n            it = np.fromfile(it_file, dtype=np.float32, count=count)\\n            if len(mz) != count or len(it) != count:\\n                raise ValueError(\\"Peak file truncated during content scan\\")\\n            mz_finite = np.isfinite(mz)\\n            it_finite = np.isfinite(it)\\n            mz_nonpositive = mz_finite & (mz <= 0)\\n            it_nonpositive = it_finite & (it <= 0)\\n            nonfinite_mz += int(np.count_nonzero(~mz_finite))\\n            nonpositive_mz += int(np.count_nonzero(mz_nonpositive))\\n            nonfinite_intensity += int(np.count_nonzero(~it_finite))\\n            nonpositive_intensity += int(np.count_nonzero(it_nonpositive))\\n            invalid_peak_pairs += int(np.count_nonzero(\\n                ~(mz_finite & it_finite) | mz_nonpositive | it_nonpositive\\n            ))\\n            read += count\\n            remaining -= count\\n\\n        if mz_file.read(1) or it_file.read(1):\\n            raise ValueError(\\"Peak file grew during content scan\\")\\n\\n    invalid = invalid_peak_pairs > 0\\n    return {\\n        \\"status\\": \\"INVALID_CLEANED_PEAKS\\" if invalid else \\"VALID_CLEANED_PEAKS\\",\\n        \\"total_peaks_scanned\\": read,\\n        \\"nonfinite_mz\\": nonfinite_mz,\\n        \\"nonpositive_mz\\": nonpositive_mz,\\n        \\"nonfinite_intensity\\": nonfinite_intensity,\\n        \\"nonpositive_intensity\\": nonpositive_intensity,\\n        \\"invalid_peak_pairs\\": invalid_peak_pairs,\\n        \\"chunk_peaks\\": chunk_peaks,\\n        \\"method\\": \\"full numeric content scan in bounded float32 chunks\\",\\n        \\"interpretation\\": (\\n            \\"Aggregate cleaned-peak numeric validity only; not chemistry, \\"\\n            \\"data-lineage, leakage or instrument accuracy\\"\\n        ),\\n    }\\n", "src/casmi/submit/__init__.py": "", "src/casmi/validate/__init__.py": "", "src/casmi/validate/analog_index_evidence.py": "\\"\\"\\"Decision-grade D2 index evidence for signed CASMI validation reports.\\n\\nLegacy cached analog index files were created without source provenance.\\nA run without a *freshly* rebuilt reference matrix remains exploratory even\\nwhen the source manifest and candidate Morgan bits have been verified.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Mapping\\n\\nINDEX_SCHEMA = \\"casmi-d2-fresh-index-v1\\"\\nINDEX_STATUS = \\"REBUILT_WITHOUT_CACHE\\"\\n\\n\\ndef verified_analog_index_evidence(run: Mapping) -> bool:\\n    \\"\\"\\"Validate reported cache-free index provenance, not external data truth.\\"\\"\\"\\n    proof = run.get(\\"analog_index_provenance\\")\\n    if proof is None:\\n        return False\\n    if not isinstance(proof, Mapping):\\n        raise ValueError(\\"Malformed D2 index provenance\\")\\n    if proof.get(\\"schema\\") != INDEX_SCHEMA or proof.get(\\"status\\") != INDEX_STATUS:\\n        raise ValueError(\\"Unverified D2 index provenance schema or status\\")\\n    if proof.get(\\"cache_used\\") is not False:\\n        raise ValueError(\\"D2 index provenance must explicitly bypass the cache\\")\\n    digest = proof.get(\\"index_sha256\\")\\n    if not isinstance(digest, str) or len(digest) != 64 or any(\\n        char not in \\"0123456789abcdef\\" for char in digest\\n    ):\\n        raise ValueError(\\"Invalid D2 index content digest\\")\\n    rows = proof.get(\\"n_references\\")\\n    nnz = proof.get(\\"n_nonzero\\")\\n    if type(rows) is not int or rows < 1:\\n        raise ValueError(\\"Invalid D2 index reference count\\")\\n    if type(nnz) is not int or nnz < 0:\\n        raise ValueError(\\"Invalid D2 index nonzero count\\")\\n    cfg = run.get(\\"config\\")\\n    analog = cfg.get(\\"analog\\") if isinstance(cfg, Mapping) else None\\n    policy = analog.get(\\"reference_policy\\") if isinstance(analog, Mapping) else None\\n    if not isinstance(policy, str) or policy not in (\\"legacy\\", \\"source_balanced\\"):\\n        raise ValueError(\\"Missing or unsupported D2 reference-selection policy\\")\\n    if proof.get(\\"reference_policy\\") != policy:\\n        raise ValueError(\\"D2 index policy contradicts model configuration\\")\\n    return True\\n", "src/casmi/validate/budget_features.py": "\\"\\"\\"Recompute list-relative features after selecting a declared candidate budget.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom casmi.rank.features import add_context\\n\\n\\ndef budget_features(frame, policy):\\n    if policy not in (\\"legacy\\", \\"expanded\\"):\\n        raise ValueError(\\"Unknown declared candidate-budget policy\\")\\n    if frame.empty:\\n        return frame.copy()\\n    if \\"audit_legacy\\" not in frame or not frame.audit_legacy.isin([0, 1]).all():\\n        raise ValueError(\\"Candidate origin audit is missing or invalid\\")\\n    selected = frame if policy == \\"expanded\\" else frame[frame.audit_legacy == 1]\\n    selected = selected.drop(columns=[c for c in selected if c.startswith(\\"audit_\\")]).reset_index(drop=True).copy()\\n    if selected.empty:\\n        return selected\\n    selected[\\"n_cand\\"] = len(selected)\\n    selected = add_context(selected)\\n    for column in (\\"d3_ll\\", \\"d3_cos\\", \\"d4_int\\", \\"d4_pk\\", \\"d4x_int\\", \\"d4x_pk\\"):\\n        if column not in selected:\\n            continue\\n        values = selected[column].to_numpy(dtype=float)\\n        sd = values.std()\\n        selected[column + \\"_rank\\"] = pd.Series(values).rank(ascending=False, method=\\"min\\").to_numpy()\\n        selected[column + \\"_gap\\"] = values.max() - values\\n        selected[column + \\"_z\\"] = (values - values.mean()) / sd if sd > 0 else np.zeros_like(values)\\n    return selected\\n", "src/casmi/validate/cohort_bootstrap.py": "\\"\\"\\"Joint fixed-family uncertainty with independent resampling across source cohorts.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\nfrom .rank_inputs import validated_bootstrap\\n\\n\\ndef joint_cohort_intervals(effects_by_cohort, *, n_boot=4000, seed=0):\\n    validated_bootstrap(n_boot, seed)\\n    if not effects_by_cohort:\\n        raise ValueError(\\"At least one declared source cohort required\\")\\n    groups, names_seen = {}, set()\\n    for cohort, effects in sorted(effects_by_cohort.items()):\\n        if not effects or set(effects) & names_seen:\\n            raise ValueError(\\"Empty or duplicate secondary contrast family\\")\\n        names_seen.update(effects)\\n        arrays = [np.asarray(value, dtype=float) for value in effects.values()]\\n        if any(a.ndim != 1 or len(a) < 2 or not np.isfinite(a).all() or np.any(np.abs(a) > 1) for a in arrays):\\n            raise ValueError(\\"Finite bounded paired effects from at least two molecules required\\")\\n        if len({len(a) for a in arrays}) != 1:\\n            raise ValueError(\\"Every contrast in one cohort must share paired molecules\\")\\n        groups[cohort] = (list(effects), np.column_stack(arrays))\\n    maximum = np.zeros(n_boot)\\n    outcomes, counts = {}, {}\\n    for child_seed, (cohort, (names, values)) in zip(np.random.SeedSequence(seed).spawn(len(groups)), groups.items()):\\n        mean = values.mean(axis=0)\\n        counts[cohort] = len(values)\\n        rng = np.random.default_rng(child_seed)\\n        for start in range(0, n_boot, 100):\\n            ids = rng.integers(len(values), size=(min(100, n_boot - start), len(values)))\\n            deviations = np.max(np.abs(values[ids].mean(axis=1) - mean), axis=1)\\n            maximum[start:start + len(ids)] = np.maximum(maximum[start:start + len(ids)], deviations)\\n        outcomes.update({name: {\\"cohort\\": cohort, \\"delta\\": float(value)} for name, value in zip(names, mean)})\\n    radius = float(np.quantile(maximum, .95))\\n    for value in outcomes.values():\\n        value.update(ci_low=value[\\"delta\\"] - radius, ci_high=value[\\"delta\\"] + radius)\\n    return {\\"method\\": \\"Independent molecule resampling across cohorts; paired within cohort; fixed-family max-deviation bootstrap\\",\\n            \\"n_bootstrap\\": n_boot, \\"seed\\": seed, \\"family_size\\": len(outcomes), \\"cohort_counts\\": counts,\\n            \\"joint_radius\\": radius, \\"outcomes\\": outcomes}\\n", "src/casmi/validate/control_alignment.py": "\\"\\"\\"Identity-verified, diagnostic-only pairing of spectral negative controls.\\n\\nThis module deliberately does not accept legacy/unverified runs or produce\\nmodel-improvement ACCEPT/REJECT judgments. It verifies that the reference and\\nperturbed reports use the same molecule IDs, underlying original spectra,\\nnovelty exclusion scope, model configuration and query sampling.\\n\\nSeparate because align_runs must continue to reject altered MS/MS evidence.\\nNo numerical/chemistry dependency is required.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Mapping\\nfrom copy import deepcopy\\n\\nfrom casmi.validate.run_alignment import CLASSES, SIGNATURE_VERSION, _check_ranks\\n\\n\\ndef _signatures(run, ids, key):\\n    values = run.get(key)\\n    if not isinstance(values, list) or len(values) != len(ids):\\n        raise ValueError(f\\"{key}: one spectrum SHA-256 required per query\\")\\n    if any(not isinstance(v, str) or len(v) != 64\\n           or any(c not in \\"0123456789abcdef\\" for c in v) for v in values):\\n        raise ValueError(f\\"{key}: invalid SHA-256 fingerprint\\")\\n    return dict(zip(ids, values))\\n\\n\\ndef _identities(run, title):\\n    count = run.get(\\"n_queries\\")\\n    ids = run.get(\\"query_ids\\")\\n    if type(count) is not int or count < 1 or not isinstance(ids, list) or len(ids) != count:\\n        raise ValueError(f\\"{title}: invalid query count or identity manifest\\")\\n    if any(not isinstance(item, str) or not item for item in ids) or len(set(ids)) != count:\\n        raise ValueError(f\\"{title}: duplicate or missing query identities\\")\\n    version = run.get(\\"query_signature_version\\")\\n    if version != SIGNATURE_VERSION:\\n        raise ValueError(f\\"{title}: unsupported query signature recipe\\")\\n    ranks = _check_ranks(run, count)\\n    return ids, _signatures(run, ids, \\"query_signatures\\"), ranks\\n\\n\\ndef align_control_runs(reference: Mapping, treatment: Mapping) -> dict:\\n    \\"\\"\\"Align diagnostic ranks by query ID, verify source fingerprints.\\n\\n    `reference` must be an unperturbed validation report. `treatment` must\\n    be a negative-control run with original signatures captured *before*\\n    treatment. A changed model/config/sampling invalidates the diagnostic.\\n    \\"\\"\\"\\n    if not isinstance(reference, Mapping) or not isinstance(treatment, Mapping):\\n        raise ValueError(\\"Both reports must be mappings\\")\\n    rc, tc = reference.get(\\"spectral_control\\"), treatment.get(\\"spectral_control\\")\\n    if not isinstance(rc, Mapping) or rc.get(\\"mode\\") != \\"none\\":\\n        raise ValueError(\\"Reference must be a signed unperturbed control=none run\\")\\n    if not isinstance(tc, Mapping) or tc.get(\\"mode\\") not in (\\"blank\\", \\"intensity_shuffle\\"):\\n        raise ValueError(\\"Treatment must be a signed negative-control run\\")\\n    if type(tc.get(\\"seed\\")) is not int or tc[\\"seed\\"] < 0:\\n        raise ValueError(\\"Treatment seed must be a nonnegative integer\\")\\n    a_ids, a_signatures, a_ranks = _identities(reference, \\"Reference\\")\\n    b_ids, b_signatures, b_ranks = _identities(treatment, \\"Treatment\\")\\n    if len(a_ids) != len(b_ids) or set(a_ids) != set(b_ids):\\n        raise ValueError(\\"Control population/query ID mismatch\\")\\n    source_a = reference.get(\\"source_manifest_sha256\\")\\n    source_b = treatment.get(\\"source_manifest_sha256\\")\\n    if source_a is not None or source_b is not None:\\n        if (not isinstance(source_a, str) or len(source_a) != 64\\n                or any(ch not in \\"0123456789abcdef\\" for ch in source_a)\\n                or source_a != source_b):\\n            raise ValueError(\\"Negative control source artifact snapshots differ\\")\\n    if reference.get(\\"config\\") != treatment.get(\\"config\\"):\\n        raise ValueError(\\"Control cannot be compared across model configurations\\")\\n    sampling_a, sampling_b = reference.get(\\"query_sampling\\"), treatment.get(\\"query_sampling\\")\\n    if not isinstance(sampling_a, Mapping) or not isinstance(sampling_b, Mapping):\\n        raise ValueError(\\"Missing query sampling provenance\\")\\n    keep_a = {k: v for k, v in sampling_a.items()\\n              if k not in (\\"spectral_control\\", \\"control_seed\\")}\\n    keep_b = {k: v for k, v in sampling_b.items()\\n              if k not in (\\"spectral_control\\", \\"control_seed\\")}\\n    if keep_a != keep_b:\\n        raise ValueError(\\"Control query sampling differs\\")\\n    if sampling_a.get(\\"spectral_control\\") != \\"none\\":\\n        raise ValueError(\\"Reference sampling not marked as unperturbed\\")\\n    if (sampling_b.get(\\"spectral_control\\") != tc[\\"mode\\"]\\n            or sampling_b.get(\\"control_seed\\") != tc[\\"seed\\"]):\\n        raise ValueError(\\"Treatment mode or seed contradicts query sampling\\")\\n\\n    source_signatures = _signatures(\\n        {\\"query_signatures\\": tc.get(\\"source_query_signatures\\",\\n                                  treatment.get(\\"spectral_control\\", {}).get(\\n                                      \\"source_query_signatures\\"))},\\n        b_ids, \\"query_signatures\\",\\n    )\\n    if source_signatures != a_signatures:\\n        raise ValueError(\\"Control source spectra or reference-exclusion scope differs\\")\\n    if not any(b_signatures[qid] != source_signatures[qid] for qid in a_ids):\\n        raise ValueError(\\"No actual spectral perturbation detected\\")\\n    index_b = {qid: i for i, qid in enumerate(b_ids)}\\n    return {\\n        \\"n_queries\\": len(a_ids),\\n        \\"control_mode\\": tc[\\"mode\\"],\\n        \\"control_seed\\": tc[\\"seed\\"],\\n        \\"verified_source_pairing\\": True,\\n        \\"changed_queries\\": sum(\\n            b_signatures[qid] != source_signatures[qid] for qid in a_ids\\n        ),\\n        \\"reference_ranks\\": deepcopy(a_ranks),\\n        \\"treatment_ranks\\": {\\n            cls: [b_ranks[cls][index_b[qid]] for qid in a_ids] for cls in CLASSES\\n        },\\n    }\\n", "src/casmi/validate/coverage_budget.py": "\\"\\"\\"Descriptive candidate-coverage and ranking-loss accounting from aggregates.\\n\\nNo per-query outcomes are inferred. Conditional means describe the observed\\ncovered subset and are unsuitable for comparing models with different pools.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\n\\n\\ndef coverage_budget(summary, truth_candidates_present):\\n    n = summary[\\"n\\"]\\n    if type(n) is not int or n < 1 or type(truth_candidates_present) is not int or not 0 <= truth_candidates_present <= n:\\n        raise ValueError(\\"Positive integer population and bounded integer coverage count required\\")\\n    values = {}\\n    for field in (\\"mrr\\", \\"top1\\", \\"top10\\", \\"recall_at_25\\"):\\n        value = summary[field]\\n        if type(value) not in (int, float) or not math.isfinite(value) or not 0 <= value <= 1:\\n            raise ValueError(\\"Finite bounded aggregate metrics required\\")\\n        values[field] = value\\n    counts = {}\\n    for field in (\\"top1\\", \\"top10\\", \\"recall_at_25\\"):\\n        count = round(values[field] * n)\\n        if not math.isclose(count, values[field] * n, abs_tol=1e-9, rel_tol=0):\\n            raise ValueError(\\"Aggregate hit rate does not correspond to integer molecules\\")\\n        counts[field] = count\\n    if not 0 <= counts[\\"top1\\"] <= counts[\\"top10\\"] <= counts[\\"recall_at_25\\"] <= truth_candidates_present:\\n        raise ValueError(\\"Ranked hits exceed candidate coverage or violate nested cutoffs\\")\\n    ceiling = truth_candidates_present / n\\n    mrr = values[\\"mrr\\"]\\n    # Non-top-1 hits contribute at most 1/2; top-10 and top-25\\n    # hit groups contribute at least 1/10 and 1/25 respectively.\\n    lower = (counts[\\"top1\\"] + (counts[\\"top10\\"] - counts[\\"top1\\"]) / 10\\n             + (counts[\\"recall_at_25\\"] - counts[\\"top10\\"]) / 25) / n\\n    upper = (counts[\\"top1\\"] + (counts[\\"top10\\"] - counts[\\"top1\\"]) / 2\\n             + (counts[\\"recall_at_25\\"] - counts[\\"top10\\"]) / 11) / n\\n    if not lower - 1e-12 <= mrr <= upper + 1e-12:\\n        raise ValueError(\\"MRR contradicts the observed top-1/top-10/top-25 counts\\")\\n    return {\\n        \\"n_molecules\\": n,\\n        \\"truth_candidates_present\\": truth_candidates_present,\\n        \\"truth_candidates_absent\\": n - truth_candidates_present,\\n        \\"top1_hits\\": counts[\\"top1\\"], \\"top10_hits\\": counts[\\"top10\\"], \\"top25_hits\\": counts[\\"recall_at_25\\"],\\n        \\"covered_outside_top25\\": truth_candidates_present - counts[\\"recall_at_25\\"],\\n        \\"coverage_ceiling\\": ceiling, \\"observed_mrr\\": mrr,\\n        \\"coverage_loss\\": 1 - ceiling, \\"within_coverage_ranking_loss\\": max(0.0, ceiling - mrr),\\n        \\"conditional_mrr_given_coverage\\": mrr / ceiling if truth_candidates_present else None,\\n        \\"conditional_hit25_given_coverage\\": counts[\\"recall_at_25\\"] / truth_candidates_present if truth_candidates_present else None,\\n    }\\n", "src/casmi/validate/error_slices.py": "\\"\\"\\"Prespecified molecule-level diagnostic slices for CASMI research validation.\\n\\nGroupings use only input metadata. All outcomes are descriptive, with small\\ncells suppressed and no post-hoc statistical ACCEPT/REJECT claims. This module\\nrequires only the Python standard library and signed validation reports.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\nfrom collections import defaultdict\\nfrom collections.abc import Mapping\\nfrom statistics import median\\n\\nfrom casmi.validate.run_alignment import CLASSES, align_runs\\n\\nFIELDS = (\\"adduct\\", \\"ionization_mode\\", \\"spectra_bin\\", \\"precursor_mz_bin\\")\\nMASS_BINS = (\\"<300\\", \\"300-600\\", \\">=600\\", \\"unknown\\")\\n\\n\\ndef describe_query(spectra) -> dict:\\n    \\"\\"\\"Compute one query\'s *original* non-label metadata, before perturbation.\\"\\"\\"\\n    if not isinstance(spectra, (list, tuple)) or not spectra:\\n        raise ValueError(\\"Query must contain at least one selected spectrum\\")\\n    def shared(field):\\n        values = {str(s[field]) for s in spectra}\\n        return next(iter(values)) if len(values) == 1 else \\"mixed\\"\\n\\n    n = len(spectra)\\n    masses = [float(s[\\"precursor_mz\\"]) for s in spectra]\\n    masses = [m for m in masses if math.isfinite(m)]\\n    midpoint = median(masses) if masses else math.nan\\n    mass_bin = (\\"unknown\\" if not math.isfinite(midpoint) else\\n                \\"<300\\" if midpoint < 300 else\\n                \\"300-600\\" if midpoint < 600 else \\">=600\\")\\n    return {\\n        \\"adduct\\": shared(\\"adduct\\"),\\n        \\"ionization_mode\\": shared(\\"mode\\"),\\n        \\"spectra_bin\\": \\"1\\" if n == 1 else \\"2-3\\" if n <= 3 else \\"4+\\",\\n        \\"precursor_mz_bin\\": mass_bin,\\n    }\\n\\n\\ndef _checked_contexts(run, ids, label):\\n    context = run.get(\\"query_context\\")\\n    if not isinstance(context, list) or len(context) != len(ids):\\n        raise ValueError(f\\"{label}: missing or incomplete query_context\\")\\n    by_id = {}\\n    for qid, item in zip(ids, context):\\n        if not isinstance(item, Mapping) or set(item) != set(FIELDS):\\n            raise ValueError(f\\"{label}: invalid query_context fields\\")\\n        row = dict(item)\\n        if any(not isinstance(row[field], str) or not row[field]\\n               for field in FIELDS):\\n            raise ValueError(f\\"{label}: empty or malformed query_context value\\")\\n        if row[\\"spectra_bin\\"] not in (\\"1\\", \\"2-3\\", \\"4+\\"):\\n            raise ValueError(f\\"{label}: invalid spectra-count bin\\")\\n        if row[\\"precursor_mz_bin\\"] not in MASS_BINS:\\n            raise ValueError(f\\"{label}: invalid precursor-mass bin\\")\\n        by_id[qid] = row\\n    return by_id\\n\\n\\ndef _mrr(ranks, indices):\\n    return sum(1.0 / ranks[i] for i in indices if ranks[i]) / len(indices)\\n\\n\\ndef diagnostic_slices(reference: Mapping, candidate: Mapping, *, min_n: int = 15) -> dict:\\n    \\"\\"\\"Compare known, query-aligned context slices; NEVER infer superiority.\\"\\"\\"\\n    if type(min_n) is not int or min_n < 2:\\n        raise ValueError(\\"min_n must be an integer >= 2\\")\\n    aligned = align_runs(reference, candidate)\\n    ids = reference[\\"query_ids\\"]\\n    b_ids = candidate[\\"query_ids\\"]\\n    context_a = _checked_contexts(reference, ids, \\"Reference\\")\\n    context_b = _checked_contexts(candidate, b_ids, \\"Candidate\\")\\n    if context_a != context_b:\\n        raise ValueError(\\"Query input metadata differ across validation runs\\")\\n    groups = {}\\n    for field in FIELDS:\\n        cells = defaultdict(list)\\n        for index, qid in enumerate(ids):\\n            cells[context_a[qid][field]].append(index)\\n        records = []\\n        for value in sorted(cells):\\n            subset = cells[value]\\n            row = {\\"value\\": value, \\"n_molecules\\": len(subset)}\\n            if len(subset) < min_n:\\n                row.update(status=\\"SUPPRESSED_SMALL_CELL\\")\\n            else:\\n                classes = {}\\n                for cls in CLASSES:\\n                    a = _mrr(aligned[\\"a\\"][cls], subset)\\n                    b = _mrr(aligned[\\"b\\"][cls], subset)\\n                    classes[cls] = {\\n                        \\"reference_mrr\\": a,\\n                        \\"candidate_mrr\\": b,\\n                        \\"candidate_minus_reference\\": b - a,\\n                    }\\n                row.update(status=\\"DESCRIPTIVE_ONLY\\", classes=classes)\\n            records.append(row)\\n        groups[field] = records\\n    return {\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"source_evidence\\": (\\n            \\"VERIFIED_REQUIRED_INPUTS\\" if aligned[\\"decision_eligible\\"]\\n            else \\"QUERY_PAIRED_ONLY_SOURCE_UNVERIFIED\\"\\n        ),\\n        \\"n_molecules\\": aligned[\\"n\\"],\\n        \\"min_cell_size\\": min_n,\\n        \\"slices\\": groups,\\n        \\"interpretation\\": (\\n            \\"Exploratory error characterization, not a model-improvement \\"\\n            \\"verdict. Multiple groups and novelty classes are inspected \\"\\n            \\"without multiplicity correction or independent holdout.\\"\\n        ),\\n    }\\n", "src/casmi/validate/external_population.py": "\\"\\"\\"Outcome-blind follow-on populations, preserving previously opened test families.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom .scaffold_study import family_key, partition, digest\\nfrom casmi.chem.canon import canonical_ik14\\n\\n\\ndef prepare_internal(meta, truth_pool, previous_protocol, previous_audit, protocol):\\n    keys = meta.train_ik14.astype(str).to_numpy()\\n    labels = dict(zip(truth_pool.key, truth_pool.smiles))\\n    masses = dict(zip(truth_pool.key, truth_pool.mass))\\n    eligible = (meta.ingest_lib.isin(previous_protocol[\\"query_libraries\\"]).to_numpy()\\n                & meta.test_adduct.to_numpy() & (meta.n_peaks.to_numpy() >= 5)\\n                & (np.abs(meta.ppm.to_numpy()) <= 20))\\n    rows = np.flatnonzero(eligible)\\n    choices = pd.DataFrame({\\"row\\": rows, \\"key\\": keys[rows], \\"library\\": meta.ingest_lib.astype(str).to_numpy()[rows]})\\n    choices = choices[choices.key.map(masses).between(*previous_protocol[\\"query_mass_range\\"])]\\n    counts = choices.groupby([\\"key\\", \\"library\\"]).size().reset_index(name=\\"count\\")\\n    best = counts.sort_values([\\"count\\", \\"key\\", \\"library\\"], ascending=[False, True, True]).drop_duplicates(\\"key\\")\\n    groups = {key: group.row.to_numpy() for key, group in choices.merge(best[[\\"key\\", \\"library\\"]], on=[\\"key\\", \\"library\\"]).groupby(\\"key\\")}\\n    historical_vin = {family_key(labels[key]) for key in pd.unique(keys[(meta.ingest_lib == \\"enveda-np-examples\\").to_numpy()]) if key in labels}\\n    representatives = {}\\n    for row in best.itertuples():\\n        family = family_key(labels[row.key])\\n        if family and family not in historical_vin and (family not in representatives or row.key < representatives[family][\\"key\\"]):\\n            representatives[family] = {\\"key\\": row.key, \\"family\\": family, \\"library\\": row.library}\\n    records = []\\n    for value in representatives.values():\\n        metric = canonical_ik14(labels[value[\\"key\\"]])\\n        if metric:\\n            records.append({**value, \\"metric\\": metric})\\n    previous = partition(records, previous_protocol[\\"split_counts\\"], previous_protocol[\\"split_salt\\"])\\n    for name, values in previous.items():\\n        for field, private_values in ((\\"identity_sha256\\", [v[\\"key\\"] for v in values]),\\n                                      (\\"family_sha256\\", [v[\\"group\\"] for v in values]),\\n                                      (\\"metric_sha256\\", [v[\\"metric\\"] for v in values])):\\n            if digest(private_values) != previous_audit[name][field]:\\n                raise ValueError(\\"Historical partitions could not be reconstructed exactly\\")\\n    old_values = [v for values in previous.values() for v in values]\\n    old_metrics = {v[\\"metric\\"] for v in old_values}\\n    old_families = {v[field] for v in old_values for field in (\\"family\\", \\"group\\")}\\n    fresh_records = [v for v in records if v[\\"metric\\"] not in old_metrics and v[\\"family\\"] not in old_families]\\n    splits = partition(fresh_records, protocol[\\"internal_counts\\"], protocol[\\"split_salt\\"])\\n    audit = {}\\n    for name, values in splits.items():\\n        audit[name] = {\\"n_molecules\\": len(values), \\"identity_sha256\\": digest([r[\\"key\\"] for r in values]),\\n                       \\"family_sha256\\": digest([r[\\"group\\"] for r in values]), \\"metric_sha256\\": digest([r[\\"metric\\"] for r in values]),\\n                       \\"libraries\\": pd.Series([r[\\"library\\"] for r in values]).value_counts().to_dict()}\\n    audit.update(previous_population_reconstructed=True, previous_selected_metric_overlap=0,\\n                 previous_selected_family_overlap=0, cross_split_identity_overlaps=0)\\n    return splits, groups, labels, audit\\n\\n\\ndef select_external(novel, splits, salt, *, maximum=150, minimum=30):\\n    blocked = {v[field] for name in (\\"train\\", \\"dev\\") for v in splits[name] for field in (\\"family\\", \\"group\\")}\\n    excluded_metrics = set(novel.loc[novel.family.isin(blocked), \\"metric_key\\"])\\n    eligible = novel[~novel.metric_key.isin(excluded_metrics)]\\n    ordered = sorted(set(eligible.metric_key), key=lambda metric: hashlib.sha256((salt + \\"|external|\\" + metric).encode()).hexdigest())[:maximum]\\n    if len(ordered) < minimum:\\n        raise ValueError(\\"Insufficient independent external molecules after supervised-family exclusion\\")\\n    selected = []\\n    for metric in ordered:\\n        spectra = eligible[eligible.metric_key == metric].sort_values(\\"accession\\").head(5)\\n        first = spectra.iloc[0]\\n        selected.append({\\"metric\\": metric, \\"key\\": first.standard_ik14, \\"smiles\\": first.smiles,\\n                         \\"family\\": first.family, \\"records\\": spectra})\\n    audit = {\\"n_molecules\\": len(selected), \\"metric_sha256\\": digest(ordered),\\n             \\"family_sha256\\": digest([v[\\"family\\"] for v in selected]),\\n             \\"accessions_sha256\\": digest([str(a) for v in selected for a in v[\\"records\\"].accession]),\\n             \\"supervised_family_overlap\\": 0, \\"excluded_supervised_family_molecules\\": len(excluded_metrics)}\\n    return selected, audit\\n", "src/casmi/validate/fingerprint_evidence.py": "\\"\\"\\"Standard-library D2 fingerprint evidence gate for scientific run comparisons.\\n\\nA verified local input inventory alone does not prove that an analog model\\nused Morgan fingerprints computed for the *same SMILES and masses*. Reports\\nwithout a verified D2 sidecar remain descriptive/exploratory; they must not\\nsupport a model ACCEPT/REJECT decision.\\n\\nThis validates report evidence, not the source data itself. The upstream\\nrun_validation step must cryptographically verify the actual artifacts.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Mapping\\n\\nFINGERPRINT_SCHEMA = \\"casmi-morgan-fingerprint-provenance-v1\\"\\nFINGERPRINT_STATUS = \\"POOL_FINGERPRINTS_VERIFIED\\"\\n\\n\\ndef _valid_sha(value) -> bool:\\n    return (isinstance(value, str) and len(value) == 64\\n            and all(char in \\"0123456789abcdef\\" for char in value))\\n\\n\\ndef verified_d2_fingerprint_evidence(a: Mapping, b: Mapping) -> bool:\\n    \\"\\"\\"Return whether all D2 runs use verified, mutually compatible vectors.\\n\\n    D1-only records do not consume D2 fingerprints. A D1/D2 ablation is\\n    permitted if the D2 run has verified provenance and BOTH runs share a\\n    complete encompassing input manifest as enforced by run_alignment.\\n    \\"\\"\\"\\n    from casmi.validate.analog_index_evidence import verified_analog_index_evidence\\n\\n    proofs = []\\n    missing = False\\n    for run in (a, b):\\n        cfg = run.get(\\"config\\")\\n        channels = cfg.get(\\"channels\\") if isinstance(cfg, Mapping) else None\\n        if not isinstance(channels, list):\\n            raise ValueError(\\"Missing model channel provenance\\")\\n        if \\"d2\\" not in channels:\\n            continue\\n        proof = run.get(\\"fingerprint_provenance\\")\\n        if proof is None:\\n            missing = True  # historical report: descriptive only\\n            continue\\n        if not isinstance(proof, Mapping):\\n            raise ValueError(\\"Malformed D2 fingerprint provenance\\")\\n        if (proof.get(\\"status\\") != FINGERPRINT_STATUS\\n                or proof.get(\\"schema\\") != FINGERPRINT_SCHEMA):\\n            raise ValueError(\\"Unverified D2 fingerprint provenance schema or status\\")\\n        count = proof.get(\\"n_structures\\")\\n        if type(count) is not int or count < 1:\\n            raise ValueError(\\"Invalid D2 fingerprint row count\\")\\n        for key in (\\"pool_rows_sha256\\", \\"fingerprint_bytes_sha256\\"):\\n            if not _valid_sha(proof.get(key)):\\n                raise ValueError(f\\"Invalid D2 fingerprint provenance: {key}\\")\\n        proofs.append(proof)\\n        # A valid fingerprint sidecar cannot certify a cached D2 reference\\n        # matrix. Unverified historical/cached runs remain exploratory.\\n        if not verified_analog_index_evidence(run):\\n            missing = True\\n\\n    if missing:\\n        return False\\n    if len(proofs) == 2 and any(\\n        proofs[0][key] != proofs[1][key] for key in\\n        (\\"n_structures\\", \\"pool_rows_sha256\\", \\"fingerprint_bytes_sha256\\")\\n    ):\\n        raise ValueError(\\"D2 fingerprint source mismatch between validation runs\\")\\n    return True\\n", "src/casmi/validate/knockout.py": "\\"\\"\\"Structure-level knockout validation.\\n\\nA query is one structure with all its spectra from the query library (e.g. enveda-np-examples),\\nmimicking a test molecule. Each query is scored under three simulated novelty classes:\\n\\n  C1  library minus the query\'s own spectra        pool keeps the structure\\n  C2  library minus EVERY spectrum of the structure, in every library   pool keeps it\\n  C3  as C2, and the structure is removed from the pool                 (retrieval must score 0)\\n\\nKnockout is by metric key (tautomer-canonical InChIKey14), never by library name \\u2014 holding a\\nlibrary out by name leaks, because the same structures appear in other libraries.\\n\\"\\"\\"\\nfrom dataclasses import dataclass, field\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nCLASSES = (\\"C1\\", \\"C2\\", \\"C3\\")\\n\\n\\n@dataclass\\nclass Query:\\n    qid: str\\n    pool_key: str                 # the structure\'s pool key (standard InChIKey14)\\n    spectra: list                 # list of dicts: mz, it, precursor_mz, adduct, ce_ev, mode\\n    spectrum_rows: np.ndarray     # selected source spectra used as query input\\n    source_library_rows: np.ndarray | None = None  # ALL same-key source-library spectra for C1 exclusion\\n\\n\\n@dataclass\\nclass Knockout:\\n    \\"\\"\\"What the engine must not see for one (query, class).\\"\\"\\"\\n    excluded_rows: np.ndarray                 # store rows hidden from library search\\n    excluded_keys: set = field(default_factory=set)   # structures hidden from the pool\\n\\n\\ndef build_queries(store, key_of_row, library=\\"enveda-np-examples\\", keys_subset=None,\\n                  max_spectra=None, max_queries=None, seed=0):\\n    \\"\\"\\"Group one library\'s test-adduct spectra by structure into molecule-like queries.\\n    `keys_subset` restricts the structures; `max_spectra` subsamples each query (test median is 3).\\n    A subsampled query still excludes ALL of that structure\'s reference rows in its own\\n    source library. Otherwise an unselected duplicate spectrum leaks into C1 scoring.\\"\\"\\"\\n    meta = store.meta\\n    rng = np.random.default_rng(seed)\\n    rows = np.flatnonzero((meta.ingest_lib == library).to_numpy() & meta.test_adduct.to_numpy()\\n                          & (meta.n_peaks.to_numpy() >= 3))\\n    keys = key_of_row[rows]\\n    if keys_subset is not None:\\n        m = np.isin(keys, list(keys_subset))\\n        rows, keys = rows[m], keys[m]\\n    order = np.argsort(keys.astype(str), kind=\\"stable\\")\\n    rows, keys = rows[order], keys[order]\\n    starts = np.r_[0, np.flatnonzero(keys[1:] != keys[:-1]) + 1] if len(keys) else np.empty(0, np.int64)\\n    ends = np.r_[starts[1:], len(keys)] if len(keys) else np.empty(0, np.int64)\\n    groups = list(zip(starts, ends))\\n    if max_queries and len(groups) > max_queries:\\n        groups = [groups[i] for i in sorted(rng.choice(len(groups), max_queries, replace=False))]\\n    # Collect all source-library rows for the selected query keys, not only\\n    # test-adduct/peak-filtered inputs. Reference indices have their own\\n    # filtering rules and cannot safely be inferred from sampled inputs.\\n    wanted = {str(keys[s_]) for s_, _ in groups}\\n    source_rows = np.flatnonzero((meta.ingest_lib == library).to_numpy())\\n    reference_rows = {key: [] for key in wanted}\\n    for index in source_rows:\\n        key = str(key_of_row[index])\\n        if key in reference_rows:\\n            reference_rows[key].append(int(index))\\n\\n    queries = []\\n    for s_, e_ in groups:\\n        r = rows[s_:e_]\\n        if max_spectra and len(r) > max_spectra:\\n            r = np.sort(rng.choice(r, max_spectra, replace=False))\\n        spectra = []\\n        for i in r:\\n            mz, it = store.peaks(i)\\n            m = meta.iloc[i]\\n            spectra.append(dict(mz=mz, it=it, precursor_mz=float(m.precursor_mz), adduct=str(m.adduct),\\n                                ce_ev=float(m.ce_ev), mode=str(m.ionization_mode)))\\n        own_rows = np.asarray(reference_rows[str(keys[s_])], dtype=np.int64)\\n        queries.append(Query(\\n            qid=f\\"{library}:{keys[s_]}\\", pool_key=keys[s_],\\n            spectra=spectra, spectrum_rows=r,\\n            source_library_rows=own_rows,\\n        ))\\n    return queries\\n\\n\\ndef knockout_for(query: Query, cls: str, rows_of_key, equivalent_keys) -> Knockout:\\n    \\"\\"\\"`equivalent_keys`: every pool key whose metric key equals the truth\'s (tautomer variants\\n    stored under a different standard key). Knocking out only the query\'s own key would leak.\\"\\"\\"\\n    if cls not in CLASSES:\\n        raise ValueError(f\\"Unknown knockout class: {cls}\\")\\n    query_rows = np.asarray(query.spectrum_rows, dtype=np.int64)\\n    own_rows = (query_rows if query.source_library_rows is None\\n                else np.union1d(query_rows, query.source_library_rows))\\n    if cls == \\"C1\\":\\n        return Knockout(own_rows)\\n    rows = [rows_of_key.get(k, np.empty(0, np.int64)) for k in equivalent_keys]\\n    excluded = np.union1d(\\n        np.concatenate(rows) if rows else np.empty(0, np.int64),\\n        own_rows,\\n    )\\n    return Knockout(excluded, set(equivalent_keys) if cls == \\"C3\\" else set())\\n\\n\\n\\ndef include_equivalent_source_rows(query: Query, equivalent_keys, source_rows, key_of_row):\\n    \\"\\"\\"Extend the query\'s C1 exclusions with its same-metric-key source refs.\\n\\n    `source_rows` must contain only indices from the query\'s own library.\\n    `key_of_row` maps every store index to its *standard* structure key.\\n    Every equivalent-key variant must be removed, not just the sampled peaks.\\n    This mutates only the optional reference exclusion field, not spectra.\\n    \\"\\"\\"\\n    source_rows = np.asarray(source_rows, dtype=np.int64)\\n    key_of_row = np.asarray(key_of_row)\\n    if np.any(source_rows < 0) or np.any(source_rows >= len(key_of_row)):\\n        raise ValueError(\\"Source-library reference row outside store\\")\\n    if not equivalent_keys:\\n        raise ValueError(\\"Expected at least the query\'s own structure key\\")\\n    matching = source_rows[np.isin(key_of_row[source_rows], list(equivalent_keys))]\\n    base = query.spectrum_rows if query.source_library_rows is None else query.source_library_rows\\n    query.source_library_rows = np.union1d(base, matching).astype(np.int64, copy=False)\\n    return query.source_library_rows\\n\\n\\n\\ndef same_library_reference_rows(rows_of_key, equivalent_keys, lib_of_row, library):\\n    \\"\\"\\"D1/analog-indexed same-structure references in the query\'s library.\\n\\n    Use alongside full source-library exclusions on build_queries outputs.\\n    The caller decides which novelty class requires this cross-library\\n    knockout; do not affect intentionally retained legacy training modes.\\n    \\"\\"\\"\\n    arrays = [np.asarray(rows_of_key.get(k, []), dtype=np.int64)\\n              for k in equivalent_keys]\\n    arrays = [arr for arr in arrays if len(arr)]\\n    if not arrays:\\n        return np.empty(0, dtype=np.int64)\\n    refs = np.concatenate(arrays)\\n    libraries = np.asarray(lib_of_row)\\n    if np.any(refs < 0) or np.any(refs >= len(libraries)):\\n        raise ValueError(\\"Reference row outside library metadata\\")\\n    return np.unique(refs[libraries[refs] == library])\\n\\n\\n\\ndef equivalent_keys(query_key, query_mass, pool, keys, window_ppm=30.0, truth_smiles=None):\\n    \\"\\"\\"Metric truth and every pool key in the mass window that is the same structure under the\\n    metric (tautomer variants stored under another standard key). The skeleton key is an exact\\n    necessary condition, so only skeleton matches pay for tautomer canonicalisation.\\n    `truth_smiles` is required when the query structure itself is not in this pool.\\"\\"\\"\\n    from casmi.chem.canon import skeleton_key\\n    if truth_smiles is None:\\n        truth_smiles = pool.smiles_of(query_key)\\n    truth, skel = keys.get(query_key, truth_smiles), skeleton_key(truth_smiles)\\n    same = {query_key} if query_key in pool.row_of else set()\\n    for k in pool.key[pool.window(query_mass, window_ppm).idx]:\\n        if k != query_key and skeleton_key(pool.smiles_of(k)) == skel and keys.get(k, pool.smiles_of(k)) == truth:\\n            same.add(k)\\n    return truth, same\\n\\n\\ndef mixture(mrr_by_class: dict, f1: float, f3: float) -> float:\\n    \\"\\"\\"Expected leaderboard score for class shares f1, f2 = 1 - f1 - f3, f3.\\"\\"\\"\\n    f2 = 1.0 - f1 - f3\\n    return f1 * mrr_by_class[\\"C1\\"] + f2 * mrr_by_class[\\"C2\\"] + f3 * mrr_by_class[\\"C3\\"]\\n\\n\\ndef mixture_grid(mrr_by_class, f1_values=(0.15, 0.20, 0.25, 0.30), f3_values=(0.2, 0.3, 0.4)):\\n    rows = [(f1, f3, mixture(mrr_by_class, f1, f3)) for f1 in f1_values for f3 in f3_values if f1 + f3 < 1]\\n    return pd.DataFrame(rows, columns=[\\"f1\\", \\"f3\\", \\"score\\"])\\n", "src/casmi/validate/massbank_records.py": "\\"\\"\\"Strict MassBank 2.x CC0 MS2 record ingestion for external research inputs.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport math\\n\\n\\nclass RecordRejected(ValueError):\\n    pass\\n\\n\\ndef parse_record(text):\\n    tags, peaks, in_peaks = {}, [], False\\n    if not text.rstrip().endswith(\\"//\\"):\\n        raise RecordRejected(\\"unterminated_record\\")\\n    for line in text.splitlines():\\n        if line == \\"//\\":\\n            break\\n        if line.startswith(\\"PK$PEAK:\\"):\\n            in_peaks = True\\n            continue\\n        if in_peaks:\\n            fields = line.split()\\n            if len(fields) != 3:\\n                raise RecordRejected(\\"malformed_peak_row\\")\\n            try:\\n                mz, intensity, relative = map(float, fields)\\n            except ValueError as exc:\\n                raise RecordRejected(\\"nonnumeric_peak\\") from exc\\n            if not all(math.isfinite(v) for v in (mz, intensity, relative)) or mz <= 0 or intensity <= 0 or relative < 0:\\n                raise RecordRejected(\\"invalid_peak\\")\\n            peaks.append((mz, intensity))\\n            continue\\n        if \\": \\" in line:\\n            name, value = line.split(\\": \\", 1)\\n            if name in {\\"AC$MASS_SPECTROMETRY\\", \\"MS$FOCUSED_ION\\"}:\\n                key, separator, value = value.partition(\\" \\")\\n                if not separator:\\n                    raise RecordRejected(\\"malformed_subtag\\")\\n                name += \\":\\" + key\\n            tags.setdefault(name, []).append(value.strip())\\n\\n    def one(name):\\n        values = tags.get(name, [])\\n        if len(values) != 1 or not values[0]:\\n            raise RecordRejected(\\"missing_or_duplicate_\\" + name)\\n        return values[0]\\n\\n    if one(\\"LICENSE\\") != \\"CC0\\":\\n        raise RecordRejected(\\"non_cc0_license\\")\\n    if one(\\"AC$MASS_SPECTROMETRY:MS_TYPE\\") != \\"MS2\\" or one(\\"AC$MASS_SPECTROMETRY:IONIZATION\\") != \\"ESI\\":\\n        raise RecordRejected(\\"unsupported_acquisition\\")\\n    mode = one(\\"AC$MASS_SPECTROMETRY:ION_MODE\\").lower()\\n    if mode not in (\\"positive\\", \\"negative\\"):\\n        raise RecordRejected(\\"unsupported_mode\\")\\n    try:\\n        precursor = float(one(\\"MS$FOCUSED_ION:PRECURSOR_M/Z\\"))\\n        declared_peaks = int(one(\\"PK$NUM_PEAK\\"))\\n    except ValueError as exc:\\n        raise RecordRejected(\\"invalid_numeric_header\\") from exc\\n    if not math.isfinite(precursor) or precursor <= 0 or declared_peaks != len(peaks) or len(peaks) < 5:\\n        raise RecordRejected(\\"invalid_precursor_or_peak_count\\")\\n    if any(a[0] >= b[0] for a, b in zip(peaks, peaks[1:])):\\n        raise RecordRejected(\\"unordered_or_duplicate_mz\\")\\n    maximum = max(v for _, v in peaks)\\n    return {\\"accession\\": one(\\"ACCESSION\\"), \\"authors\\": one(\\"AUTHORS\\"), \\"license\\": \\"CC0\\",\\n            \\"record_sha256\\": hashlib.sha256(text.encode(\\"utf-8\\")).hexdigest(),\\n            \\"smiles\\": one(\\"CH$SMILES\\"), \\"precursor_mz\\": precursor,\\n            \\"adduct\\": one(\\"MS$FOCUSED_ION:PRECURSOR_TYPE\\"), \\"ionization_mode\\": mode,\\n            \\"instrument\\": one(\\"AC$INSTRUMENT_TYPE\\"), \\"ms2_mzs\\": [m for m, _ in peaks],\\n            \\"ms2_normalized_intensities\\": [100 * v / maximum for _, v in peaks]}\\n", "src/casmi/validate/metric.py": "\\"\\"\\"MRR@25 with the official metric\'s semantics, plus the per-molecule detail we need for analysis.\\n\\nSemantics reproduced from the official scoring notebook (`metric/casmi-mean-reciprocal-rank`):\\n  * rank = position in the list; an unparseable or duplicate guess still consumes its position,\\n  * only the first correct guess counts (reciprocal rank, not average precision),\\n  * a molecule absent from the submission scores 0.\\n\\"\\"\\"\\nfrom dataclasses import dataclass\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom casmi.chem.canon import canonical_ik14, heavy_atom_key\\nfrom casmi.validate.paired_bootstrap import paired_difference\\nfrom casmi.validate.ranked_slots import split_ranked_guesses\\nfrom casmi.validate.rank_inputs import validated_ranks, validated_bootstrap\\n\\nMAX_RANK = 25\\n\\n\\ndef reciprocal_rank(guesses, answer_ik14s, answer_heavy_keys, max_rank=MAX_RANK):\\n    for rank, guess in enumerate(guesses[:max_rank], 1):\\n        if heavy_atom_key(guess) not in answer_heavy_keys:\\n            continue\\n        if canonical_ik14(guess) in answer_ik14s:\\n            return 1.0 / rank\\n    return 0.0\\n\\n\\ndef score_submission(solution: pd.DataFrame, submission: pd.DataFrame, max_rank=MAX_RANK) -> pd.Series:\\n    \\"\\"\\"Per-molecule reciprocal rank. `solution` has molecule_id + smiles (comma-separated if several\\n    structures are accepted); `submission` has molecule_id + semicolon-joined guesses.\\"\\"\\"\\n    if submission[\\"molecule_id\\"].duplicated().any():\\n        raise ValueError(\\"Each molecule_id must appear exactly once\\")\\n    # Never compress malformed empty slots: a blank candidate still occupies\\n    # its original rank, even though a valid competition submission forbids it.\\n    guesses = {m: split_ranked_guesses(str(s))\\n               for m, s in zip(submission[\\"molecule_id\\"], submission[\\"smiles\\"])}\\n    too_many = [m for m, g in guesses.items() if len(g) > max_rank]\\n    if too_many:\\n        raise ValueError(f\\"More than {max_rank} guesses for {too_many[:3]}\\")\\n    out = {}\\n    for m, answer in zip(solution[\\"molecule_id\\"], solution[\\"smiles\\"]):\\n        accepted = [s for s in str(answer).split(\\",\\") if s]\\n        ik14s = {k for k in map(canonical_ik14, accepted) if k}\\n        heavy = {k for k in map(heavy_atom_key, accepted) if k}\\n        if not ik14s:\\n            raise ValueError(f\\"Unparseable solution SMILES for {m}\\")\\n        out[m] = reciprocal_rank(guesses.get(m, []), ik14s, heavy, max_rank)\\n    return pd.Series(out, name=\\"rr\\")\\n\\n\\ndef mrr(solution, submission, max_rank=MAX_RANK) -> float:\\n    return float(score_submission(solution, submission, max_rank).mean())\\n\\n\\n# ----------------------------------------------------------------------------- key-level scoring\\ndef rank_of_truth(ranked_ik14s, true_ik14, max_rank=MAX_RANK):\\n    \\"\\"\\"1-based rank of the true key in an already canonicalised, deduplicated list (0 = absent).\\n    Validation runs on keys directly, which is exact once the output writer guarantees\\n    canonical, deduplicated lists \\u2014 and ~1000x faster than re-parsing SMILES.\\"\\"\\"\\n    for rank, key in enumerate(ranked_ik14s[:max_rank], 1):\\n        if key == true_ik14:\\n            return rank\\n    return 0\\n\\n\\n@dataclass\\nclass Summary:\\n    mrr: float\\n    ci_low: float\\n    ci_high: float\\n    top1: float\\n    top10: float\\n    recall_at_25: float\\n    n: int\\n\\n    def __str__(self):\\n        return (f\\"MRR {self.mrr:.4f} [{self.ci_low:.4f}, {self.ci_high:.4f}]  top1 {self.top1:.3f}  \\"\\n                f\\"top10 {self.top10:.3f}  hit@25 {self.recall_at_25:.3f}  n={self.n}\\")\\n\\n\\ndef summarize(ranks, n_boot=2000, seed=0) -> Summary:\\n    \\"\\"\\"MRR with a molecule-level bootstrap 95% CI, from 1-based ranks (0 = not in top 25).\\"\\"\\"\\n    ranks = validated_ranks(ranks)\\n    validated_bootstrap(n_boot, seed)\\n    rr = np.where(ranks > 0, 1.0 / np.maximum(ranks, 1), 0.0)\\n    rng = np.random.default_rng(seed)\\n    boots = rr[rng.integers(0, len(rr), size=(n_boot, len(rr)))].mean(axis=1)\\n    return Summary(float(rr.mean()), float(np.quantile(boots, .025)), float(np.quantile(boots, .975)),\\n                   float((ranks == 1).mean()), float(((ranks > 0) & (ranks <= 10)).mean()),\\n                   float((ranks > 0).mean()), len(rr))\\n\\n\\n", "src/casmi/validate/overlap_audit.py": "\\"\\"\\"Aggregate-only, exact-content public-example overlap audit.\\n\\nRaw identifiers and peak arrays stay in memory and are never exported. Equality\\nof stored numeric arrays is distinct from proof of acquisition provenance.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom collections import defaultdict\\n\\nfrom .spectrum_overlap import coarse_spectrum_bucket, full_spectrum_match\\n\\n\\ndef _spectrum(row):\\n    return (row.get(\\"precursor_mz\\"), row.get(\\"ms2_mzs\\"),\\n            row.get(\\"ms2_normalized_intensities\\"))\\n\\n\\ndef _identifier(value):\\n    if value is None or not str(value).strip():\\n        raise ValueError(\\"Public example identifiers must be present\\")\\n    return str(value)\\n\\n\\nclass PublicOverlapAudit:\\n    def __init__(self, public_rows):\\n        self.targets = list(public_rows)\\n        if not self.targets:\\n            raise ValueError(\\"Public example input must not be empty\\")\\n        self.buckets = defaultdict(list)\\n        self.molecules = defaultdict(set)\\n        seen = set()\\n        for index, row in enumerate(self.targets):\\n            sid = _identifier(row.get(\\"spectrum_id\\"))\\n            molecule = _identifier(row.get(\\"molecule_id\\"))\\n            if sid in seen:\\n                raise ValueError(\\"Duplicate public spectrum identifiers\\")\\n            seen.add(sid)\\n            spectrum = _spectrum(row)\\n            if not full_spectrum_match(*spectrum, *spectrum):\\n                raise ValueError(\\"Public spectrum must have finite aligned nonempty arrays\\")\\n            self.buckets[coarse_spectrum_bucket(spectrum[0], spectrum[1])].append(index)\\n            self.molecules[molecule].add(index)\\n        self.precursors = {float(row[\\"precursor_mz\\"]) for row in self.targets}\\n        self.scanned = 0\\n        self.candidate_rows = 0\\n        self.coarse_pairs = 0\\n        self.exact_pairs = 0\\n        self.false_pairs = 0\\n        self.matched = set()\\n        self.metadata_matched = set()\\n        self.library_matches = defaultdict(set)\\n        self.metadata_disagreement_pairs = 0\\n\\n    def scan_batch(self, candidate_rows, *, scanned_count=None):\\n        \\"\\"\\"Count all scanned rows, considering only exact-precursor candidates.\\n\\n        A caller can filter Arrow batches before converting peak lists to Python.\\n        `scanned_count` must include filtered-out rows; finish checks full coverage.\\n        \\"\\"\\"\\n        rows = list(candidate_rows)\\n        count = len(rows) if scanned_count is None else scanned_count\\n        if isinstance(count, bool) or not isinstance(count, int) or count < len(rows):\\n            raise ValueError(\\"Scanned row count must cover every considered row\\")\\n        self.scanned += count\\n        for row in rows:\\n            try:\\n                precursor = float(row.get(\\"precursor_mz\\"))\\n            except (TypeError, ValueError):\\n                continue\\n            if precursor not in self.precursors:\\n                continue\\n            self.candidate_rows += 1\\n            spectrum = _spectrum(row)\\n            bucket = coarse_spectrum_bucket(spectrum[0], spectrum[1])\\n            for index in self.buckets.get(bucket, ()):\\n                self.coarse_pairs += 1\\n                target = self.targets[index]\\n                if not full_spectrum_match(*spectrum, *_spectrum(target)):\\n                    self.false_pairs += 1\\n                    continue\\n                library = row.get(\\"ingest_lib\\")\\n                if not isinstance(library, str) or not library.strip():\\n                    raise ValueError(\\"Matching training spectra need a source-library label\\")\\n                self.exact_pairs += 1\\n                self.matched.add(index)\\n                self.library_matches[library].add(index)\\n                aligned = all(target.get(key) is not None and row.get(key) == target[key]\\n                              for key in (\\"adduct\\", \\"ionization_mode\\"))\\n                if aligned:\\n                    self.metadata_matched.add(index)\\n                else:\\n                    self.metadata_disagreement_pairs += 1\\n\\n    def finish(self, *, expected_train_rows):\\n        if (isinstance(expected_train_rows, bool) or not isinstance(expected_train_rows, int)\\n                or expected_train_rows <= 0 or self.scanned != expected_train_rows):\\n            raise ValueError(\\"A complete scan of the declared training row count is required\\")\\n        return {\\n            \\"schema_version\\": 1,\\n            \\"status\\": \\"COMPLETE_CONTENT_AUDIT\\",\\n            \\"matching_method\\": \\"exact stored precursor and full ordered m/z/intensity numeric arrays; no rounding or tolerance\\",\\n            \\"metadata_fields\\": [\\"adduct\\", \\"ionization_mode\\"],\\n            \\"train_rows_scanned\\": self.scanned,\\n            \\"public_spectra\\": len(self.targets),\\n            \\"public_molecules\\": len(self.molecules),\\n            \\"exact_precursor_candidate_rows\\": self.candidate_rows,\\n            \\"coarse_candidate_pairs_after_precursor_filter\\": self.coarse_pairs,\\n            \\"coarse_false_match_pairs\\": self.false_pairs,\\n            \\"exact_matching_pairs\\": self.exact_pairs,\\n            \\"content_matched_spectra\\": len(self.matched),\\n            \\"metadata_consistent_matched_spectra\\": len(self.metadata_matched),\\n            \\"metadata_disagreement_pairs\\": self.metadata_disagreement_pairs,\\n            \\"fully_content_matched_molecules\\": sum(indices <= self.matched for indices in self.molecules.values()),\\n            \\"fully_metadata_consistent_molecules\\": sum(indices <= self.metadata_matched for indices in self.molecules.values()),\\n            \\"content_matches_by_library\\": {key: len(value) for key, value in sorted(self.library_matches.items())},\\n            \\"limitations\\": [\\n                \\"Source-library match counts overlap and must not be summed as unique public spectra.\\",\\n                \\"Stored numerical equality does not independently prove common acquisition identity.\\",\\n                \\"This audit concerns mounted public examples, not the hidden competition evaluation set.\\",\\n                \\"No model evaluation, training or competition submission was performed.\\",\\n            ],\\n        }\\n", "src/casmi/validate/paired_bootstrap.py": "\\"\\"\\"Minimal numerical bootstrap for CASMI ranking comparisons.\\n\\nSeparate from RDKit/pandas scoring so synthetic CI regression tests and\\nexperiment comparisons need only NumPy, not the entire inference stack.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nfrom casmi.validate.rank_inputs import validated_ranks, validated_bootstrap\\n\\n\\ndef paired_difference(ranks_a, ranks_b, n_boot=4000, seed=0):\\n    \\"\\"\\"Molecule-level bootstrap CI of MRR(B)-MRR(A).\\n\\n    Input sequences MUST already have been aligned on verified query IDs and\\n    spectra. This routine does not infer or validate pairing.\\n    \\"\\"\\"\\n    a, b = validated_ranks(ranks_a), validated_ranks(ranks_b)\\n    if a.shape != b.shape:\\n        raise ValueError(\\"Paired ranks must be nonempty, one-dimensional and equal length\\")\\n    validated_bootstrap(n_boot, seed)\\n    to_rr = lambda r: np.where(r > 0, 1.0 / np.maximum(r, 1), 0.0)\\n    d = to_rr(b) - to_rr(a)\\n    rng = np.random.default_rng(seed)\\n    boots = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(axis=1)\\n    return float(d.mean()), float(np.quantile(boots, .025)), float(np.quantile(boots, .975))\\n", "src/casmi/validate/rank_inputs.py": "\\"\\"\\"Shared numerical contracts for fixed MRR@25 research summaries.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\n\\ndef validated_ranks(values):\\n    ranks = np.asarray(values)\\n    if ranks.ndim != 1 or not len(ranks):\\n        raise ValueError(\\"Ranks must be a nonempty one-dimensional integer sequence\\")\\n    if not np.issubdtype(ranks.dtype, np.integer):\\n        raise ValueError(\\"Ranks must be integers from 0 to 25; 0 denotes a miss\\")\\n    if np.any((ranks < 0) | (ranks > 25)):\\n        raise ValueError(\\"Ranks must lie between 0 and 25; out-of-list hits are encoded as 0\\")\\n    return ranks\\n\\n\\ndef validated_bootstrap(n_boot, seed):\\n    if not isinstance(n_boot, (int, np.integer)) or isinstance(n_boot, (bool, np.bool_)) or n_boot < 100:\\n        raise ValueError(\\"At least 100 integer bootstrap draws are required\\")\\n    if not isinstance(seed, (int, np.integer)) or isinstance(seed, (bool, np.bool_)) or seed < 0:\\n        raise ValueError(\\"Bootstrap seed must be a nonnegative integer\\")\\n", "src/casmi/validate/ranked_slots.py": "\\"\\"\\"Position-preserving parsing for CASMI ranked SMILES predictions.\\n\\nA semicolon defines the next ranked position even when the candidate is\\nblank or invalid. Never filter an empty field when computing MRR.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\n\\ndef split_ranked_guesses(text: str) -> list[str]:\\n    \\"\\"\\"Keep every guess slot, including interior/leading/trailing blanks.\\"\\"\\"\\n    if not isinstance(text, str):\\n        raise TypeError(\\"Ranked SMILES must be a string\\")\\n    return [part.strip() for part in text.split(\\";\\")]\\n", "src/casmi/validate/run_alignment.py": "\\"\\"\\"Fail-closed identity checks for paired CASMI validation comparisons.\\n\\nPaired confidence intervals require exactly the same query molecules and the\\nsame source spectra, not merely the same number of queries. Unlike rank\\nsummaries, this module needs no numerical or chemistry dependencies.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport math\\nfrom collections.abc import Mapping, Sequence\\n\\nCLASSES = (\\"C1\\", \\"C2\\", \\"C3\\")\\nSIGNATURE_VERSION = \\"casmi-query-spectra-v2\\"\\n\\n\\ndef query_signature(query) -> str:\\n    \\"\\"\\"Fingerprint *actual* source spectra and exclusion rows for one query.\\n\\n    Signatures are validation provenance, not a tamper-proof dataset checksum.\\n    This prevents pairing runs that resample spectra, change source-library\\n    exclusion scope, or change store row order. Version the signature recipe.\\n    \\"\\"\\"\\n    payload = {\\n        \\"version\\": SIGNATURE_VERSION,\\n        \\"qid\\": str(query.qid),\\n        \\"pool_key\\": str(query.pool_key),\\n        \\"selected_query_rows\\": sorted(int(i) for i in query.spectrum_rows),\\n        \\"source_library_excluded_rows\\": (\\n            sorted(int(i) for i in query.source_library_rows)\\n            if getattr(query, \\"source_library_rows\\", None) is not None else None\\n        ),\\n        \\"spectra\\": [\\n            {\\n                \\"mz\\": [float(x).hex() for x in spec[\\"mz\\"]],\\n                \\"intensity\\": [float(x).hex() for x in spec[\\"it\\"]],\\n                \\"precursor_mz\\": float(spec[\\"precursor_mz\\"]).hex(),\\n                \\"adduct\\": str(spec[\\"adduct\\"]),\\n                \\"ce_ev\\": float(spec[\\"ce_ev\\"]).hex(),\\n                \\"mode\\": str(spec[\\"mode\\"]),\\n            }\\n            for spec in query.spectra\\n        ],\\n    }\\n    encoded = json.dumps(payload, sort_keys=True, separators=(\\",\\", \\":\\"), ensure_ascii=True)\\n    return hashlib.sha256(encoded.encode(\\"utf-8\\")).hexdigest()\\n\\n\\ndef _check_ranks(run: Mapping, n: int) -> dict[str, list[int]]:\\n    classes = run.get(\\"classes\\")\\n    if not isinstance(classes, dict):\\n        raise ValueError(\\"Missing classes in validation run\\")\\n    result = {}\\n    for cls in CLASSES:\\n        block = classes.get(cls)\\n        if not isinstance(block, dict):\\n            raise ValueError(f\\"Missing class {cls}\\")\\n        ranks = block.get(\\"ranks\\")\\n        if not isinstance(ranks, list) or len(ranks) != n:\\n            raise ValueError(f\\"{cls}: expected {n} per-query ranks\\")\\n        for rank in ranks:\\n            if type(rank) is not int or not 0 <= rank <= 25:\\n                raise ValueError(f\\"{cls}: rank must be an integer from 0 to 25\\")\\n        computed = sum(1.0 / rank for rank in ranks if rank) / n\\n        recorded = block.get(\\"mrr\\")\\n        if recorded is not None and (\\n            isinstance(recorded, bool) or not isinstance(recorded, (float, int))\\n            or not math.isfinite(recorded)\\n            or abs(computed - recorded) > 0.000005\\n        ):\\n            raise ValueError(f\\"{cls}: MRR summary disagrees with per-query ranks\\")\\n        result[cls] = ranks\\n    return result\\n\\n\\n\\ndef _source_snapshot_verified(a: Mapping, b: Mapping) -> bool:\\n    \\"\\"\\"Check local input inventory identity and complete known input coverage.\\n\\n    Historical reports without inventory hashes remain exploratory. Merely\\n    naming an identical SHA-256 does not prove that a subset manifest covers\\n    the store and active candidate pool used by either run.\\n    \\"\\"\\"\\n    from casmi.validate.source_manifest import (\\n        MANIFEST_SCHEMA, required_validation_inputs,\\n    )\\n\\n    hashes = (a.get(\\"source_manifest_sha256\\"), b.get(\\"source_manifest_sha256\\"))\\n    if hashes == (None, None):\\n        return False\\n    if None in hashes:\\n        raise ValueError(\\"Source artifact snapshot mismatch: one run lacks a manifest\\")\\n    if any(not isinstance(s, str) or len(s) != 64 or\\n           any(ch not in \\"0123456789abcdef\\" for ch in s) for s in hashes):\\n        raise ValueError(\\"Malformed source artifact snapshot SHA-256\\")\\n    if hashes[0] != hashes[1]:\\n        raise ValueError(\\"Source artifact snapshot mismatch: input files differ\\")\\n\\n    for run in (a, b):\\n        # Old signed reports did not yet validate required-file coverage.\\n        # They can be inspected, but never yield conclusive model verdicts.\\n        if run.get(\\"source_manifest_coverage\\") != \\"required-validation-inputs-v1\\":\\n            return False\\n        if run.get(\\"source_manifest_schema\\") != MANIFEST_SCHEMA:\\n            raise ValueError(\\"Invalid source artifact manifest schema\\")\\n        pool_tag = run.get(\\"source_manifest_pool_tag\\")\\n        channels = run.get(\\"config\\", {}).get(\\"channels\\") if isinstance(run.get(\\"config\\"), Mapping) else None\\n        if not isinstance(channels, list) or \\"d1\\" not in channels:\\n            raise ValueError(\\"Missing validation channel provenance\\")\\n        expected = required_validation_inputs(\\n            pool_tag=pool_tag, analog=(\\"d2\\" in channels),\\n        )\\n        provided = run.get(\\"source_manifest_required_inputs\\")\\n        if not isinstance(provided, list) or provided != list(expected):\\n            raise ValueError(\\"Incomplete or contradictory source artifact inventory\\")\\n        count = run.get(\\"source_manifest_file_count\\")\\n        if type(count) is not int or count < len(expected):\\n            raise ValueError(\\"Invalid source artifact file count\\")\\n    from casmi.validate.fingerprint_evidence import verified_d2_fingerprint_evidence\\n\\n    # An unchanged pool/fingerprint FILE inventory does not establish that\\n    # D2 vectors belong to the pool\'s exact SMILES, masses and row ordering.\\n    # Historical D2 runs without a verified sidecar remain exploratory.\\n    return verified_d2_fingerprint_evidence(a, b)\\n\\n\\ndef align_runs(a: Mapping, b: Mapping, *, allow_unverified_legacy_order: bool = False) -> dict:\\n    \\"\\"\\"Validate evidence and align B to A by query ID.\\n\\n    New-format reports must have unique IDs, per-query signatures, and the\\n    same ID->spectrum-signature mapping. Older runs can only be compared with\\n    an explicit unsafe override, and should not justify acceptance claims.\\n    \\"\\"\\"\\n    # Paired improvement inference requires identical *actual* MS/MS inputs.\\n    # Negative controls intentionally change spectra, so they must never\\n    # receive model-improvement ACCEPT/REJECT judgments from this comparator.\\n    for run in (a, b):\\n        control = run.get(\\"spectral_control\\")\\n        if control is not None:\\n            if not isinstance(control, Mapping):\\n                raise ValueError(\\"Malformed spectral-control provenance\\")\\n            if control.get(\\"mode\\", \\"none\\") != \\"none\\":\\n                raise ValueError(\\n                    \\"Negative-control reports are diagnostic only; cannot \\"\\n                    \\"be compared as model-improvement runs\\"\\n                )\\n\\n    n_a, n_b = a.get(\\"n_queries\\"), b.get(\\"n_queries\\")\\n    if type(n_a) is not int or type(n_b) is not int or n_a < 1 or n_a != n_b:\\n        raise ValueError(\\"Runs must contain the same positive number of queries\\")\\n    n = n_a\\n    ar, br = _check_ranks(a, n), _check_ranks(b, n)\\n    a_ids, b_ids = a.get(\\"query_ids\\"), b.get(\\"query_ids\\")\\n    a_sig, b_sig = a.get(\\"query_signatures\\"), b.get(\\"query_signatures\\")\\n    has_manifest = all(isinstance(x, list) for x in (a_ids, b_ids, a_sig, b_sig))\\n    if not has_manifest:\\n        if not allow_unverified_legacy_order:\\n            raise ValueError(\\n                \\"Cannot validate paired query identities/spectra in legacy runs. \\"\\n                \\"Regenerate reports with query_ids + query_signatures; \\"\\n                \\"use --allow-unverified-legacy-order only for exploratory comparison.\\"\\n            )\\n        return {\\"a\\": ar, \\"b\\": br, \\"n\\": n, \\"verified\\": False,\\n                \\"source_verified\\": False, \\"decision_eligible\\": False}\\n\\n    for run_ids, run_sigs, name in ((a_ids, a_sig, \\"A\\"), (b_ids, b_sig, \\"B\\")):\\n        if len(run_ids) != n or len(run_sigs) != n:\\n            raise ValueError(f\\"Run {name}: manifest length differs from n_queries\\")\\n        if any(not isinstance(i, str) or not i for i in run_ids):\\n            raise ValueError(f\\"Run {name}: empty or invalid query ID\\")\\n        if len(set(run_ids)) != n:\\n            raise ValueError(f\\"Run {name}: duplicate query IDs\\")\\n        if any(not isinstance(s, str) or len(s) != 64 or\\n               any(c not in \\"0123456789abcdef\\" for c in s) for s in run_sigs):\\n            raise ValueError(f\\"Run {name}: malformed query signatures\\")\\n\\n    # A changed signature recipe makes otherwise comparable digests\\n    # uninterpretable; old records without version tags remain supported.\\n    version_a = a.get(\\"query_signature_version\\")\\n    version_b = b.get(\\"query_signature_version\\")\\n    if version_a != version_b:\\n        raise ValueError(\\"Query signature algorithm versions differ\\")\\n    if version_a is not None and (not isinstance(version_a, str) or not version_a):\\n        raise ValueError(\\"Malformed query signature version\\")\\n\\n    if set(a_ids) != set(b_ids):\\n        raise ValueError(\\"Query identity mismatch: validation populations differ\\")\\n    a_by_id = dict(zip(a_ids, a_sig))\\n    b_by_id = dict(zip(b_ids, b_sig))\\n    if any(a_by_id[mid] != b_by_id[mid] for mid in a_ids):\\n        raise ValueError(\\"Spectrum/knockout mismatch: query inputs differ\\")\\n    source_verified = _source_snapshot_verified(a, b)\\n    b_positions = {mid: pos for pos, mid in enumerate(b_ids)}\\n    return {\\n        \\"a\\": ar,\\n        \\"b\\": {cls: [br[cls][b_positions[mid]] for mid in a_ids] for cls in CLASSES},\\n        \\"n\\": n,\\n        \\"verified\\": True,\\n        \\"source_verified\\": source_verified,\\n        \\"decision_eligible\\": source_verified,\\n    }\\n", "src/casmi/validate/scaffold_study.py": "\\"\\"\\"Predeclared internal chemical-family partitions and joint paired uncertainty.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\n\\nimport numpy as np\\n\\n\\ndef digest(value):\\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\\",\\", \\":\\"), default=str).encode()).hexdigest()\\n\\n\\ndef family_key(smiles):\\n    \\"\\"\\"Generic Murcko ring topology; generic whole connectivity for acyclics.\\n\\n    Atomic/bond types are intentionally removed to make exclusion conservative.\\n    This is a computational partition, not a chemical similarity guarantee.\\n    \\"\\"\\"\\n    from rdkit import Chem\\n    from rdkit.Chem.Scaffolds import MurckoScaffold\\n\\n    try:\\n        mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None\\n        if mol is None or not mol.GetNumHeavyAtoms():\\n            return None\\n        parts = Chem.GetMolFrags(mol, asMols=True)\\n        mol = max(parts, key=lambda part: (part.GetNumHeavyAtoms(), Chem.MolToSmiles(part)))\\n        core = MurckoScaffold.GetScaffoldForMol(mol)\\n        prefix = \\"ring:\\" if core.GetNumHeavyAtoms() else \\"acyclic:\\"\\n        core = core if core.GetNumHeavyAtoms() else mol\\n        return prefix + Chem.MolToSmiles(MurckoScaffold.MakeScaffoldGeneric(core), canonical=True)\\n    except (ValueError, RuntimeError, TypeError):\\n        return None\\n\\n\\ndef partition(records, sizes, salt):\\n    \\"\\"\\"One representative per family; union metric identities across families.\\n\\n    Input: key/family/metric plus query metadata. No model scores are accepted.\\n    The union prevents metric-equivalent labels crossing an unusual core change.\\n    \\"\\"\\"\\n    parents = {}\\n    def find(value):\\n        parents.setdefault(value, value)\\n        while parents[value] != value:\\n            parents[value] = parents[parents[value]]\\n            value = parents[value]\\n        return value\\n    first = {}\\n    for row in records:\\n        if not row.get(\\"family\\") or not row.get(\\"metric\\"):\\n            raise ValueError(\\"Every partition row requires a valid family and metric identity\\")\\n        family = find(row[\\"family\\"])\\n        previous = first.setdefault(row[\\"metric\\"], family)\\n        a, b = find(previous), find(family)\\n        if a != b:\\n            parents[max(a, b)] = min(a, b)\\n    groups = {}\\n    for row in records:\\n        groups.setdefault(find(row[\\"family\\"]), []).append(row)\\n    ordered = sorted(groups, key=lambda family: hashlib.sha256((salt + \\"|\\" + family).encode()).hexdigest())\\n    if set(sizes) != {\\"train\\", \\"dev\\", \\"test\\"} or any(type(n) is not int or n < 1 for n in sizes.values()):\\n        raise ValueError(\\"Positive train/dev/test counts required\\")\\n    if len(ordered) < sum(sizes.values()):\\n        raise ValueError(\\"Insufficient independent family groups for declared population\\")\\n    selected, offset = {}, 0\\n    for split in (\\"train\\", \\"dev\\", \\"test\\"):\\n        rows = []\\n        for family in ordered[offset:offset + sizes[split]]:\\n            representative = min(groups[family], key=lambda row: row[\\"key\\"])\\n            rows.append({**representative, \\"group\\": family,\\n                         \\"fold\\": int(hashlib.sha256((salt + \\"|fold|\\" + family).encode()).hexdigest(), 16) % 2})\\n        selected[split] = rows\\n        offset += sizes[split]\\n    for identity in (\\"family\\", \\"group\\", \\"metric\\", \\"key\\"):\\n        sets = {name: {row[identity] for row in values} for name, values in selected.items()}\\n        if any(sets[a] & sets[b] for a, b in ((\\"train\\", \\"dev\\"), (\\"train\\", \\"test\\"), (\\"dev\\", \\"test\\"))):\\n            raise ValueError(\\"Partition identity overlap\\")\\n    return selected\\n\\n\\ndef joint_intervals(effects, *, n_boot=4000, seed=0):\\n    \\"\\"\\"Joint max-absolute-deviation paired bootstrap across a fixed outcome family.\\"\\"\\"\\n    names = list(effects)\\n    values = np.column_stack([effects[name] for name in names])\\n    if values.ndim != 2 or not len(values) or not np.isfinite(values).all() or n_boot < 100:\\n        raise ValueError(\\"Finite paired effect vectors and at least 100 draws required\\")\\n    mean = values.mean(axis=0)\\n    rng = np.random.default_rng(seed)\\n    maximum = np.empty(n_boot)\\n    for start in range(0, n_boot, 100):\\n        ids = rng.integers(len(values), size=(min(100, n_boot - start), len(values)))\\n        maximum[start:start + len(ids)] = np.max(np.abs(values[ids].mean(axis=1) - mean), axis=1)\\n    radius = float(np.quantile(maximum, .95))\\n    return {\\"method\\": \\"molecule-paired max-deviation bootstrap over all prespecified secondary contrasts\\",\\n            \\"n_molecules\\": len(values), \\"n_bootstrap\\": n_boot, \\"seed\\": seed,\\n            \\"family_size\\": len(names), \\"joint_radius\\": radius,\\n            \\"outcomes\\": {name: {\\"delta_mrr\\": float(value), \\"ci_low\\": float(value - radius),\\n                                \\"ci_high\\": float(value + radius)} for name, value in zip(names, mean)},\\n            \\"limitations\\": \\"Approximate fixed-family uncertainty; no correction for domain shift or historical hypothesis selection\\"}\\n", "src/casmi/validate/simultaneous_ci.py": "\\"\\"\\"Simultaneous paired-bootstrap uncertainty for CASMI novelty mixtures.\\n\\nThe old comparator calculated 12 pointwise 95% intervals and inspected\\nall of them. They are not a joint 95% confidence statement.\\nEach bootstrap draw here reuses the same molecules across C1/C2/C3;\\na max-deviation statistic gives approximately simultaneous intervals\\nover the predeclared novelty-share grid.\\n\\nThese bounds capture validation sampling variation, not Kaggle domain shift,\\nhyperparameter search multiplicity, model bias, or data leakage.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\nDEFAULT_GRID = tuple(\\n    (f1, f3) for f1 in (0.15, 0.20, 0.25, 0.30)\\n    for f3 in (0.2, 0.3, 0.4)\\n)\\n\\n\\ndef _ranks_to_rr(values, cls, side):\\n    ranks = np.asarray(values)\\n    if ranks.ndim != 1 or not len(ranks):\\n        raise ValueError(f\\"{side} {cls}: ranks must be nonempty 1D\\")\\n    if not np.issubdtype(ranks.dtype, np.integer):\\n        raise ValueError(f\\"{side} {cls}: ranks must be integers\\")\\n    if np.any((ranks < 0) | (ranks > 25)):\\n        raise ValueError(f\\"{side} {cls}: invalid rank (valid: 0 to 25)\\")\\n    return np.where(ranks > 0, 1.0 / np.maximum(ranks, 1), 0.0)\\n\\n\\ndef paired_novelty_deltas(a: dict, b: dict) -> np.ndarray:\\n    \\"\\"\\"Molecule x class effect matrix, with preverified aligned query ordering.\\"\\"\\"\\n    columns = []\\n    n = None\\n    for cls in (\\"C1\\", \\"C2\\", \\"C3\\"):\\n        if cls not in a or cls not in b:\\n            raise ValueError(f\\"Missing novelty class {cls}\\")\\n        x, y = _ranks_to_rr(a[cls], cls, \\"A\\"), _ranks_to_rr(b[cls], cls, \\"B\\")\\n        if x.shape != y.shape or (n is not None and len(x) != n):\\n            raise ValueError(\\"Unpaired class ranks, unequal lengths\\")\\n        n = len(x)\\n        columns.append(y - x)\\n    return np.stack(columns, axis=1)\\n\\n\\ndef compare_mixtures(\\n    a: dict,\\n    b: dict,\\n    *,\\n    grid=DEFAULT_GRID,\\n    n_boot: int = 4000,\\n    seed: int = 0,\\n    confidence: float = 0.95,\\n) -> dict:\\n    \\"\\"\\"Approximate simultaneous CI over a fixed mixture grid.\\n\\n    Must follow align_runs; this function is not an identity check.\\n    Bootstrap rows remain paired across all three novelty classes.\\n    \\"\\"\\"\\n    if type(n_boot) is not int or n_boot < 100:\\n        raise ValueError(\\"Use at least 100 bootstrap resamples\\")\\n    if not 0 < confidence < 1:\\n        raise ValueError(\\"Confidence must lie strictly between 0 and 1\\")\\n    if not isinstance(seed, (int, np.integer)) or seed < 0:\\n        raise ValueError(\\"Seed must be a nonnegative integer\\")\\n    shares = np.asarray(grid, dtype=float)\\n    if shares.ndim != 2 or shares.shape[1] != 2 or len(shares) == 0:\\n        raise ValueError(\\"Grid must contain (f1, f3) pairs\\")\\n    if not np.isfinite(shares).all():\\n        raise ValueError(\\"Mixture shares must be finite\\")\\n    f1, f3 = shares[:, 0], shares[:, 1]\\n    f2 = 1.0 - f1 - f3\\n    if np.any(f1 < 0) or np.any(f3 < 0) or np.any(f2 < -1e-12):\\n        raise ValueError(\\"Mixture fractions must be nonnegative and sum to 1\\")\\n    f2 = np.maximum(f2, 0.0)\\n    weights = np.stack([f1, f2, f3], axis=1)\\n\\n    effects = paired_novelty_deltas(a, b)\\n    mean = effects.mean(axis=0) @ weights.T\\n    rng = np.random.default_rng(int(seed))\\n    indices = rng.integers(0, effects.shape[0], size=(n_boot, effects.shape[0]))\\n    sampled = effects[indices].mean(axis=1) @ weights.T\\n    max_abs_deviation = np.max(np.abs(sampled - mean), axis=1)\\n    radius = float(np.quantile(max_abs_deviation, confidence))\\n    lower, upper = mean - radius, mean + radius\\n    verdict = (\\"ACCEPT\\" if np.all(lower > 0) else\\n               \\"REJECT\\" if np.all(upper < 0) else \\"NOT PROVEN\\")\\n    scenarios = [\\n        {\\"f1\\": float(f1[i]), \\"f2\\": float(f2[i]), \\"f3\\": float(f3[i]),\\n         \\"delta_mrr\\": float(mean[i]), \\"ci_low\\": float(lower[i]),\\n         \\"ci_high\\": float(upper[i])}\\n        for i in range(len(mean))\\n    ]\\n    return {\\n        \\"method\\": \\"paired bootstrap; maximum absolute deviation over fixed mixture grid\\",\\n        \\"confidence\\": confidence,\\n        \\"familywise\\": \\"approximate for a fixed prespecified grid; not post-selection corrected\\",\\n        \\"n_molecules\\": int(len(effects)),\\n        \\"n_boot\\": n_boot,\\n        \\"seed\\": int(seed),\\n        \\"half_width\\": radius,\\n        \\"verdict\\": verdict,\\n        \\"scenarios\\": scenarios,\\n    }\\n", "src/casmi/validate/source_manifest.py": "\\"\\"\\"Privacy-preserving SHA-256 manifests for approved local CASMI artifacts.\\n\\nNo dataset payload, raw spectra, private labels, credentials or absolute paths\\nare written into a manifest. This is a provenance check, not a security\\nsignature or an assertion that upstream Kaggle/BigQuery data are correct.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom pathlib import Path, PurePosixPath\\n\\nMANIFEST_SCHEMA = \\"casmi-local-artifact-manifest-v1\\"\\nCHUNK_BYTES = 1024 * 1024\\n\\n\\ndef _safe_relative(root: Path, name: str) -> Path:\\n    if not isinstance(name, str) or not name or \\"\\\\\\\\\\" in name:\\n        raise ValueError(\\"Manifest entries must be normalized relative POSIX paths\\")\\n    normalized = PurePosixPath(name)\\n    if (normalized.is_absolute() or name != normalized.as_posix()\\n            or any(part in (\\".\\", \\"..\\") for part in name.split(\\"/\\"))):\\n        raise ValueError(f\\"Unsafe artifact path: {name!r}\\")\\n    path = (root / name).resolve(strict=True)\\n    if not path.is_relative_to(root.resolve()):\\n        raise ValueError(f\\"Artifact resolves outside source root: {name!r}\\")\\n    if not path.is_file():\\n        raise ValueError(f\\"Expected an artifact file: {name!r}\\")\\n    return path\\n\\n\\ndef sha256_file(path: Path) -> str:\\n    digest = hashlib.sha256()\\n    with path.open(\\"rb\\") as file:\\n        for chunk in iter(lambda: file.read(CHUNK_BYTES), b\\"\\"):\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\\n\\ndef _canonical_files_digest(files: list[dict]) -> str:\\n    payload = json.dumps(files, ensure_ascii=True, sort_keys=True,\\n                         separators=(\\",\\", \\":\\"))\\n    return hashlib.sha256(payload.encode(\\"utf-8\\")).hexdigest()\\n\\n\\ndef create_manifest(root: Path, relative_paths) -> dict:\\n    \\"\\"\\"Hash existing explicit input artifacts; no directory glob or traversal.\\"\\"\\"\\n    root = Path(root).resolve(strict=True)\\n    if not root.is_dir():\\n        raise ValueError(\\"Source root must be a directory\\")\\n    if not isinstance(relative_paths, (list, tuple)) or not relative_paths:\\n        raise ValueError(\\"Provide at least one explicit input-artifact path\\")\\n    if len(set(relative_paths)) != len(relative_paths):\\n        raise ValueError(\\"Duplicate artifact manifest path\\")\\n    files = []\\n    for name in sorted(relative_paths):\\n        path = _safe_relative(root, name)\\n        files.append({\\n            \\"path\\": name, \\"bytes\\": path.stat().st_size,\\n            \\"sha256\\": sha256_file(path),\\n        })\\n    return {\\n        \\"schema\\": MANIFEST_SCHEMA,\\n        \\"method\\": \\"streaming SHA-256 per local input file\\",\\n        \\"files\\": files,\\n        \\"files_digest_sha256\\": _canonical_files_digest(files),\\n        \\"source_assertion\\": (\\n            \\"local byte identity only; no independent Kaggle/BigQuery validation\\"\\n        ),\\n    }\\n\\n\\ndef verify_manifest(root: Path, manifest: dict) -> dict:\\n    \\"\\"\\"Fail closed when a local artifact differs from the recorded snapshot.\\"\\"\\"\\n    if not isinstance(manifest, dict) or manifest.get(\\"schema\\") != MANIFEST_SCHEMA:\\n        raise ValueError(\\"Unsupported artifact manifest schema\\")\\n    entries = manifest.get(\\"files\\")\\n    if not isinstance(entries, list) or not entries:\\n        raise ValueError(\\"Manifest must contain artifact entries\\")\\n    paths = [x.get(\\"path\\") for x in entries if isinstance(x, dict)]\\n    if len(paths) != len(entries) or len(paths) != len(set(paths)):\\n        raise ValueError(\\"Invalid or duplicate manifest paths\\")\\n    for row in entries:\\n        if type(row.get(\\"bytes\\")) is not int or row[\\"bytes\\"] < 0:\\n            raise ValueError(\\"Invalid manifest file size\\")\\n        sha = row.get(\\"sha256\\")\\n        if not isinstance(sha, str) or len(sha) != 64 or any(c not in \\"0123456789abcdef\\" for c in sha):\\n            raise ValueError(\\"Invalid manifest file hash\\")\\n    if entries != sorted(entries, key=lambda e: e[\\"path\\"]):\\n        raise ValueError(\\"Manifest entries must be sorted\\")\\n    digest = manifest.get(\\"files_digest_sha256\\")\\n    if digest != _canonical_files_digest(entries):\\n        raise ValueError(\\"Manifest file list/digest mismatch\\")\\n    fresh = create_manifest(root, paths)\\n    if fresh[\\"files\\"] != entries:\\n        raise ValueError(\\"Local input artifacts differ from the signed-inventory snapshot\\")\\n    return {\\n        \\"status\\": \\"VERIFIED_LOCAL_ARTIFACTS\\",\\n        \\"schema\\": MANIFEST_SCHEMA,\\n        \\"files_digest_sha256\\": fresh[\\"files_digest_sha256\\"],\\n        \\"files_checked\\": len(entries),\\n    }\\n\\ndef required_validation_inputs(*, pool_tag: str = \\"\\", analog: bool = False) -> tuple[str, ...]:\\n    \\"\\"\\"Enumerate the minimum immutable data inputs used by validation.\\n\\n    Only the required static store and pool artifacts are included here.\\n    Derived caches and mutable metric-key memoization remain outside this\\n    minimal source-input attestation.\\n    \\"\\"\\"\\n    import re\\n\\n    if not isinstance(pool_tag, str) or not re.fullmatch(r\\"(?:_[A-Za-z0-9_-]+)?\\", pool_tag):\\n        raise ValueError(\\"Invalid candidate-pool tag\\")\\n    files = [\\n        \\"store/meta.parquet\\", \\"store/offsets.i64\\", \\"store/mz.f32\\",\\n        \\"store/it.f32\\", f\\"pool{pool_tag}.parquet\\",\\n    ]\\n    if analog:\\n        files.extend((\\n            f\\"pool_fp2048{pool_tag}.npy\\",\\n            f\\"pool_fp_keys{pool_tag}.parquet\\",\\n        ))\\n    return tuple(sorted(files))\\n\\n\\ndef require_validation_coverage(manifest: dict, *, pool_tag: str = \\"\\",\\n                                analog: bool = False) -> tuple[str, ...]:\\n    \\"\\"\\"Fail closed if an opt-in inventory omits an input consumed by this run.\\n\\n    Must be followed by verify_manifest to validate file contents and hashes.\\n    \\"\\"\\"\\n    if not isinstance(manifest, dict) or not isinstance(manifest.get(\\"files\\"), list):\\n        raise ValueError(\\"Invalid local artifact inventory\\")\\n    paths = set()\\n    for entry in manifest[\\"files\\"]:\\n        if not isinstance(entry, dict) or not isinstance(entry.get(\\"path\\"), str):\\n            raise ValueError(\\"Malformed local artifact path entry\\")\\n        paths.add(entry[\\"path\\"])\\n    expected = required_validation_inputs(pool_tag=pool_tag, analog=analog)\\n    missing = sorted(set(expected) - paths)\\n    if missing:\\n        raise ValueError(\\n            \\"Source manifest omits required validation inputs: \\" + \\", \\".join(missing)\\n        )\\n    return expected\\n\\n", "src/casmi/validate/spectral_controls.py": "\\"\\"\\"Reproducible negative-control perturbations of query MS/MS peaks.\\n\\nThese transformations are *diagnostics*, never submission-model features:\\n- blank: suppress all fragment evidence while preserving precursor mass,\\n  adduct, ionization mode and collision energy; this exposes non-spectral\\n  candidate-pool/mass priors.\\n- intensity_shuffle: independently permute fragment intensities within each\\n  spectrum. m/z and precursor stay fixed, while intensity/mz relationships\\n  are intentionally broken. Identical intensities cannot be changed.\\n\\nOriginal query spectra are never mutated. For molecule-level reproducibility\\npermutations derive from (seed, query ID, spectrum index), not list position.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\n\\nimport numpy as np\\n\\nMODES = (\\"none\\", \\"blank\\", \\"intensity_shuffle\\")\\n\\n\\ndef perturb_spectra(spectra, *, mode: str = \\"none\\",\\n                    seed: int = 0, query_id: str = \\"\\") -> list[dict]:\\n    \\"\\"\\"Make an independent copy of each spectrum, perturbing only peak arrays.\\"\\"\\"\\n    if mode not in MODES:\\n        raise ValueError(f\\"Unsupported spectral control: {mode}\\")\\n    if type(seed) is not int or seed < 0:\\n        raise ValueError(\\"Control seed must be a nonnegative integer\\")\\n    if not isinstance(query_id, str) or not query_id:\\n        raise ValueError(\\"query_id must identify one molecule\\")\\n    controlled = []\\n    for index, spectrum in enumerate(spectra):\\n        original_mz = np.array(spectrum[\\"mz\\"], copy=True)\\n        original_it = np.array(spectrum[\\"it\\"], copy=True)\\n        if original_mz.ndim != 1 or original_it.ndim != 1 or len(original_mz) != len(original_it):\\n            raise ValueError(\\"Fragment m/z and intensity arrays must have matching 1D shapes\\")\\n        item = dict(spectrum)\\n        if mode == \\"blank\\":\\n            item[\\"mz\\"] = np.empty(0, dtype=original_mz.dtype)\\n            item[\\"it\\"] = np.empty(0, dtype=original_it.dtype)\\n        elif mode == \\"intensity_shuffle\\" and len(original_it) > 1:\\n            digest = hashlib.sha256(\\n                f\\"casmi-negative-control-v1:{seed}:{query_id}:{index}\\".encode()\\n            ).digest()\\n            rng = np.random.default_rng(int.from_bytes(digest[:8], \\"big\\"))\\n            permutation = rng.permutation(len(original_it))\\n            if np.array_equal(permutation, np.arange(len(original_it))):\\n                permutation = np.roll(permutation, 1)\\n            item[\\"mz\\"] = original_mz\\n            item[\\"it\\"] = original_it[permutation]\\n        else:\\n            item[\\"mz\\"], item[\\"it\\"] = original_mz, original_it\\n        controlled.append(item)\\n    return controlled\\n", "src/casmi/validate/spectrum_overlap.py": "\\"\\"\\"Exact-peak overlap helpers for CASMI validation leakage checks.\\n\\nA precursor mass, number of peaks and first m/z are a *bucket*, not evidence\\nof a duplicate spectrum. Only full arrays, including intensities, can confirm\\na same-spectrum match. This module deliberately has no numerical dependencies.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\nfrom collections.abc import Sequence\\n\\n\\ndef coarse_spectrum_bucket(precursor_mz: float, ms2_mzs: Sequence) -> tuple | None:\\n    \\"\\"\\"Cheap candidate retrieval; NEVER use the bucket as a match on its own.\\"\\"\\"\\n    if ms2_mzs is None or len(ms2_mzs) == 0:\\n        return None\\n    try:\\n        precursor, first = float(precursor_mz), float(ms2_mzs[0])\\n    except (TypeError, ValueError):\\n        return None\\n    if not math.isfinite(precursor) or not math.isfinite(first):\\n        return None\\n    return round(precursor, 4), len(ms2_mzs), round(first, 4)\\n\\n\\ndef full_spectrum_match(\\n    precursor_a, mz_a: Sequence, intensities_a: Sequence,\\n    precursor_b, mz_b: Sequence, intensities_b: Sequence,\\n) -> bool:\\n    \\"\\"\\"Require full precursor, fragment m/z and normalized intensity equality.\\n\\n    Converts array scalar dtypes to Python float without rounding. An equal\\n    coarse bucket with shifted *any* peak or intensity is not a verified match.\\n    \\"\\"\\"\\n    if mz_a is None or mz_b is None or intensities_a is None or intensities_b is None:\\n        return False\\n    if len(mz_a) == 0 or len(mz_b) == 0:\\n        return False\\n    if len(mz_a) != len(mz_b) or len(intensities_a) != len(intensities_b):\\n        return False\\n    if len(mz_a) != len(intensities_a):\\n        return False\\n    try:\\n        values_a = [float(precursor_a)] + [float(x) for x in mz_a] + [float(x) for x in intensities_a]\\n        values_b = [float(precursor_b)] + [float(x) for x in mz_b] + [float(x) for x in intensities_b]\\n    except (TypeError, ValueError):\\n        return False\\n    if not all(math.isfinite(x) for x in values_a + values_b):\\n        return False\\n    return values_a == values_b\\n", "scripts/run_external_audit.py": "\\"\\"\\"Audit pinned external CC0 structures and stored spectra against full CASMI train.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport sys\\nfrom datetime import UTC, datetime\\n\\nROOT = Path(__file__).resolve().parents[1]\\nsys.path[:0] = [str(ROOT), str(ROOT / \\"src\\")]\\n\\n\\ndef canonical_digest(value):\\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\\",\\", \\":\\")).encode()).hexdigest()\\n\\n\\ndef audit(train_path, external_path, manifest_path, protocol_path):\\n    import pandas as pd\\n    import pyarrow as pa\\n    import pyarrow.compute as pc\\n    import pyarrow.parquet as pq\\n    from casmi.chem.canon import skeleton_key, canonical_ik14\\n    from casmi.validate.overlap_audit import PublicOverlapAudit\\n    from casmi.validate.source_manifest import sha256_file\\n\\n    protocol = json.loads(Path(protocol_path).read_text(encoding=\\"utf-8\\"))\\n    source = json.loads(Path(manifest_path).read_text(encoding=\\"utf-8\\"))\\n    external_path, train_path = Path(external_path), Path(train_path)\\n    if source[\\"source\\"] != protocol[\\"external_source\\"] or sha256_file(external_path) != source[\\"parquet_sha256\\"]:\\n        raise ValueError(\\"External source manifest differs from declared inputs\\")\\n    external = pd.read_parquet(external_path)\\n    if set(external.license) != {\\"CC0\\"} or external.accession.duplicated().any():\\n        raise ValueError(\\"External license or accession contract failed\\")\\n    targets = external.rename(columns={\\"accession\\": \\"spectrum_id\\", \\"metric_key\\": \\"molecule_id\\"}).to_dict(\\"records\\")\\n    content = PublicOverlapAudit(targets)\\n    target_skeletons = set(external.skeleton)\\n    target_metrics = set(external.metric_key)\\n    training_standard, overlapping_metrics, seen_smiles = set(), set(), set()\\n    invalid_structures = matching_skeletons = 0\\n    train = pq.ParquetFile(train_path)\\n    precursors = pa.array(sorted(content.precursors), type=pa.float64())\\n    columns = [\\"normalized_smiles\\", \\"inchikey14\\", \\"precursor_mz\\", \\"ms2_mzs\\", \\"ms2_normalized_intensities\\", \\"adduct\\", \\"ionization_mode\\", \\"ingest_lib\\"]\\n    for index, batch in enumerate(train.iter_batches(batch_size=8192, columns=columns)):\\n        metadata = batch.select([\\"normalized_smiles\\", \\"inchikey14\\"]).to_pylist()\\n        for row in metadata:\\n            key, smiles = row[\\"inchikey14\\"], row[\\"normalized_smiles\\"]\\n            if not isinstance(key, str) or not key:\\n                raise ValueError(\\"Training identity is missing\\")\\n            training_standard.add(key)\\n            if smiles in seen_smiles:\\n                continue\\n            seen_smiles.add(smiles)\\n            skeleton = skeleton_key(smiles)\\n            if skeleton is None:\\n                invalid_structures += 1\\n            elif skeleton in target_skeletons:\\n                matching_skeletons += 1\\n                metric = canonical_ik14(smiles)\\n                if metric is None:\\n                    raise ValueError(\\"Matching training skeleton has unverifiable metric identity\\")\\n                if metric in target_metrics:\\n                    overlapping_metrics.add(metric)\\n        mask = pc.is_in(pc.cast(batch.column(\\"precursor_mz\\"), pa.float64()), value_set=precursors)\\n        content.scan_batch(batch.filter(mask).to_pylist(), scanned_count=batch.num_rows)\\n        if index % 25 == 0:\\n            print(f\\"External audit scanned {content.scanned}/{train.metadata.num_rows} training rows\\", flush=True)\\n    content_summary = content.finish(expected_train_rows=train.metadata.num_rows)\\n    # All spectra belonging to a matched external molecule are excluded together.\\n    content_metrics = {targets[index][\\"molecule_id\\"] for index in content.matched}\\n    standard_metrics = set(external.loc[external.standard_ik14.isin(training_standard), \\"metric_key\\"])\\n    rejected = overlapping_metrics | content_metrics | standard_metrics\\n    novel = external[~external.metric_key.isin(rejected)].copy()\\n    novel_metrics = sorted(set(novel.metric_key))\\n    verified = invalid_structures == 0\\n    enough = len(novel_metrics) >= 30\\n    report = {\\n        \\"schema_version\\": 1,\\n        \\"status\\": \\"EXTERNAL_INDEPENDENCE_FEASIBLE\\" if verified and enough else \\"EXTERNAL_INDEPENDENCE_NOT_ESTABLISHED\\",\\n        \\"checked_at\\": datetime.now(UTC).isoformat(),\\n        \\"protocol_sha256\\": sha256_file(Path(protocol_path)),\\n        \\"inputs\\": {\\"train.parquet\\": {\\"sha256\\": sha256_file(train_path), \\"bytes\\": train_path.stat().st_size},\\n                   external_path.name: {\\"sha256\\": sha256_file(external_path), \\"bytes\\": external_path.stat().st_size}},\\n        \\"external_source\\": source,\\n        \\"identity_audit\\": {\\"training_rows_scanned\\": content.scanned, \\"training_standard_identities\\": len(training_standard),\\n                           \\"distinct_training_smiles\\": len(seen_smiles), \\"unparseable_training_labels\\": invalid_structures,\\n                           \\"training_smiles_sharing_external_skeleton\\": matching_skeletons,\\n                           \\"external_standard_overlap_molecules\\": len(standard_metrics),\\n                           \\"external_metric_overlap_molecules\\": len(overlapping_metrics),\\n                           \\"external_exact_content_overlap_molecules\\": len(content_metrics),\\n                           \\"excluded_external_molecules_union\\": len(rejected),\\n                           \\"novel_external_molecules\\": len(novel_metrics), \\"novel_external_spectra\\": len(novel),\\n                           \\"novel_metric_identities_sha256\\": canonical_digest(novel_metrics),\\n                           \\"novel_accessions_sha256\\": canonical_digest(sorted(novel.accession))},\\n        \\"stored_content_audit\\": content_summary,\\n        \\"model_performance_inspected\\": False,\\n        \\"limitations\\": [\\"External source acquisition is independently retrieved public reference data, not a new laboratory replication.\\",\\n                        \\"Identity checks rely on source chemical annotations and the official computational canonicalizer.\\",\\n                        \\"Exact stored-content comparison cannot detect arbitrary preprocessing/normalization variants of a common acquisition.\\",\\n                        \\"This feasibility stage does not establish supervised-family separation, model accuracy or a C3 coverage improvement.\\"]}\\n    return report, novel\\n\\n\\nif __name__ == \\"__main__\\":\\n    import argparse\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--train\\", required=True, type=Path)\\n    parser.add_argument(\\"--external\\", required=True, type=Path)\\n    parser.add_argument(\\"--manifest\\", required=True, type=Path)\\n    parser.add_argument(\\"--protocol\\", required=True, type=Path)\\n    parser.add_argument(\\"--output\\", required=True, type=Path)\\n    args = parser.parse_args()\\n    report, _ = audit(args.train, args.external, args.manifest, args.protocol)\\n    with args.output.open(\\"x\\", encoding=\\"utf-8\\") as destination:\\n        json.dump(report, destination, indent=2, allow_nan=False)\\n        destination.write(\\"\\\\n\\")\\n    print(\\"External metadata/content feasibility audit completed; no model outcomes inspected\\", flush=True)\\n"}')
protocol_text = '{\n  "schema_version": 1,\n  "parent_protocol": "external-c3-v1.json",\n  "registered_on": "2026-10-27",\n  "reason": "The completed MassBank feasibility audit found zero novel molecules among 491 eligible identities. No model performance was inspected. Add two directly contributed GNPS2 standard libraries without relaxing identity novelty, acquisition, minimum population or family-separation gates.",\n  "external_source": {\n    "repository": "https://library.gnps2.org/libraries",\n    "retrieved_on": "2026-10-27",\n    "license_policy": "Directly contributed GNPS reference spectra default to CC0; exclude third-party imported and propagated/MassQL/suspect libraries.",\n    "license_reference": "https://ccms-ucsd.github.io/GNPSDocumentation/browselibraries/#license",\n    "allowed_record_licenses": ["CC0"],\n    "files": {\n      "CEMBIO_OXYLIPINS.json": {"url": "https://library.gnps2.org/api/libraries/CEMBIO_OXYLIPINS/json", "sha256": "5bb461648eb111304b8e47765a177a01b443b9ad0589038ea5f261428814130e", "bytes": 192299},\n      "USER-CONTRIBUTED-GNPS2.json": {"url": "https://library.gnps2.org/api/libraries/USER-CONTRIBUTED-GNPS2/json", "sha256": "3146f450ba71009edac2375041a05de3fb1dae282903119074701a84fbc15e7a", "bytes": 302638}\n    },\n    "eligibility": "Source-classified standard spectra; MS2, ESI evidenced in source/instrument annotation; supported adduct and matching mode; connected valid molecular structure, 150-1170 Da, precursor error <=20 ppm, at least five finite positive ordered peaks; reject explicit low-quality or inferred/synthetic acquisition annotations. Preserve raw record hash, original metadata and source URL. GNPS2 snapshot-relative record IDs are derived hashes, not claimed official accessions."\n  },\n  "feasibility_stage": "Audit every eligible source molecule against every CASMI training row using the unchanged standard/metric/exact-content exclusion policy. At least 30 novel molecules are required. Counts alone may determine feasibility; no prediction or candidate-coverage inspection.",\n  "failure_policy": "Preserve the failed MassBank source and this outcome-blind addition separately. Insufficient population cannot be relabeled as external validation. No model study starts on this source unless all original independence requirements pass."\n}\n'
work=Path('/tmp/casmi_external_audit')
work.mkdir(exist_ok=False)
for name,text in sources.items():
    path=work/name
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(text,encoding='utf-8')
protocol=work/'protocol.json'
protocol.write_text(protocol_text,encoding='utf-8')
def unique(pattern):
    paths=sorted(Path('/kaggle/input').rglob(pattern))
    if len(paths)!=1:raise ValueError('Required input missing or ambiguous')
    return paths[0]
wheel=unique(f'rdkit-2026.3.3-cp{sys.version_info.major}{sys.version_info.minor}-*.whl')
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--force-reinstall',str(wheel)],check=True)
output=Path('/kaggle/working/external_audit.json')
command=[sys.executable,str(work/'scripts/run_external_audit.py'),'--train',str(unique('train.parquet')),
         '--external',str(unique('gnps_cc0.parquet')),'--manifest',str(unique('external_input_manifest.json')),
         '--protocol',str(protocol),'--output',str(output)]
env={**os.environ,'PYTHONPATH':str(work/'src')+os.pathsep+str(work),'PYTHONUNBUFFERED':'1'}
with (work/'audit-private.log').open('w',encoding='utf-8') as log:
    process=subprocess.Popen(command,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in process.stdout:
        log.write(line)
        if line.startswith(('External audit scanned','External metadata/content')):print(line.strip(),flush=True)
    status=process.wait()
if status:
    text=(work/'audit-private.log').read_text(encoding='utf-8',errors='replace')
    errors=re.findall(r'^([A-Za-z.]*Error|AssertionError):',text,flags=re.M)
    Path('/kaggle/working/external_audit_failure.json').write_text(json.dumps({'status':'FAILED_CLOSED','exception_type':errors[-1] if errors else 'Unknown'}))
    raise RuntimeError('External source audit failed closed')
report=json.loads(output.read_text())
report['source_code_sha256']={n:hashlib.sha256(t.encode()).hexdigest() for n,t in sources.items()}
report['execution']={'platform':'Kaggle','internet_enabled':False,'gpu_enabled':False,'model_training':False,'submitted_predictions':False,'protocol_registered_commit':'83c8037'}
output.write_text(json.dumps(report,indent=2,allow_nan=False)+'\n')
print('Aggregate external feasibility evidence exported; no raw source records or model outcomes exported.')
